# Piloto de audiolivro adaptado: Reinações de Narizinho

Narra, numa GPU grátis do Google Colab, os capítulos 1 a 40 de *Reinações de Narizinho*, numa adaptação para crianças e adolescentes com deficiência visual, síndrome de Down, autismo e TDAH. O guia de adaptação e de narração está no `GUIA.md` da pasta Audiolivros.

**Antes de começar:** menu `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4`. Depois rode as células de cima para baixo. Rodar de novo qualquer célula é seguro.

**Saída** (no seu Google Drive, em `MyDrive/Audiolivros/narizinho/capNN/`), para cada capítulo:
- `capNN_A_so_voz.mp3`: só a narração, para quem é sensível a sons;
- `capNN_B_com_ambiente.mp3`: narração com vinheta suave, alguns sons curtos e baixinhos nas pausas (sonoplastia leve) e, nas cenas do riacho, som de água bem baixinho.

Escolha os capítulos na célula 2 (`CAPITULOS_PARA_GERAR`). As vozes ficam em `narizinho/vozes/` e são as mesmas em todos os capítulos.

**Licença:** a voz padrão (OmniVoice) é de uso **não comercial** (CC-BY-NC). Serve para um piloto gratuito. As vozes são sintéticas e originais, sem clonar ninguém.

**Sotaque:** o modelo fala português, mas às vezes puxa para o sotaque de Portugal. Se acontecer, troque a `semente` da voz na célula 5. Outra saída é usar uma gravação de 5 a 10 segundos de alguém que **autorizou por escrito** o uso da própria voz (campo `amostra`).

In [ ]:
# ── 1. Conferir a GPU ───────────────────────────────────────────────────────
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("ATENÇÃO: sem GPU. Vá em Ambiente de execução -> Alterar o tipo de ambiente "
          "de execução -> GPU T4, e rode o notebook de novo do início.")

In [ ]:
# ── 2. Google Drive e pastas ────────────────────────────────────────────────
import os
from google.colab import drive
drive.mount("/content/drive")

CAPITULOS_PARA_GERAR = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]   # quais capítulos narrar (ex.: [2, 3])

LIVRO = "/content/drive/MyDrive/Audiolivros/narizinho"
PASTA_VOZES = os.path.join(LIVRO, "vozes")   # uma fala de referência por personagem
os.makedirs(PASTA_VOZES, exist_ok=True)
print("Os arquivos vão para:", LIVRO)

In [ ]:
# ── 3. Instalar o VoiceStudio (primeira vez: ~5-8 min) ──────────────────────
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/sdpaula36-stack/VoiceStudio"
REPO_DIR = "/content/VoiceStudio"
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}")
    if subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode != 0:
        raise SystemExit(f"\nFALHOU: {what or shown}. Veja o erro acima; rodar de novo é seguro.")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    run(["git", "-C", REPO_DIR, "pull", "--ff-only"], what="git pull")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], what="git clone")

run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get install")
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
run(["uv", "pip", "install", "--system", "--no-cache",
     "--constraint", "deploy/torch-constraints.txt",
     "--override", "deploy/uv-overrides.txt", "."],
    cwd=REPO_DIR, what="instalação do backend")

from huggingface_hub import snapshot_download
print("Baixando o modelo de voz (alguns GB, só na primeira vez)...")
snapshot_download("k2-fsa/OmniVoice")
print("Pronto.")

In [ ]:
# ── 4. Ligar o servidor do VoiceStudio ──────────────────────────────────────
import json, os, subprocess, sys, time, urllib.request

PORT = 3900
BASE = f"http://127.0.0.1:{PORT}"
LOG_PATH = "/content/voicestudio_backend.log"

def health():
    try:
        with urllib.request.urlopen(f"{BASE}/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = health()
if not info:
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    env = os.environ.copy()
    env.update(OMNIVOICE_SERVER_MODE="1", PYTHONUNBUFFERED="1")
    env.setdefault("OMNIVOICE_DATA_DIR", "/content/omnivoice_data")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--app-dir", "backend",
         "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content/VoiceStudio", env=env,
        stdout=open(LOG_PATH, "ab"), stderr=subprocess.STDOUT)
    deadline = time.time() + 300
    while time.time() < deadline and proc.poll() is None and not info:
        print(".", end="", flush=True)
        time.sleep(3)
        info = health()
    print()
    if not info:
        raise SystemExit("O servidor não ligou em 5 min. Final do log:\n"
                         + "".join(open(LOG_PATH, errors="replace").readlines()[-40:]))
print("Servidor ligado:", info)

In [ ]:
# ── 4.5. Aceitar as licenças dos modelos (obrigatório antes de gerar) ───────
# Sem este passo o /generate falha. Rodar esta célula aceita as licenças em seu nome:
# OmniVoice é CC-BY-NC (uso NÃO comercial: o áudio não pode ser vendido).
import requests
BASE = "http://127.0.0.1:3900"

for repo in ("k2-fsa/OmniVoice", "eustlb/higgs-audio-v2-tokenizer"):
    st = requests.get(f"{BASE}/models/licenses/acceptance/{repo}", timeout=30).json()
    if st.get("accepted"):
        print(f"[ok] {repo} já aceito")
        continue
    r = requests.post(
        f"{BASE}/models/licenses/accept",
        json={"repo_id": repo, "fingerprint": st["fingerprint"], "accepted": True},
        timeout=30,
    )
    r.raise_for_status()
    print(f"[aceito] {repo} ({st.get('license')})")

## Roteiro

Cada trecho é `[personagem, texto, velocidade, pausa depois em segundos]` e, às vezes, um quinto item: o **efeito** que toca na pausa, só na versão B (`transicao`, `magia`, `agua`, `galope`, `porta`, `zumbido`, `festa`). Cada capítulo tem `"ambiente"`: `"riacho"` (água baixinha durante a fala) ou `"nenhum"`. Regras em `SONOPLASTIA.md`.

- **velocidade:** 1.0 é normal; até 1.1 na ação; 0.85 a 0.9 no suspense e no sono.
- **pausa:** o silêncio depois do trecho. Pausas maiores antes das revelações prendem a atenção.
- **voz** aceita só estes termos: `male`/`female`; `child`, `teenager`, `young adult`, `middle-aged`, `elderly`; `very low pitch` … `very high pitch`; `whisper`.
- A voz de cada personagem nasce da fala mais longa dele no **primeiro** capítulo em que aparece, e é a mesma em todos os outros.

Edite à vontade. Os trechos que você mudar são gerados de novo na célula 7.

In [ ]:
# ── 5. Roteiro (EDITE AQUI) ─────────────────────────────────────────────────
# "amostra" (opcional, em cada voz): caminho de um .wav de 5-10 s de alguém que
# AUTORIZOU por escrito o uso da voz. Nunca use a voz de um dublador ou artista.
VOZES = {'narrador': {'voz': 'male, young adult, moderate pitch', 'semente': 201},
 'narizinho': {'voz': 'female, child, high pitch', 'semente': 211},
 'principe': {'voz': 'male, teenager, high pitch', 'semente': 221},
 'major': {'voz': 'male, middle-aged, very low pitch', 'semente': 231},
 'aranha': {'voz': 'female, middle-aged, high pitch', 'semente': 241},
 'caramujo': {'voz': 'male, elderly, low pitch', 'semente': 251},
 'emilia': {'voz': 'female, young adult, very high pitch', 'semente': 261},
 'nastacia': {'voz': 'female, middle-aged, low pitch', 'semente': 271},
 'dona_benta': {'voz': 'female, elderly, moderate pitch', 'semente': 281},
 'rabico': {'voz': 'male, teenager, low pitch', 'semente': 301},
 'formiga': {'voz': 'female, young adult, high pitch', 'semente': 311},
 'pedrinho': {'voz': 'male, child, moderate pitch', 'semente': 291},
 'tom_mix': {'voz': 'male, young adult, low pitch', 'semente': 321},
 'libelula': {'voz': 'female, child, very high pitch', 'semente': 361},
 'abelha': {'voz': 'female, teenager, high pitch', 'semente': 351},
 'rainha': {'voz': 'female, middle-aged, moderate pitch', 'semente': 341},
 'visconde': {'voz': 'male, elderly, moderate pitch', 'semente': 371},
 'sardinha': {'voz': 'female, young adult, moderate pitch', 'semente': 381},
 'gato': {'voz': 'male, young adult, high pitch', 'semente': 391},
 'cinderela': {'voz': 'female, young adult, high pitch', 'semente': 401},
 'branca': {'voz': 'female, young adult, moderate pitch', 'semente': 411},
 'polegar': {'voz': 'male, child, high pitch', 'semente': 421},
 'aladim': {'voz': 'male, young adult, moderate pitch', 'semente': 431},
 'capinha': {'voz': 'female, child, high pitch', 'semente': 441},
 'fazdeconta': {'voz': 'male, teenager, moderate pitch', 'semente': 451},
 'vespa': {'voz': 'female, elderly, high pitch', 'semente': 461},
 'penninha': {'voz': 'male, child, moderate pitch', 'semente': 471},
 'lafontaine': {'voz': 'male, middle-aged, low pitch', 'semente': 481},
 'carneirinho': {'voz': 'male, child, high pitch', 'semente': 491},
 'lobo': {'voz': 'male, middle-aged, very low pitch', 'semente': 501},
 'cigarra': {'voz': 'female, young adult, high pitch', 'semente': 511},
 'formiga_velha': {'voz': 'female, elderly, low pitch', 'semente': 521},
 'esopo': {'voz': 'male, elderly, low pitch', 'semente': 531},
 'laura': {'voz': 'female, teenager, high pitch', 'semente': 541},
 'leao': {'voz': 'male, middle-aged, very low pitch', 'semente': 551},
 'raposa': {'voz': 'female, middle-aged, moderate pitch', 'semente': 561},
 'burro': {'voz': 'male, middle-aged, low pitch', 'semente': 571},
 'rei_macaco': {'voz': 'male, elderly, high pitch', 'semente': 581},
 'barao': {'voz': 'male, middle-aged, moderate pitch', 'semente': 591}}

CAPITULOS = {
    1: {"titulo": "Um peixinho no nariz", "ambiente": "riacho", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo um: Um peixinho no nariz.", 0.95, 1.2],
        ["narrador", "Numa casinha branca, no meio do mato, fica o Sítio do Picapau Amarelo.", 1.0, 0.5],
        ["narrador", "Lá mora uma senhora de óculos, muito boazinha, chamada Dona Benta.", 1.0, 0.5],
        ["narrador", "Dona Benta é avó de uma menina de sete anos. O nome dela é Lúcia.", 1.0, 0.6],
        ["narrador", "Mas ninguém chama a menina de Lúcia. Sabe por quê?", 1.0, 0.8],
        ["narrador", "Porque o nariz dela é pequenininho e virado para cima. Um nariz arrebitado!", 1.05, 0.5],
        ["narrador", "Por isso, todo mundo chama a menina de... Narizinho!", 1.0, 0.9],
        ["narrador", "No sítio também mora Tia Nastácia. Ela faz os melhores bolinhos do mundo e cuida de todos com muito carinho.", 1.0, 0.5],
        ["narrador", "Foi Tia Nastácia quem fez a melhor amiga de Narizinho: uma boneca de pano chamada Emília.", 1.0, 0.5],
        ["narrador", "Emília tem olhos de linha preta e sobrancelhas tão lá no alto que parece estar sempre espantada.", 1.0, 0.6],
        ["narrador", "Só tem um problema. Emília não fala. Nem uma palavrinha.", 0.95, 0.8],
        ["narrador", "Pelo menos... por enquanto.", 0.9, 1.4],
        ["narrador", "Atrás da casa passa um riacho de água bem clarinha. Narizinho adora ir lá.", 1.0, 0.5],
        ["narrador", "Toda tarde ela pega Emília no colo, senta na beira da água e joga migalhas de pão para os peixinhos.", 1.0, 0.5],
        ["narrador", "Os peixinhos já conhecem a menina. Quando ela chega, eles vêm nadando, todos juntos: plic, plic, plic!", 1.05, 0.8],
        ["narrador", "Naquela tarde, o sol estava quentinho. O vento balançava as folhas, bem devagar...", 0.9, 0.6],
        ["narrador", "E Narizinho foi ficando com sono... com sono...", 0.85, 0.8],
        ["narrador", "Até que fechou os olhos e dormiu.", 0.85, 1.8],
        ["narrador", "De repente, alguma coisa fez cócegas no nariz dela!", 1.1, 0.5],
        ["narrador", "Era um peixinho! Um peixinho pequeno, com roupa de escamas brilhantes e uma coroa dourada na cabeça.", 1.05, 0.5],
        ["narrador", "E sabe o que ele estava fazendo? Estava em pé, bem em cima do nariz de Narizinho!", 1.05, 0.7],
        ["principe", "Que montanha mais esquisita! É macia, é quentinha... e tem dois buracos que sopram vento!", 1.0, 0.7],
        ["narrador", "Narizinho ficou quietinha, quietinha, segurando a risada.", 0.95, 0.6],
        ["narrador", "Então, do meio da água, saiu um caranguejo enorme, com as garras abertas. Era o Major Agarra-e-Não-Larga-Mais, o guarda do príncipe.", 1.0, 0.5],
        ["major", "Alteza! Cuidado! Essa montanha pode ser perigosa!", 1.0, 0.5],
        ["principe", "Bobagem, Major. Montanha não faz mal a ninguém.", 1.0, 0.7],
        ["narrador", "Nessa hora, Narizinho não aguentou mais... e espirrou! Atchim!", 1.1, 0.4],
        ["narrador", "O peixinho saiu voando e caiu de costas na grama!", 1.1, 0.4],
        ["principe", "Socorro! A montanha está viva!", 1.1, 0.6],
        ["narizinho", "Calma! Eu não sou montanha. Eu sou uma menina! Meu nome é Narizinho.", 1.0, 0.7],
        ["narrador", "O peixinho se levantou, arrumou a coroa e fez uma reverência bem educada.", 1.0, 0.5],
        ["principe", "Muito prazer, menina. Eu sou o Príncipe Escamado, rei do Reino das Águas Claras.", 0.95, 0.4],
        ["principe", "Desculpe ter subido no seu nariz. Ele é tão bonitinho que eu pensei que fosse uma ilha!", 1.0, 0.7],
        ["narrador", "Narizinho riu. Ninguém nunca tinha elogiado o nariz dela daquele jeito.", 1.0, 0.6],
        ["narizinho", "E onde fica esse tal Reino das Águas Claras?", 1.0, 0.5],
        ["principe", "Lá no fundo do riacho. Tem palácio de conchas, peixes de todas as cores e muitas coisas que você nunca viu.", 1.0, 0.4],
        ["principe", "Quer conhecer?", 0.95, 1.0],
        ["narrador", "Narizinho olhou para a água. Depois olhou para Emília. Emília, como sempre, não disse nada.", 0.95, 0.6],
        ["narizinho", "Eu quero! Mas a Emília vai junto.", 1.05, 0.5],
        ["principe", "Então segure a minha nadadeira. E não solte por nada!", 1.05, 0.6],
        ["narrador", "Narizinho segurou a nadadeira do príncipe, apertou Emília contra o peito...", 1.0, 0.4],
        ["narrador", "E, tchibum! Mergulhou no riacho!", 1.1, 1.6],
        ["narrador", "Lá embaixo, tudo era azul e brilhante. E no palácio de conchas, Narizinho ia conhecer um médico muito esquisito: o Doutor Caramujo.", 0.9, 0.6],
        ["narrador", "O Doutor Caramujo guardava umas pílulas mágicas. E uma dessas pílulas ia mudar a vida da Emília para sempre...", 0.85, 1.0],
        ["narrador", "Mas isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    2: {"titulo": "O palácio de conchas", "ambiente": "riacho", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo dois: O palácio de conchas.", 0.95, 1.2],
        ["narrador", "No capítulo passado, Narizinho dormiu na beira do riacho e acordou com um peixinho de coroa em cima do nariz.", 1.0, 0.4],
        ["narrador", "Era o Príncipe Escamado. Ele convidou Narizinho para conhecer o Reino das Águas Claras. E ela mergulhou, com Emília no colo!", 1.0, 1.2],
        ["narrador", "Debaixo da água, Narizinho percebeu uma coisa incrível. Ela conseguia respirar! E nem estava molhada!", 1.05, 0.6],
        ["narrador", "Em volta dela passavam peixes amarelos, peixes azuis, peixes listrados. Uns grandes, outros do tamanho de um grão de feijão.", 1.0, 0.5],
        ["narrador", "E lá na frente, brilhando como um castelo de festa, estava o palácio de conchas.", 0.95, 0.8],
        ["narizinho", "Que lindo! Parece feito de pérolas!", 1.05, 0.6],
        ["principe", "Seja bem-vinda ao meu palácio, Narizinho. Aqui você é a minha convidada de honra.", 1.0, 0.7],
        ["narrador", "Na porta do palácio, quem estava de guarda? Ele mesmo: o Major Agarra-e-Não-Larga-Mais, com as garras para cima.", 1.0, 0.4],
        ["major", "Alto lá! Quem vem com o príncipe pode passar. Mas a boneca de pano fica de olho fechado!", 1.0, 0.6],
        ["narrador", "Narizinho riu. Emília não podia fechar os olhos, porque eles eram costurados com linha preta.", 1.0, 0.9],
        ["narrador", "Lá dentro, o príncipe levou Narizinho até uma salinha cheia de fios e agulhas.", 1.0, 0.4],
        ["narrador", "Era o ateliê da Dona Aranha, a costureira mais famosa do reino. Ela tinha oito patas, e cada pata segurava uma agulha diferente.", 1.0, 0.5],
        ["aranha", "Ora, ora, uma menina de verdade! Fique paradinha, que eu vou fazer um vestido de festa para você.", 1.0, 0.6],
        ["narrador", "E as oito patas começaram a costurar, depressa, depressa, depressa! Zip, zip, zip!", 1.1, 0.5],
        ["narrador", "Em um minuto, o vestido estava pronto. Era azul como o céu, com bolinhas de espuma do mar.", 1.0, 0.5],
        ["narizinho", "Dona Aranha, é o vestido mais bonito que eu já vi!", 1.05, 0.5],
        ["aranha", "Fiz com fio de teia e luz da lua. Cuidado para não sujar de lama!", 1.0, 1.0],
        ["narrador", "Depois, o príncipe disse que queria apresentar um amigo muito importante.", 1.0, 0.4],
        ["narrador", "Bateram numa porta pequenininha. Toc, toc, toc.", 0.95, 0.8],
        ["narrador", "A porta abriu bem devagar... e apareceu um caramujo de óculos, com uma maleta preta e cara de sabido.", 0.9, 0.5],
        ["caramujo", "Boa tarde, boa tarde. Eu sou o Doutor Caramujo, o médico do reino. Quem está doente aqui?", 0.95, 0.6],
        ["principe", "Ninguém está doente, doutor. Mas a boneca da Narizinho não fala.", 1.0, 0.5],
        ["caramujo", "Não fala? Hum... Deixe-me ver.", 0.95, 0.6],
        ["narrador", "O doutor olhou a boca de Emília com uma lupa. Olhou de um lado. Olhou do outro.", 0.95, 0.5],
        ["caramujo", "Já sei o que é. Esta boneca tem falta de fala. Mas eu tenho o remédio certo!", 1.0, 0.6],
        ["narrador", "Ele abriu a maleta preta e tirou um vidrinho cheio de bolinhas coloridas.", 1.0, 0.4],
        ["caramujo", "São as minhas pílulas falantes. Uma só, e qualquer um fala pelos cotovelos!", 1.0, 0.8],
        ["narrador", "Narizinho segurou Emília bem firme. O doutor pegou uma pílula e colocou na boca da boneca.", 0.95, 0.5],
        ["narrador", "Emília engoliu. Gulp!", 0.95, 1.2],
        ["narrador", "Todo mundo ficou olhando. O príncipe, a Dona Aranha, o Major, o doutor... e Narizinho, de olhos arregalados.", 0.9, 0.8],
        ["narrador", "Um segundo. Dois segundos. Três segundos...", 0.85, 1.0],
        ["narrador", "E então a boca de pano da Emília começou a mexer.", 0.85, 1.2],
        ["narrador", "Qual será a primeira palavra da Emília? Você vai descobrir no próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    3: {"titulo": "Emília fala!", "ambiente": "riacho", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo três: Emília fala!", 0.95, 1.2],
        ["narrador", "No capítulo passado, Narizinho ganhou um vestido da Dona Aranha e conheceu o Doutor Caramujo.", 1.0, 0.4],
        ["narrador", "O doutor deu para a Emília uma pílula falante. E a boca de pano da boneca começou a mexer...", 0.95, 1.2],
        ["emilia", "Estou com um gosto horrível de sapo na boca!", 1.05, 0.9],
        ["narrador", "Todo mundo levou um susto. A Emília estava falando!", 1.05, 0.5],
        ["narizinho", "Emília! Você fala! Você fala de verdade!", 1.1, 0.5],
        ["emilia", "Claro que falo. Eu sempre pensei um monte de coisas. Só faltava a boca funcionar!", 1.05, 0.6],
        ["narrador", "E a Emília não parou mais. Falava, falava, falava, igual a uma torneira aberta.", 1.05, 0.5],
        ["emilia", "Esse príncipe é muito molhado. Esse caranguejo é muito bravo. E esse doutor tem cara de quem come alface no café da manhã!", 1.1, 0.6],
        ["caramujo", "Hum. Acho que a pílula era forte demais.", 0.95, 0.6],
        ["narrador", "Narizinho ficou um pouco envergonhada. Mas também não conseguia parar de rir.", 1.0, 0.5],
        ["narizinho", "Emília, fale com educação! Eles são nossos amigos.", 1.0, 0.5],
        ["emilia", "Está bem, está bem. Muito obrigada pela pílula, senhor doutor. Mas da próxima vez, faça com gosto de goiabada!", 1.05, 0.9],
        ["narrador", "O príncipe achou tudo muito engraçado e resolveu fazer uma festa.", 1.0, 0.4],
        ["principe", "Hoje tem baile no palácio! Para comemorar a boneca que aprendeu a falar!", 1.05, 0.6],
        ["narrador", "Os peixes tocaram música com as bolhas: blub, blub, blub. Os camarões dançaram em fila. E até o Major mexeu as garras no ritmo.", 1.05, 0.5],
        ["narrador", "Narizinho dançou com o príncipe, de vestido azul. E Emília dançou sozinha, girando, girando, e falando sem parar.", 1.0, 0.9],
        ["narrador", "No meio da festa, de repente, Narizinho ouviu uma voz que vinha lá de cima. De muito longe.", 0.9, 0.8],
        ["nastacia", "Narizinho! Vem lanchar, menina! Tem bolinho quentinho!", 0.95, 0.8],
        ["narizinho", "É a Tia Nastácia! Preciso voltar para casa.", 1.0, 0.5],
        ["principe", "Já vai? Mas a festa mal começou!", 1.0, 0.5],
        ["narizinho", "Eu volto, prometo. Obrigada por tudo, príncipe!", 1.0, 0.8],
        ["narrador", "Narizinho fechou os olhos. Quando abriu de novo... estava deitada na grama, na beira do riacho.", 0.9, 0.8],
        ["narrador", "Será que tinha sido tudo um sonho?", 0.85, 1.0],
        ["narrador", "Ela olhou para a boneca no colo. E a boneca olhou de volta para ela.", 0.9, 0.8],
        ["emilia", "Que foi? Nunca viu boneca falar, não?", 1.05, 1.0],
        ["narrador", "Não tinha sido sonho! Narizinho saiu correndo para casa, com a Emília falando sem parar.", 1.1, 0.5],
        ["narrador", "E o que será que a Dona Benta e a Tia Nastácia vão dizer quando ouvirem a Emília falar?", 0.95, 0.6],
        ["narrador", "Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    4: {"titulo": "A boneca que fala", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo quatro: A boneca que fala.", 0.95, 1.2],
        ["narrador", "No capítulo passado, a Emília tomou a pílula falante do Doutor Caramujo e começou a falar sem parar.", 1.0, 0.4],
        ["narrador", "Depois, Narizinho voltou do Reino das Águas Claras para o sítio, com a Emília no colo.", 1.0, 1.0],
        ["narrador", "Narizinho entrou correndo em casa. Na sala estava Dona Benta, a avó dela, sentada na cadeirinha de balanço.", 1.0, 0.5],
        ["dona_benta", "Narizinho, tenho uma novidade! Seu primo Pedrinho vem passar uns tempos aqui no sítio.", 1.0, 0.5],
        ["narizinho", "O Pedrinho! Que bom, que bom, que bom!", 1.1, 0.5],
        ["narrador", "Narizinho deu três pulos de alegria. Pedrinho é um menino de dez anos, que mora na cidade com a mãe.", 1.0, 0.6],
        ["dona_benta", "Mas antes, arrume essa boneca. Ela está toda amassada e com o vestido molhado!", 1.0, 0.6],
        ["emilia", "A culpa não é minha, Dona Benta! Foi a Narizinho que me levou para o fundo do riacho.", 1.05, 1.0, "magia"],
        ["narrador", "Dona Benta levou um susto tão grande que quase caiu da cadeira. A boneca de pano estava falando!", 1.05, 0.5],
        ["dona_benta", "Tia Nastácia! Corra aqui! Venha ver uma coisa que nunca se viu!", 1.05, 0.5],
        ["narrador", "Tia Nastácia veio da cozinha, enxugando as mãos no avental.", 1.0, 0.4],
        ["nastacia", "Boneca falando, Dona Benta? Isso não existe. A Narizinho está brincando com a senhora.", 0.95, 0.6],
        ["emilia", "Não estou brincando coisa nenhuma! Eu falo, sim. O doutor me deu uma pílula, e agora vou falar a vida inteira!", 1.05, 0.7],
        ["narrador", "Tia Nastácia ficou de boca aberta. Ela mesma tinha costurado aquela boneca, com um paninho bem simples.", 1.0, 0.5],
        ["nastacia", "Credo! Se eu soubesse que ela ia falar, tinha feito a Emília de seda!", 1.0, 0.9],
        ["narrador", "No dia seguinte chegou uma carta da mãe de Pedrinho. Ele ia chegar dali a uma semana.", 1.0, 0.5],
        ["narizinho", "Uma semana inteira? Que demora!", 1.0, 0.5],
        ["narrador", "Enquanto isso, a Emília só queria ouvir histórias. Pedia a história de tudo: do tapete, do relógio, do armário.", 1.0, 0.5],
        ["emilia", "Dona Benta, conte a história do Pedrinho!", 1.05, 0.5],
        ["dona_benta", "Pedrinho não tem história, Emília. Ele nunca saiu de casa.", 1.0, 0.5],
        ["emilia", "Ora essa! O livro de capa vermelha também nunca saiu de casa. E tem mais de dez histórias dentro!", 1.05, 0.8],
        ["narrador", "Dona Benta riu. A Emília falava muita bobagem, mas sempre tinha uma resposta.", 1.0, 0.8],
        ["narrador", "Até agora: a Emília fala, Dona Benta e Tia Nastácia levaram um susto, e Pedrinho vai chegar daqui a uma semana.", 0.95, 1.0, "transicao"],
        ["narrador", "Para a semana passar mais depressa, Narizinho tinha uma coisa boa: era tempo de jabuticaba!", 1.05, 0.5],
        ["narrador", "Jabuticaba é uma fruta redonda e preta, do tamanho de uma bolinha de gude. Ela cresce grudada no tronco da árvore.", 1.0, 0.5],
        ["narrador", "A casca é lisinha. Dentro tem uma polpa branca, muito doce.", 1.0, 0.6],
        ["narrador", "Narizinho subia na jabuticabeira e escolhia as mais bonitas. Mordia, e fazia tloc! Engolia o caldo e cuspia o caroço: pluf!", 1.05, 0.5],
        ["narrador", "Lá embaixo esperava o Rabicó, um leitãozinho comilão, de rabinho bem curtinho. Ele pegava as cascas no chão: nhoc!", 1.05, 0.5],
        ["narrador", "E a música da jabuticabeira era assim: tloc, pluf, nhoc! Tloc, pluf, nhoc!", 1.05, 0.8],
        ["narrador", "Mas a árvore tinha outros fregueses: as vespas. As vespas furavam as frutas com o ferrão e entravam dentro, para chupar o doce.", 1.0, 0.6],
        ["narrador", "Uma tarde, Narizinho pôs na boca uma jabuticaba furadinha. Ela não viu que tinha uma vespa lá dentro.", 0.95, 0.8],
        ["narrador", "Por que será que, dessa vez, não fez tloc?", 0.95, 2.5],
        ["narrador", "Porque a vespa picou a ponta da língua dela! Ai, ai, ai!", 1.05, 0.6],
        ["narrador", "Tia Nastácia ouviu o grito e veio correndo. Sentou Narizinho no colo.", 1.0, 0.4],
        ["nastacia", "Calma, menina. Põe a língua para fora, que eu tiro o ferrão. Dói, mas passa.", 0.95, 0.6],
        ["narrador", "Com uma pinça, Tia Nastácia tirou o ferrãozinho, bem devagar. Narizinho chorou um pouco, de língua inchada. Depois a dor passou.", 0.95, 0.6],
        ["narrador", "Ela foi contar tudo para a Emília.", 1.0, 0.4],
        ["emilia", "Bem feito! Se fosse eu, olhava cada fruta com o binóculo da Dona Benta antes de comer.", 1.05, 0.8],
        ["narrador", "Narizinho não aguentou e deu uma gargalhada. E daí a pouco já estava na árvore de novo: tloc, pluf, nhoc!", 1.05, 1.0],
        ["narrador", "Resumindo: Narizinho levou uma picada de vespa, Tia Nastácia cuidou dela, e a Emília deu um conselho bem maluco.", 0.95, 0.8],
        ["narrador", "Mas naquela noite, Narizinho esqueceu a Emília debaixo da jabuticabeira, sozinha no escuro.", 0.9, 0.8],
        ["narrador", "E lá, a boneca viu uma coisa muito curiosa acontecer com aquela vespa...", 0.9, 1.0],
        ["narrador", "O que será que a Emília viu? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    5: {"titulo": "A condessa pescadora", "ambiente": "riacho", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo cinco: A condessa pescadora.", 0.95, 1.2],
        ["narrador", "No capítulo passado, a Emília deu um susto em Dona Benta e Tia Nastácia, porque agora ela fala.", 1.0, 0.4],
        ["narrador", "Depois, uma vespa picou a língua de Narizinho. E de noite, a menina esqueceu a Emília debaixo da jabuticabeira.", 1.0, 1.0],
        ["narrador", "Na hora de dormir, Narizinho lembrou da boneca.", 1.0, 0.4],
        ["narizinho", "Coitada da Emília! Deve estar com medo das corujas. Tia Nastácia, vai buscar a Emília para mim?", 1.0, 0.6],
        ["narrador", "Tia Nastácia foi e trouxe a boneca. A Emília estava molhada de orvalho, que são gotinhas de água que aparecem de noite nas folhas.", 1.0, 0.5],
        ["narrador", "E estava muito brava, porque tinha sido esquecida.", 1.0, 0.5],
        ["narizinho", "Desculpa, Emília. Eu faço um vestido novo para você. Cor-de-rosa, com bolinhas.", 1.0, 0.5],
        ["emilia", "Quero de saia bem comprida. Sujei o joelho e não quero que apareça.", 1.05, 0.5],
        ["narizinho", "É só lavar o joelho, ora.", 1.0, 0.5],
        ["emilia", "Nada disso! Por dentro eu sou recheada de macela. Se eu me molhar, fico mofada!", 1.05, 0.6],
        ["narrador", "Macela é uma florzinha seca, macia e cheirosa. Tia Nastácia usava macela para encher a boneca, como se fosse algodão.", 1.0, 0.6],
        ["emilia", "E um dia eu vou ser condessa. Quero ser a Condessa de Três Estrelinhas!", 1.05, 0.6],
        ["narrador", "Condessa é uma moça importante, quase uma princesa. Narizinho achou graça e pintou três estrelinhas na testa da boneca.", 1.0, 0.6],
        ["narizinho", "Pronto! De hoje em diante, você é a Condessa de Três Estrelinhas.", 1.0, 0.8],
        ["narrador", "Então a Emília contou o que viu lá fora, no escuro.", 1.0, 0.4],
        ["emilia", "A vespa que picou você ficou presa dentro da jabuticaba. E as formigas vieram buscar.", 1.0, 0.5],
        ["emilia", "Cada formiga pegou uma perninha e puxou, puxou, até o buraquinho onde elas moram. Era o enterro da vespa!", 1.05, 0.5],
        ["emilia", "Um besourinho quis fazer um discurso bem comprido. Mas as formigas ficaram com fome e foram embora sem ouvir!", 1.05, 0.8],
        ["narrador", "Narizinho riu tanto que até esqueceu da língua inchada.", 1.0, 1.0, "transicao"],
        ["narrador", "Até agora: a Emília virou condessa e contou o enterro da vespa. Agora vem uma pescaria!", 0.95, 0.8],
        ["narrador", "Uns dias depois, as jabuticabas acabaram. Tia Nastácia foi lavar roupa no riacho, e Narizinho foi junto, com a Emília no bolso do avental.", 1.0, 0.5],
        ["narizinho", "Tia Nastácia, faz um anzol para a Emília? Ela quer tanto pescar!", 1.0, 0.5],
        ["nastacia", "Era só o que faltava! Mas está bem, eu faço.", 1.0, 0.5],
        ["narrador", "Tia Nastácia dobrou um alfinete, que virou um anzolzinho. Amarrou numa linha e numa varinha do comprimento de uma régua.", 1.0, 0.5],
        ["narrador", "Narizinho amarrou a vara no braço da Emília. Depois sentou a boneca na beira da água, com uma pedra no colo para ela não cair.", 1.0, 0.5],
        ["narizinho", "Agora, Emília, fica quietinha. Se um peixe morder, você puxa a linha!", 1.0, 0.6],
        ["narrador", "Narizinho foi ajudar Tia Nastácia com a roupa. Mas, de repente... tchibum!", 1.05, 1.0, "agua"],
        ["narrador", "A Emília tinha caído dentro do riacho, com pedra e tudo!", 1.1, 0.5],
        ["narizinho", "Socorro, Tia Nastácia! A Emília está se afogando!", 1.1, 0.5],
        ["narrador", "Por que será que a Emília caiu na água?", 0.95, 2.5],
        ["narrador", "Porque um peixe mordeu o anzol e puxou a boneca para o meio do rio!", 1.05, 0.5],
        ["narrador", "Tia Nastácia pegou uma vara comprida e, com muito jeito, puxou a Emília para a beira.", 1.0, 0.5],
        ["narrador", "E, preso no anzol da boneca, vinha um peixe! Era uma traíra, um peixe comprido, de dentinhos pontudos, sacudindo o rabo sem parar.", 1.05, 0.6],
        ["nastacia", "Credo! Até parece mágica!", 1.0, 0.6],
        ["narrador", "Narizinho correu para casa com o peixe na mão.", 1.05, 0.4],
        ["narizinho", "Vovó, adivinha quem pescou esta traíra!", 1.05, 0.5],
        ["dona_benta", "Ora, foi você, minha filha.", 1.0, 0.4],
        ["narizinho", "Errou! Foi a Emília!", 1.1, 0.5],
        ["narrador", "Tia Nastácia chegou com a trouxa de roupa e confirmou: foi a boneca mesmo.", 1.0, 0.5],
        ["dona_benta", "Quanto mais a gente vive, mais aprende. Tenho mais de sessenta anos e todo dia esta neta me ensina uma coisa nova!", 1.0, 0.6],
        ["narrador", "E Tia Nastácia fritou a traíra para o jantar.", 1.0, 1.0],
        ["narrador", "Resumindo: a Condessa de Três Estrelinhas caiu no rio, mas pescou um peixe de verdade.", 0.95, 0.8],
        ["narrador", "Só que, depois do jantar, Narizinho lembrou: a Emília ainda estava encharcada!", 1.0, 0.6],
        ["narrador", "E, enquanto secava no sol, a Emília descobriu que entendia a língua das formigas...", 0.9, 1.0],
        ["narrador", "O que será que as formigas falam? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    6: {"titulo": "O presente da rainha das formigas", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo seis: O presente da rainha das formigas.", 0.95, 1.2],
        ["narrador", "No capítulo passado, a Emília virou a Condessa de Três Estrelinhas.", 1.0, 0.4],
        ["narrador", "Depois, ela caiu no riacho e pescou uma traíra. Mas ficou toda encharcada!", 1.0, 1.0],
        ["narrador", "Narizinho tirou a roupa molhada da Emília e deixou a boneca no sol, para secar.", 1.0, 0.5],
        ["emilia", "Eu não fico aqui sozinha! O Rabicó pode me comer. Tia Nastácia diz que ele come tudo o que encontra.", 1.05, 0.5],
        ["narizinho", "Está bem, eu fico com você. Mas só porque você pescou a traíra.", 1.0, 0.8],
        ["narrador", "O sol estava quente. No chão, passavam formiguinhas ruivas, quer dizer, de cor avermelhada. Iam e vinham, sem parar.", 1.0, 0.5],
        ["narizinho", "Que pena a gente não entender o que as formigas conversam.", 1.0, 0.5],
        ["emilia", "Você não entende. Eu entendo tudo!", 1.05, 0.5],
        ["narrador", "Duas formigas se encontraram e encostaram as anteninhas, que são dois fiozinhos que elas têm na cabeça.", 1.0, 0.5],
        ["emilia", "Uma disse: Achou o grilinho? A outra disse: Não. Então a primeira mandou procurar perto da pedra.", 1.05, 0.6],
        ["narrador", "Logo chegou uma terceira formiga, bem apressada.", 1.0, 0.4],
        ["emilia", "Essa aí disse que acharam uma minhoca gorda perto da porteira. E que precisam de ajuda para carregar.", 1.05, 0.6],
        ["narizinho", "Emília, você está inventando! Vou lá ver.", 1.0, 0.5],
        ["narrador", "Narizinho correu até a porteira. E não é que era verdade? Muitas formiguinhas puxavam juntas uma minhoca grande para o formigueiro.", 1.05, 0.6],
        ["narrador", "Por que será que as formigas trabalham todas juntas?", 0.95, 2.5],
        ["narrador", "Porque, sozinha, uma formiga não consegue carregar uma coisa tão pesada. Juntas, elas conseguem.", 1.0, 0.8],
        ["narrador", "Até agora: a Emília diz que entende a língua das formigas. E Narizinho começou a acreditar.", 0.95, 0.8],
        ["narrador", "De noite, as duas deitaram juntas na cama, como faziam desde que a Emília aprendeu a falar.", 1.0, 0.5],
        ["narizinho", "Emília, como é que você entende as formigas?", 1.0, 0.4],
        ["emilia", "Porque eu sou de pano.", 1.0, 0.5],
        ["narizinho", "Meu vestido também é de pano, e não entende nada!", 1.0, 0.4],
        ["emilia", "Então é porque eu sou de macela.", 1.0, 0.5],
        ["narizinho", "Este travesseiro também é de macela!", 1.0, 0.4],
        ["emilia", "Então... então eu não sei.", 0.95, 0.8],
        ["narrador", "Foi a primeira vez que a Emília ficou sem resposta. A primeira e a última!", 1.0, 1.0],
        ["narrador", "Mais tarde, quando as duas dormiam, alguém bateu na porta.", 0.9, 1.0, "porta"],
        ["narrador", "A porta abriu. Quem empurrava com o focinho era o Rabicó.", 1.0, 0.4],
        ["rabico", "Narizinho, tem uma senhora aqui que quer entrar.", 1.0, 0.5],
        ["narrador", "Entrou uma formiga de saiote vermelho e avental de renda. Na cabeça, ela equilibrava uma bandeja de prata.", 1.0, 0.5],
        ["formiga", "Boa noite. Trago um presente da Rainha das Formigas para a Senhora Condessa de Três Estrelinhas.", 1.0, 0.6],
        ["narrador", "Narizinho sentiu ciúme, porque o presente não era para ela. Era para a boneca! Ela acordou a Emília com uma cotovelada.", 1.0, 0.5],
        ["narrador", "Na bandeja havia croquetes. São bolinhos compridos e fritos, sequinhos por fora, de dar água na boca.", 1.0, 0.5],
        ["emilia", "Diga à Rainha que muito obrigada. E que os croquetes estão lindos!", 1.05, 0.5],
        ["narizinho", "Presente se paga com presente, condessa. Mande para a rainha uma perninha daquele pernilongo que eu peguei hoje.", 1.0, 0.5],
        ["narrador", "A Emília enfeitou a perninha do mosquito com um laço de fita e pôs na bandeja. A formiga agradeceu e foi embora, contente.", 1.0, 0.8],
        ["narizinho", "Eu não como isso, Emília. Aposto que é croquete de minhoca!", 1.0, 0.5],
        ["emilia", "E daí? Você comeu linguiça no jantar e nem reclamou!", 1.05, 0.6],
        ["narrador", "As duas discutiram, discutiram. E ninguém prestou atenção no Rabicó, que foi chegando de mansinho...", 0.95, 0.8],
        ["narrador", "Quando a Emília foi pegar um croquete, a bandeja estava vazia!", 1.05, 0.5],
        ["emilia", "Cadê os meus croquetes? Quero os meus croquetes!", 1.1, 0.5],
        ["narrador", "O Rabicó tinha comido tudo, de uma vez só! A Emília pegou a vassoura e correu atrás do leitão pelo quarto.", 1.1, 0.6],
        ["narrador", "E Narizinho rolava na cama de tanto rir.", 1.0, 1.0],
        ["narrador", "Resumindo: a rainha mandou croquetes para a Emília, mas o guloso do Rabicó comeu todos.", 0.95, 0.8],
        ["narrador", "Enquanto ria, Narizinho teve uma ideia maluca: casar a Emília com o Rabicó!", 1.0, 0.8],
        ["narrador", "Mas antes disso, chegou o grande dia. Um cavalinho apareceu na porteira, trazendo alguém muito esperado.", 0.95, 1.0, "galope"],
        ["narrador", "Quem será? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    7: {"titulo": "Pedrinho chegou!", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo sete: Pedrinho chegou!", 0.95, 1.2],
        ["narrador", "No capítulo passado, a Rainha das Formigas mandou croquetes para a Emília. Mas o Rabicó comeu todos.", 1.0, 0.4],
        ["narrador", "E Narizinho teve uma ideia: casar a Emília com o Rabicó.", 1.0, 1.0],
        ["narrador", "Um dia antes, chegou uma carta de Pedrinho para Dona Benta. Ele dizia como queria ser recebido.", 1.0, 0.5],
        ["narrador", "Narizinho devia esperar na porteira, com a Emília de vestido novo e o Rabicó com um laço de fita no rabo.", 1.0, 0.5],
        ["narrador", "E Tia Nastácia devia fazer café com aqueles bolinhos de frigideira que só ela sabe fazer.", 1.0, 0.6],
        ["narrador", "Por isso, Narizinho acordou bem cedo. Vestiu a Emília de cor-de-rosa com bolinhas e amarrou duas fitas no Rabicó.", 1.0, 0.8],
        ["narrador", "Então, na estrada, apareceu um cavalinho trotando.", 1.0, 1.0, "galope"],
        ["narrador", "Em cima dele vinha Pedrinho, queimado de sol e alegre como um passarinho.", 1.05, 0.5],
        ["narizinho", "Viva! Desce depressa, Pedrinho, que temos mil coisas para conversar!", 1.1, 0.5],
        ["narrador", "Pedrinho desceu, abraçou a prima e escondeu um pacote atrás das costas.", 1.0, 0.4],
        ["pedrinho", "Adivinha o que eu trouxe para você!", 1.05, 0.5],
        ["narizinho", "Já sei. Uma boneca que chora e que abre e fecha os olhos.", 1.0, 0.6],
        ["narrador", "Pedrinho ficou de boca aberta, porque era isso mesmo!", 1.0, 0.4],
        ["pedrinho", "Como você adivinhou?", 1.0, 0.5],
        ["narizinho", "Porque eu conheço você, bobo. As meninas são muito mais espertas que os meninos.", 1.0, 0.5],
        ["pedrinho", "Mas não têm tanto muque!", 1.05, 0.4],
        ["narrador", "Muque é a força dos braços. Pedrinho dobrou o braço para mostrar o músculo duro, que ele treinava na escola.", 1.0, 0.5],
        ["pedrinho", "Com o meu muque e a sua esperteza, Narizinho, ninguém pode com a gente!", 1.05, 0.8],
        ["narrador", "Depois, ele deu os outros presentes. O Rabicó ganhou uma fita de seda e os restos do lanche. Gostou mais do lanche, claro.", 1.0, 0.5],
        ["narrador", "A Emília ganhou uma cozinha inteirinha de brinquedo: fogãozinho de lata, panelinhas e até um rolo de abrir massa.", 1.0, 0.8],
        ["narrador", "Até agora: Pedrinho chegou de cavalinho e trouxe presentes para todos.", 0.95, 0.8],
        ["narrador", "Dona Benta apareceu na varanda e Pedrinho correu para abraçar a avó.", 1.0, 0.5],
        ["narrador", "Logo estavam todos na mesa, ouvindo as novidades da cidade. Tia Nastácia trouxe a massa dos bolinhos para a sala, para não perder nenhuma palavra.", 1.0, 0.6],
        ["dona_benta", "Agora vão brincar, vocês dois. A Narizinho tem um mundo de coisas para contar.", 1.0, 0.8, "transicao"],
        ["narrador", "Os primos foram para o pomar, o lugar dos segredos e dos planos de aventura.", 1.0, 0.5],
        ["pedrinho", "Quando você me escreveu que a Emília falava, eu não acreditei. Mas ela fala mesmo!", 1.05, 0.5],
        ["narizinho", "E já é condessa! Mas eu quero mais. Quero fazer o Rabicó virar marquês e casar a Emília com ele.", 1.0, 0.6],
        ["narrador", "Marquês é um título importante, como conde e condessa. Pedrinho achou a ideia ótima.", 1.0, 0.8],
        ["narrador", "Depois, Narizinho contou tudo sobre o Reino das Águas Claras: o Príncipe Escamado, o palácio de conchas, o Doutor Caramujo.", 1.0, 0.5],
        ["narrador", "Pedrinho ficou com muita vontade de conhecer o príncipe. E não estranhou nada, porque para os dois primos as maravilhas eram uma coisa natural.", 1.0, 0.6],
        ["pedrinho", "Eu tenho uma ideia. E se os personagens das histórias saíssem dos livros e viessem brincar aqui no sítio?", 1.05, 0.6],
        ["narizinho", "Que lindo! Eu queria brincar com a Chapeuzinho Vermelho e com a Branca de Neve!", 1.05, 0.5],
        ["pedrinho", "E eu queria encontrar o Aladim, aquele da lâmpada mágica!", 1.05, 0.6],
        ["narrador", "Por que será que Pedrinho queria os personagens aqui no sítio?", 0.95, 2.5],
        ["narrador", "Porque ele queria viver aventuras novas, junto com eles, em vez de só ouvir as histórias de sempre.", 1.0, 0.6],
        ["nastacia", "Narizinho! Pedrinho! O café está na mesa!", 1.0, 0.5],
        ["narrador", "E os dois saíram correndo para comer os bolinhos de Tia Nastácia.", 1.05, 1.0],
        ["narrador", "Resumindo: Pedrinho chegou, adorou a Emília falante e quer trazer os personagens das histórias para o sítio.", 0.95, 0.8],
        ["narrador", "Naquela noite, todos dormiram tarde. E bem cedinho, antes do sol nascer, alguém bateu no vidro da janela de Narizinho...", 0.9, 0.8],
        ["narrador", "Era um convite para uma viagem a um reino muito diferente.", 0.9, 0.8],
        ["narrador", "Que reino será esse? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    8: {"titulo": "O ouro da Emília", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo oito: O ouro da Emília.", 0.95, 1.2],
        ["narrador", "No capítulo passado, Pedrinho chegou ao sítio, com presentes para todos.", 1.0, 0.4],
        ["narrador", "E bem cedinho, alguém bateu na janela de Narizinho, com um convite de viagem.", 1.0, 1.0],
        ["narrador", "Narizinho abriu a janela. Lá fora estava o Rabicó, montado num cavalinho de pau.", 1.0, 0.5],
        ["rabico", "Pula da cama, Narizinho! O sol já vai nascer. Está na hora de partir.", 1.05, 0.5],
        ["narrador", "Agora o leitão tinha um título: era o Marquês de Rabicó. E a Emília já esperava lá embaixo, num cavalinho malhado.", 1.0, 1.0, "magia"],
        ["narrador", "Para onde eles iam? Para um reino de bichinhos voadores. No dia anterior, um marimbondo tinha trazido um convite.", 1.0, 0.5],
        ["narrador", "Mas o papel estava rasgado bem no nome. Era da Rainha das Abelhas ou da Rainha das Vespas? Ninguém sabia.", 1.0, 0.5],
        ["narrador", "Para responder, a Emília inventou o borboletograma. Ela escreveu a resposta na asa de uma borboleta azul e mandou a borboleta voar.", 1.0, 0.5],
        ["narizinho", "Por que você não pôs o nome do Pedrinho, Emília?", 1.0, 0.4],
        ["emilia", "Porque ele não é nobre. Não é nem barão!", 1.05, 0.8],
        ["narrador", "Narizinho vestiu o vestido vermelho de bolso, para levar os bolinhos de Tia Nastácia. Montou no cavalinho dela, e lá foram todos pela estrada.", 1.0, 1.0, "galope"],
        ["narizinho", "Vamos apostar corrida, Emília? Pode ir na frente!", 1.05, 0.5],
        ["narrador", "A Emília saiu galopando. Mas Narizinho ficou para trás de propósito. Por que será?", 0.95, 2.5],
        ["narrador", "Porque ela queria conversar sozinha com o Rabicó sobre o casamento.", 1.0, 0.5],
        ["narizinho", "Então, marquês, quer ou não quer casar com a condessa?", 1.0, 0.4],
        ["rabico", "Caso, sim. Se me derem dois carrinhos cheios de milho, eu caso com quem vocês quiserem!", 1.05, 0.6],
        ["narrador", "Que guloso! Quando alcançaram a Emília, ela estava furiosa, porque tinha caído na brincadeira da corrida.", 1.0, 0.8],
        ["narrador", "Até agora: os três viajam para o reino das abelhas, ou das vespas, e o Rabicó só pensa em comida.", 0.95, 0.8],
        ["narrador", "De repente, o mato da beira do caminho se mexeu. Os cavalinhos empinaram, assustados.", 0.95, 0.5],
        ["narrador", "Saiu do mato um bando de lagartos. Na frente vinha o chefe: o Tom Mix, um caubói de cinema, com chapéu de abas largas.", 1.0, 0.5],
        ["narrador", "E sabe o que o Marquês de Rabicó fez? Pulou do cavalo e fugiu correndo, sem ajudar ninguém!", 1.05, 0.6],
        ["tom_mix", "Passem para cá o dinheiro!", 1.0, 0.6],
        ["narrador", "Narizinho ficou com medo, mas pensou depressa.", 1.0, 0.4],
        ["narizinho", "Dinheiro nós não temos, senhor Tom Mix. Mas tenho uns bolinhos muito gostosos. Quer um?", 1.0, 0.5],
        ["tom_mix", "Bolinho de ontem? Não quero. Eu quero ouro!", 1.0, 0.6],
        ["narrador", "Ouro é um metal amarelo e brilhante, que vale muito. Narizinho teve uma ideia genial.", 1.0, 0.5],
        ["narizinho", "Está bem. Vou dar ao senhor um montinho de ouro bem amarelo. Me empresta a sua bolsa e uma tesourinha.", 1.0, 0.6],
        ["narrador", "Ela cochichou alguma coisa no ouvido da Emília.", 0.95, 0.5],
        ["emilia", "Nunca! Prefiro morrer!", 1.1, 0.6],
        ["narrador", "Mas Narizinho insistiu tanto que a Emília, chorando, esticou a perna no colo da amiga.", 0.95, 0.5],
        ["narrador", "Narizinho descosturou a perna da boneca e despejou toda a macela na bolsa do bandido. A macela é amarelinha, parece pó de ouro!", 1.0, 0.6],
        ["tom_mix", "Muito bem! De hoje em diante, sou seu amigo, princesa. Se estiver em perigo, é só chamar: Mix, Mix, Mix! E eu apareço.", 1.0, 0.6],
        ["narrador", "E o caubói foi embora com os lagartos. Narizinho soltou o ar, aliviada.", 1.0, 0.6],
        ["emilia", "Ai! Agora não consigo andar nem montar com esta perna vazia!", 1.05, 0.5],
        ["narizinho", "Você me salvou, Emília. Obrigada! Vem na garupa do meu cavalinho, bem agarradinha.", 1.0, 0.8],
        ["narrador", "Resumindo: o Rabicó fugiu, e a Emília salvou todo mundo dando o recheio da própria perna como se fosse ouro.", 0.95, 0.8],
        ["narrador", "Então Narizinho lembrou do Rabicó, e ficou brava por ele ter abandonado as amigas no perigo. Ela chamou: Mix, Mix, Mix!", 1.0, 0.5],
        ["narizinho", "Tom Mix, o Rabicó fugiu e deixou a gente sozinha. Pegue esse medroso e dê um bom susto nele, para ele aprender!", 1.0, 0.8],
        ["narrador", "Será que o Tom Mix vai encontrar o Rabicó? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    9: {"titulo": "No reino das abelhas", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo nove: No reino das abelhas.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o bandido Tom Mix parou Narizinho e a Emília na estrada. O Rabicó fugiu, com medo.", 1.0, 0.4],
        ["narrador", "A Emília deu a macela da perna como se fosse ouro. E Narizinho pediu ao Tom Mix um bom susto no Rabicó.", 1.0, 1.0],
        ["narrador", "O Tom Mix seguiu as pegadas do leitão até a floresta. E lá estava o Rabicó, com a cabeça enfiada dentro de uma abóbora enorme, comendo.", 1.0, 0.5],
        ["narrador", "O caubói chegou bem devagar, sem fazer barulho, e segurou o Rabicó pela perna.", 0.95, 0.5],
        ["rabico", "Coin, coin, coin! Me solta!", 1.1, 0.5],
        ["tom_mix", "Ordens da princesa do nariz arrebitado, senhor marquês. O senhor abandonou as amigas no perigo.", 1.0, 0.5],
        ["rabico", "Me dá cinco minutos? Preciso escrever uma carta.", 1.0, 0.5],
        ["narrador", "Uma libélula ia passando. Libélula é um inseto comprido, de quatro asas transparentes, que voa muito depressa.", 1.0, 0.5],
        ["rabico", "Amiga libélula, leve esta carta para a princesa Narizinho. Em troca, eu te dou um lindo lago azul!", 1.05, 0.5],
        ["narrador", "E a libélula saiu voando, rápida como um pensamento.", 1.05, 1.0, "transicao"],
        ["narrador", "Enquanto isso, Narizinho e a Emília chegavam ao palácio das abelhas. Era uma colmeia gigante, feita de cera, cheia de salinhas de seis lados.", 1.0, 1.0, "zumbido"],
        ["narrador", "Os zangões, que são as abelhas machos, vieram receber as duas. Eles contaram que o convite era mesmo da Rainha das Abelhas.", 1.0, 0.5],
        ["narrador", "E a Rainha das Vespas estava furiosa com Narizinho, por causa daquela vespa da jabuticaba. Ainda bem que elas não erraram o caminho!", 1.0, 0.6],
        ["narrador", "No palácio estava também um visitante conhecido: o Doutor Caramujo. Ele examinou a perna vazia da Emília.", 1.0, 0.5],
        ["caramujo", "Caso grave! Esta perna precisa de um tratamento muito científico.", 0.95, 0.5],
        ["narizinho", "Que nada, doutor. Tia Nastácia põe macela nova, costura com agulha e linha, e pronto.", 1.0, 0.5],
        ["caramujo", "Tia Nastácia? Uma curandeira qualquer! Isso não é ciência!", 0.95, 0.5],
        ["narizinho", "Não fale assim da Tia Nastácia! Ela cuida de todo mundo lá no sítio, e cuida muito bem. Peça desculpas!", 1.05, 0.6],
        ["narrador", "O Doutor Caramujo ficou envergonhado e foi embora de fininho. E os zangões arranjaram duas muletinhas para a Emília andar.", 1.0, 0.8],
        ["narrador", "Depois, Narizinho encontrou o Major Agarra, que tinha vindo do Reino das Águas Claras.", 1.0, 0.4],
        ["major", "O príncipe anda muito triste, menina. Não quer casar com nenhuma princesa. Dizem que ele gosta de alguém... mas ninguém sabe quem é.", 0.95, 0.6],
        ["narrador", "O coração de Narizinho bateu mais depressa. Por que será?", 0.95, 2.5],
        ["narrador", "Porque ela desconfiou que esse alguém podia ser ela mesma!", 1.0, 0.5],
        ["narizinho", "Major, diga ao príncipe que eu estou bem. E que um dia vou convidar ele para visitar o sítio.", 1.0, 0.8],
        ["narrador", "Até agora: o Tom Mix pegou o Rabicó, a Emília ganhou muletas, e o príncipe está triste de saudade.", 0.95, 0.8],
        ["narrador", "Nessa hora chegou a libélula com a carta do Rabicó. A letra era torta e as palavras estavam todas erradas.", 1.0, 0.5],
        ["narrador", "A carta dizia: Desculpe a minha covardia. O Tom Mix me pegou. Tenha dó deste infeliz.", 0.95, 0.6],
        ["narizinho", "Esse Rabicó! Está bem, eu perdoo. Leve este bilhete para o Tom Mix, depressa!", 1.0, 0.5],
        ["narrador", "O Tom Mix leu o perdão e soltou o Rabicó. Mas a libélula cobrou o lago azul.", 1.0, 0.5],
        ["rabico", "Lago? Você pode se afogar num lago. Tome esta sementinha de abóbora, que é melhor!", 1.05, 0.5],
        ["narrador", "A libélula ficou furiosa, porque promessa é promessa.", 1.0, 0.8],
        ["narrador", "No palácio, uma abelhinha explicou como era a vida da colmeia.", 1.0, 0.4],
        ["abelha", "Aqui ninguém precisa mandar. Cada uma já nasce sabendo o seu trabalho. Todas buscam o mel das flores e cuidam da casa.", 1.0, 0.5],
        ["abelha", "E a rainha não é uma rainha emproada. Ela é a mãe de todas nós.", 1.0, 0.6],
        ["narrador", "Então Narizinho conheceu a rainha, sentada num trono de cera.", 1.0, 0.4],
        ["rainha", "Seja bem-vinda, menina. Leve todo o mel que quiser. Aqui o dinheiro não vale nada.", 0.95, 0.6],
        ["narrador", "Narizinho escolheu mel de flor de jabuticabeira, para a avó, e cera bem branquinha, para Tia Nastácia.", 1.0, 0.5],
        ["narrador", "O Tom Mix chegou com uma fila de grilos, cada um carregando dois barrilzinhos nas costas. E encheram os barris de mel.", 1.0, 1.0],
        ["narrador", "Resumindo: Narizinho perdoou o Rabicó, conheceu a rainha das abelhas e ganhou mel para levar para casa.", 0.95, 0.8],
        ["narrador", "Mas na volta, a Emília, que ia na frente dentro de um barril vazio, ficou de pé e gritou:", 0.95, 0.5],
        ["emilia", "Estou vendo um monstro lá na frente! Com cabeça de porco e pés de tartaruga!", 1.1, 0.8],
        ["narrador", "Que monstro será esse? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    10: {"titulo": "O feitiço da velha", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo dez: O feitiço da velha.", 0.95, 1.2],
        ["narrador", "No capítulo passado, Narizinho visitou o reino das abelhas e ganhou mel. O Tom Mix soltou o Rabicó.", 1.0, 0.4],
        ["narrador", "Na volta, a Emília viu um monstro com cabeça de porco e pés de tartaruga.", 1.0, 1.0],
        ["narrador", "O Tom Mix foi na frente para ver o que era. Logo voltou, rindo.", 1.0, 0.5],
        ["tom_mix", "Não é monstro nenhum, princesa! É o marquês, montado num jabuti!", 1.05, 0.5],
        ["narrador", "Jabuti é uma tartaruga que vive na terra. Anda muito devagar e tem um casco duro nas costas.", 1.0, 0.5],
        ["narrador", "E o Rabicó gritava com o coitado do jabuti, para ele andar mais depressa.", 1.0, 0.5],
        ["rabico", "Anda, jabuti preguiçoso! Mais depressa!", 1.05, 0.6],
        ["narrador", "Narizinho ficou indignada, porque o Rabicó estava maltratando um bicho mais fraco que ele.", 1.0, 0.4],
        ["narizinho", "Desce já daí, Rabicó! Agora é a sua vez de carregar. Senhor jabuti, pode montar no marquês!", 1.05, 0.6],
        ["narrador", "O jabuti, bem devagarinho, subiu nas costas do Rabicó. E o Rabicó teve que carregar o jabuti pelo caminho.", 1.0, 0.5],
        ["emilia", "Bem feito! Isso é por ter comido os meus croquetes!", 1.1, 0.5],
        ["narrador", "E uma vozinha fina passou voando lá no alto. Era a libélula.", 1.0, 0.4],
        ["libelula", "E isso é pelo meu lago azul!", 1.1, 0.8],
        ["narrador", "Nesse dia, o Rabicó perdeu pelo menos um quilo e aprendeu um pouco a respeitar os outros.", 1.0, 1.0, "transicao"],
        ["narrador", "Perto do sítio, todos pararam para almoçar debaixo de uma figueira. O almoço era só mel.", 1.0, 0.5],
        ["narrador", "Então um passarinho cantou no galho. Narizinho olhou bem para ele.", 1.0, 0.4],
        ["narizinho", "Emília, esse passarinho não tem a cara do Pedrinho?", 1.0, 0.5],
        ["narrador", "O passarinho desceu e pousou no ombro dela.", 1.0, 0.4],
        ["pedrinho", "Sou eu mesmo, Narizinho! Apareceu lá no sítio uma velha de bengala e cesta no braço. Ela perguntou pela menina do nariz arrebitado.", 1.0, 0.5],
        ["pedrinho", "Eu respondi mal para ela. Então ela me virou passarinho! E virou a vovó numa tartaruga, e a Tia Nastácia numa galinha!", 1.05, 0.6],
        ["narizinho", "Que horror! O que vai ser de nós agora?", 1.05, 0.6],
        ["tom_mix", "Deixe comigo, princesa. Eu vou encontrar essa velha e trazer ela de volta, para desfazer o feitiço.", 1.0, 0.8],
        ["narrador", "Até agora: o Rabicó levou uma lição, mas uma velha transformou a família de Narizinho em bichos.", 0.95, 0.8],
        ["narrador", "Narizinho voltou para o sítio, muito triste, com o passarinho Pedrinho no ombro.", 0.95, 0.5],
        ["narrador", "Na cozinha, uma galinha cacarejava. Na sala, uma tartaruga de óculos costurava na rede.", 0.95, 0.5],
        ["narizinho", "Vovó! Não me conhece mais, vovó?", 0.95, 0.8],
        ["narrador", "A tartaruga ficou quieta, quieta. Narizinho chorou, abraçada com a Emília.", 0.9, 0.8],
        ["narrador", "Passou um dia. Passaram dois dias. E nada do Tom Mix.", 0.9, 0.6],
        ["emilia", "Calma, Narizinho. A conta para tudo é três. No terceiro dia ele aparece.", 1.0, 0.8],
        ["narrador", "No terceiro dia, as duas ficaram na janela, olhando a estrada. Por que será que a Emília tinha tanta esperança?", 0.95, 2.5],
        ["narrador", "Porque o Tom Mix tinha prometido ajudar. E a Emília achava que ele era como nos filmes, onde o herói sempre chega no fim.", 1.0, 0.6],
        ["narrador", "De repente, um cachorrinho latiu no terreiro. Era o Tom Mix! E na frente dele vinha a velha, de braços para cima.", 1.05, 0.5],
        ["tom_mix", "Agora você vai desfazer o feitiço que fez!", 1.0, 0.6],
        ["narrador", "Narizinho fechou os olhos, com medo da cara da velha. Depois foi abrindo, bem devagarinho...", 0.85, 1.2, "magia"],
        ["narrador", "E quem ela viu? A Tia Nastácia, de verdade, sorrindo para ela!", 1.0, 0.4],
        ["nastacia", "Acorda, menina! Parece que você estava sonhando.", 0.95, 0.6],
        ["narrador", "Narizinho sentou na cama, esfregando os olhos. Tudo tinha sido um sonho!", 1.0, 0.5],
        ["narizinho", "E a vovó? E o Pedrinho? E o Tom Mix?", 1.05, 0.5],
        ["nastacia", "A sua avó está lá dentro, costurando. O Pedrinho está brincando no quintal. E deixe de bobagem, que o café está esfriando!", 1.0, 1.0],
        ["narrador", "Resumindo: a viagem ao reino das abelhas, o bandido e o feitiço foram um sonho comprido de Narizinho.", 0.95, 0.8],
        ["narrador", "Mas uma coisa não era sonho: Narizinho ainda queria casar a Emília com o Rabicó. E a Emília não queria de jeito nenhum!", 1.0, 0.6],
        ["narrador", "Como será que Narizinho vai convencer a boneca? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    11: {"titulo": "O Visconde de Sabugosa", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo onze: O Visconde de Sabugosa.", 0.95, 1.2],
        ["narrador", "No capítulo passado, Narizinho sonhou uma aventura comprida: o reino das abelhas, o bandido Tom Mix e o feitiço da velha.", 1.0, 0.4],
        ["narrador", "Quando acordou, ainda tinha um plano: casar a Emília com o Rabicó.", 1.0, 1.0],
        ["narrador", "O Rabicó era um leitãozinho ruivo, com manchas brancas e um rabinho bem curto. Por isso o nome dele era Rabicó.", 1.0, 0.5],
        ["narrador", "No sítio, quando os leitões ficavam gordinhos, Tia Nastácia levava para a cozinha, para virar comida. Mas o Rabicó tinha uma amiga que protegia ele: Narizinho.", 1.0, 0.6],
        ["narrador", "Uma tarde, Narizinho ouviu Dona Benta falar com Tia Nastácia.", 1.0, 0.4],
        ["dona_benta", "Amanhã é aniversário do Pedrinho. Vamos fazer um jantar especial, com leitão assado.", 1.0, 0.5],
        ["narrador", "Narizinho entendeu tudo e correu atrás do Rabicó. Por que será que ela correu?", 0.95, 2.5],
        ["narrador", "Porque o único leitão gordinho no sítio era o Rabicó! E ela não queria que o amigo virasse jantar.", 1.0, 0.5],
        ["narizinho", "Vem comigo, Rabicó. Vou esconder você num lugar que só eu sei. E nada de barulho!", 1.0, 0.5],
        ["narrador", "Ela deixou o leitão escondido, com milho, uma abóbora e uma lata de água. Tia Nastácia procurou o sítio inteiro e não achou.", 1.0, 0.5],
        ["narrador", "No jantar do aniversário, comeram galinha. E no dia seguinte, o Rabicó voltou para o terreiro, todo tranquilo.", 1.0, 0.5],
        ["nastacia", "Olha só quem apareceu! Escapou desta vez, seu danadinho!", 1.0, 1.0, "transicao"],
        ["narrador", "Agora, Narizinho podia cuidar do casamento.", 1.0, 0.4],
        ["narizinho", "Senhora condessa, está na hora de casar. Que tal o Marquês de Rabicó?", 1.0, 0.5],
        ["emilia", "Nunca! Ele é um medroso. Fugiu quando o bandido apareceu!", 1.05, 0.6],
        ["narizinho", "Mas você não sabe o segredo. O Rabicó é um príncipe! Uma fada má virou ele em porco.", 1.0, 0.5],
        ["narizinho", "Ele só vai virar príncipe de novo quando achar um anel mágico, escondido na barriga de uma minhoca. Por isso ele vive cavando a terra!", 1.0, 0.6],
        ["narrador", "Era tudo invenção de Narizinho. Mas a Emília sonhava ser princesa. E pensou, pensou, pensou.", 0.95, 0.5],
        ["emilia", "Está bem, eu aceito. Mas só vou morar com ele quando ele virar príncipe!", 1.05, 0.8],
        ["narrador", "Narizinho correu até o pomar, onde estava Pedrinho.", 1.05, 0.4],
        ["narizinho", "Pedrinho, depressa! Faz um visconde de sabugo, com cartola e uma marca de coroa na testa. Ele vai ser o pai do Rabicó!", 1.05, 0.6],
        ["narrador", "Sabugo é o miolo da espiga de milho, depois que tiram os grãos. É comprido e duro, do tamanho de uma mão.", 1.0, 0.5],
        ["narrador", "Pedrinho pegou um sabugo e fez braços, pernas e cara. As palhinhas do milho viraram barba. E na cabeça, uma cartolinha, que é um chapéu alto e redondo.", 1.0, 0.8],
        ["narrador", "Até agora: Narizinho salvou o Rabicó, enganou a Emília, e Pedrinho fez um visconde de sabugo de milho.", 0.95, 0.8],
        ["narrador", "Pedrinho levou o visconde até o quarto da Emília e bateu na porta.", 1.0, 0.8, "porta"],
        ["visconde", "Sou o Visconde de Sabugosa. Venho pedir a mão da Condessa de Três Estrelinhas para o meu filho, o Marquês de Rabicó.", 0.95, 0.6],
        ["narizinho", "Entre, senhor visconde. E como vai a sua senhora mãe, Dona Palha-de-Milho?", 1.0, 0.5],
        ["visconde", "Coitada. Foi comida pela vaca.", 0.9, 0.6],
        ["narizinho", "Que pena. Mas a vida é assim: a vaca come o milho, e a gente come... bom, deixa para lá!", 1.0, 0.8],
        ["narrador", "A Emília entrou, de vestido de bolinhas. O visconde tirou a cartola para cumprimentar, e ela viu a marca de coroa na testa dele.", 1.0, 0.5],
        ["narizinho", "A Emília sabe fazer tudo, visconde. Até este vestido foi ela que costurou!", 1.0, 0.5],
        ["emilia", "Não fui eu, não. Foi a Tia Nastácia.", 1.0, 0.6],
        ["narrador", "A Emília ainda não sabia mentir. E isso é uma coisa boa!", 1.0, 0.5],
        ["visconde", "O meu filho tem bom gênio. Gosta de dormir no sol e de cavar a terra atrás de minhocas. O único defeito é comer tudo o que encontra.", 0.95, 0.5],
        ["emilia", "Comigo, ele só vai comer coisas limpinhas e cheirosas!", 1.05, 0.5],
        ["narizinho", "Então, Emília? Aceita ou não aceita o Marquês de Rabicó?", 1.0, 0.8],
        ["narrador", "A Emília pensou, pensou, e respondeu:", 0.95, 0.6],
        ["emilia", "Quero!", 1.05, 1.0, "festa"],
        ["narrador", "O visconde deu um beijo na testa da noiva. E a Emília, vermelhinha de vergonha, saiu correndo para o quarto.", 1.0, 1.0],
        ["narrador", "Resumindo: a Emília aceitou casar com o Rabicó, porque acha que ele é um príncipe encantado.", 0.95, 0.8],
        ["narrador", "Agora vem o noivado, com doces e festa. Mas o Rabicó é muito guloso. Será que ele vai se comportar?", 1.0, 0.6],
        ["narrador", "Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    12: {"titulo": "O casamento da Emília", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo doze: O casamento da Emília.", 0.95, 1.2],
        ["narrador", "No capítulo passado, Narizinho contou para a Emília que o Rabicó era um príncipe encantado.", 1.0, 0.4],
        ["narrador", "Pedrinho fez o Visconde de Sabugosa, de sabugo de milho. E a Emília aceitou o pedido de casamento.", 1.0, 1.0],
        ["narrador", "Começou o noivado. Noivado é o tempo antes do casamento, quando os noivos se visitam.", 1.0, 0.5],
        ["narrador", "Toda tarde, Pedrinho trazia o Rabicó para visitar a noiva. Mas o leitão só cheirava o chão, procurando comida. Nem ouvia a conversa!", 1.0, 0.5],
        ["pedrinho", "Assim não dá. O Rabicó não sabe brincar. Vou arranjar alguém para ficar no lugar dele.", 1.0, 0.5],
        ["narrador", "Pedrinho achou no quintal um vidro azul, vazio, e pôs uma cartolinha nele. Era o Senhor Vidro Azul, que ia namorar no lugar do noivo.", 1.0, 0.6],
        ["narrador", "E o Rabicó voltou para o pomar, para cavar a terra atrás da tal minhoca do anel.", 1.0, 0.8],
        ["pedrinho", "O marquês mandou construir um castelo de ouro para a noiva. Com um lago na frente!", 1.0, 0.5],
        ["emilia", "Um lago? Aposto que é uma lata de água!", 1.1, 0.6],
        ["narrador", "Pedrinho, fazendo a voz do Vidro Azul, leu uns versinhos que o marquês tinha mandado:", 1.0, 0.5],
        ["pedrinho", "Pirolito que bate, bate. Pirolito que já bateu. Quem adora o marquês é ela. Quem adora a Emília sou eu.", 0.95, 0.8],
        ["emilia", "Está errado! Eu vou casar, mas não adoro ninguém. Já pensou, adorar um leitão?", 1.05, 0.8],
        ["narrador", "Uma semana depois, Narizinho pediu a Dona Benta umas moedinhas para comprar os doces da festa. E Dona Benta deu.", 1.0, 0.8, "transicao"],
        ["narrador", "Até agora: o noivo comilão foi trocado por um vidro azul, e a Emília continua bem desconfiada.", 0.95, 0.8],
        ["narrador", "Chegou o dia do casamento. A mesa ficou debaixo de uma laranjeira, com seis cocadas, seis pés-de-moleque e uma rapadura.", 1.0, 0.5],
        ["narrador", "Os convidados eram de mentirinha: pedras, tijolos e pedaços de pau, cada um fazendo o papel de uma pessoa.", 1.0, 0.5],
        ["narrador", "A Emília veio de vestido branco e véu. O Rabicó veio de cartola, com uma faixa de seda no pescoço.", 1.0, 0.5],
        ["narrador", "Mas quando o Rabicó sentiu o cheiro das cocadas, ficou com água na boca e não viu mais nada.", 1.0, 0.5],
        ["narrador", "Os noivos se casaram. Narizinho pôs um anelzinho no braço da Emília, porque a boneca não tem dedos.", 1.0, 1.0, "festa"],
        ["narrador", "Pedrinho pôs no Rabicó uma aliança de casca de laranja. E o Rabicó tentou comer a aliança duas vezes!", 1.05, 0.6],
        ["narrador", "Depois, os primos se distraíram conversando. E ninguém vigiou a mesa dos doces...", 0.95, 0.8],
        ["narrador", "O que será que o Rabicó fez?", 0.95, 2.5],
        ["narrador", "Nhoc! Ele abocanhou a cocada mais bonita e saiu correndo pelo campo, com a cocada na boca!", 1.1, 0.5],
        ["narrador", "Pedrinho correu atrás dele, furioso. A festa virou uma bagunça.", 1.05, 0.5],
        ["emilia", "Eu sabia! Eu não queria casar com esse guloso! Ele não respeita a esposa!", 1.1, 0.5],
        ["narizinho", "Calma, Emília. Um dia ele vira príncipe, lembra?", 1.0, 0.5],
        ["pedrinho", "Que príncipe nada! A Narizinho enganou você. O Rabicó é porco e vai ser porco para sempre.", 1.05, 0.6],
        ["narrador", "Quando ouviu isso, a Emília caiu para trás, desmaiada.", 1.0, 1.0],
        ["narrador", "Desde esse dia, a Emília ficou casada com o Rabicó, mas cada um morando no seu canto. Ela nunca perdoou a mentira.", 1.0, 0.8, "transicao"],
        ["narrador", "Chegou o Ano-Bom, que é o primeiro dia do ano. Dona Benta fez um jantar grande, com muitos pratos.", 1.0, 0.5],
        ["narrador", "E de repente, na mesa, apareceu um leitão assado numa travessa.", 0.95, 0.6],
        ["narrador", "Narizinho procurou o Rabicó no terreiro inteiro e não achou. Voltou chorando.", 0.95, 0.4],
        ["narizinho", "Não come esse leitão, Pedrinho! É o Rabicó!", 1.05, 0.6],
        ["narrador", "Pedrinho ficou com os olhos cheios de lágrimas. Mas a Emília bateu palmas!", 1.0, 0.4],
        ["emilia", "Estou viúva! Agora posso casar com o visconde!", 1.1, 0.6],
        ["narrador", "Viúva é quem perdeu o marido. Narizinho ficou tão brava que deu um peteleco na boneca.", 1.0, 0.5],
        ["narizinho", "Vou pedir ao Doutor Caramujo para pôr nela o que está faltando!", 1.05, 0.4],
        ["dona_benta", "E o que está faltando na Emília, minha filha?", 1.0, 0.5],
        ["narizinho", "Coração, vovó! A Emília não tem nem um pedacinho de coração!", 1.05, 0.8],
        ["narrador", "Mas quantas lágrimas à toa! O leitão da travessa não era o Rabicó.", 1.0, 0.5],
        ["narrador", "O espertinho tinha fugido para o mato na véspera. E depois voltou para o terreiro, muito tranquilo, como se nada tivesse acontecido.", 1.0, 1.0],
        ["narrador", "Resumindo: o casamento da Emília deu errado, e o Rabicó escapou de novo do forno.", 0.95, 0.8],
        ["narrador", "Enquanto isso, lá no fundo do riacho, o Príncipe Escamado estava doente de tristeza. E ele ia mandar um pedido muito importante para Narizinho...", 0.9, 0.8],
        ["narrador", "Que pedido será? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    13: {"titulo": "Uma carta dentro da concha", "ambiente": "riacho", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo treze: Uma carta dentro da concha.", 0.95, 1.2],
        ["narrador", "No capítulo passado, a Emília casou com o Rabicó, mas o casamento deu errado. E o Rabicó escapou de novo do forno.", 1.0, 0.4],
        ["narrador", "Enquanto isso, no fundo do riacho, o Príncipe Escamado estava doente de tristeza.", 1.0, 1.0, "agua"],
        ["narrador", "Desde que Narizinho foi embora, o príncipe não comia, não dormia e só suspirava. Ficou tão magro que as escamas pareciam papel.", 0.95, 0.5],
        ["narrador", "Ele passava horas olhando para o trono onde Narizinho tinha sentado no dia do baile.", 0.95, 0.6],
        ["narrador", "Então chamaram o Doutor Caramujo. O doutor olhou a língua do príncipe e mediu o pulso.", 1.0, 0.5],
        ["caramujo", "Vossa Majestade está com narizinho-arrebitadite. É uma doença muito séria. O único remédio é casar com uma certa menina.", 0.95, 0.6],
        ["principe", "Você tem razão, doutor. Minha doença não é do corpo. É saudade da Narizinho!", 0.95, 0.6],
        ["caramujo", "Então deixe comigo, Majestade. Vou pedir a mão dela em casamento.", 1.0, 0.6],
        ["narrador", "O príncipe prometeu: se desse certo, o doutor ia virar duque, o Duque da Pílula! O Doutor Caramujo saiu todo contente.", 1.0, 0.8],
        ["narrador", "Quem escreveu a carta foi a Dona Lula. A lula é um bicho do mar que tem tinta dentro do corpo. Por isso era a escrevente do reino.", 1.0, 0.5],
        ["narrador", "A carta foi fechada dentro de uma concha, para não molhar. E uns peixinhos escoteiros levaram a concha, rolando pelo fundo do riacho.", 1.0, 0.6],
        ["narrador", "Por que será que a carta foi dentro de uma concha?", 0.95, 2.5],
        ["narrador", "Porque, embaixo da água, o papel ia molhar e a tinta ia apagar!", 1.0, 1.0, "transicao"],
        ["narrador", "Até agora: o príncipe está doente de saudade, e uma carta dentro de uma concha está viajando até o sítio.", 0.95, 0.8],
        ["narrador", "Os peixinhos deixaram a concha entre duas pedras, na beira do riacho, perto do pé de ingá.", 1.0, 0.5],
        ["narrador", "Logo apareceu Pedrinho, com a vara de pescar. Ele viu a concha, sacudiu perto do ouvido e escutou um barulhinho de papel lá dentro.", 1.0, 0.5],
        ["pedrinho", "Uma carta! Narizinho! Tem uma carta para você!", 1.1, 0.5],
        ["narrador", "Narizinho estava na cozinha, ajudando Tia Nastácia a enrolar rosquinhas de polvilho. Ela limpou as mãos e leu a carta.", 1.0, 0.5],
        ["narrador", "A carta dizia: Nosso príncipe está doente de saudade. A menina aceita casar com ele e salvar o Reino das Águas Claras?", 0.95, 0.8],
        ["narizinho", "Pedrinho, responde que sim! Assim que acabar estas rosquinhas, eu vou casar com ele.", 1.05, 0.6],
        ["dona_benta", "Casar com quem, menina? Que história é essa?", 1.0, 0.5],
        ["narizinho", "Com o Príncipe Escamado, vovó! Ele me pediu em casamento e eu aceitei.", 1.0, 0.5],
        ["dona_benta", "Mas, minha filha, ele é um peixe!", 1.0, 0.5],
        ["narizinho", "Ele é um príncipe, vovó! E a Emília, que é boneca, não casou com o Rabicó, que é leitão?", 1.05, 0.6],
        ["narrador", "Dona Benta olhou para Tia Nastácia, sem saber se a neta falava de verdade ou de brincadeira.", 1.0, 0.5],
        ["dona_benta", "Este sítio está virando um livro de histórias. Daqui a pouco aparece aqui um neto-peixe!", 1.0, 0.6],
        ["nastacia", "Credo, Dona Benta! Se isso acontecer, eu preciso de óculos novos, para não fritar o bisneto da senhora pensando que é traíra!", 1.0, 0.8],
        ["narrador", "Pedrinho escreveu a resposta, guardou na concha e deixou a concha no mesmo lugar. Os peixinhos levaram de volta, rolando pelo fundo do rio.", 1.0, 0.8],
        ["narrador", "No palácio, o príncipe abriu a concha. A resposta tinha uma palavra só: Sim! Com ponto de exclamação!", 1.05, 0.5],
        ["narrador", "O príncipe ficou tão feliz que pulou no trono, igual a um peixe fora da água.", 1.05, 0.5],
        ["principe", "Não é um sim qualquer! Tem ponto de exclamação! Quer dizer que ela aceitou com alegria!", 1.1, 0.8, "festa"],
        ["narrador", "Daí em diante, o príncipe escreveu cartas de amor sem parar. As sardinhas fizeram uma fila no fundo do rio e passavam as cartas de uma para outra, como tijolos.", 1.0, 0.5],
        ["narrador", "Narizinho respondia com presentes: uma flor, um grilinho verde. E mandou também uma rosquinha de polvilho, enrolada por ela mesma.", 1.0, 0.5],
        ["narrador", "O príncipe não comeu a rosquinha. Mandou enfeitar com diamantes, e a rosquinha virou a coroa dele!", 1.0, 1.0],
        ["narrador", "Resumindo: o príncipe pediu Narizinho em casamento numa carta dentro de uma concha, e ela disse sim.", 0.95, 0.8],
        ["narrador", "Agora todos vão viajar para o fundo do mar: Narizinho, Pedrinho, a Emília, o Rabicó e um visconde que ficou muito sabido.", 1.0, 0.6],
        ["narrador", "Como será essa viagem? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    14: {"titulo": "A viagem para o casamento", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo catorze: A viagem para o casamento.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o Príncipe Escamado pediu Narizinho em casamento, numa carta dentro de uma concha.", 1.0, 0.4],
        ["narrador", "Narizinho respondeu: Sim! E a rosquinha que ela mandou virou a coroa do príncipe.", 1.0, 1.0],
        ["narrador", "Chegou o dia da viagem. A Emília ia ser madrinha do casamento, mas não gostou do vestido.", 1.0, 0.5],
        ["emilia", "Quero vestido de cauda! Uma madrinha não pode ir assim!", 1.05, 0.5],
        ["narizinho", "Lá a gente arranja um. No fundo da água tem muito coral espinhento, e a cauda ia enganchar.", 1.0, 0.6],
        ["narrador", "O Visconde de Sabugosa ia ser o padrinho. E o visconde tinha mudado muito.", 1.0, 0.5],
        ["narrador", "Ele tinha caído atrás da estante de livros da Dona Benta e ficou lá esquecido três semanas. Por que será que ele mudou?", 0.95, 2.5],
        ["narrador", "Porque, de tanto ficar no meio dos livros, o visconde ficou sábio! Agora ele falava difícil, de ciência e de química.", 1.0, 0.6],
        ["narrador", "E o Rabicó? Pedrinho não queria levar, porque ele estragava todas as festas. Mas Narizinho disse que a Emília não podia ir sem o marido.", 1.0, 0.5],
        ["pedrinho", "Está bem, ele vai. Mas o meu estilingue vai também. Se ele se comportar mal, já sabe!", 1.0, 0.6],
        ["narrador", "Pedrinho pôs dois amendoins nas orelhas do Rabicó, como brincos.", 1.0, 0.4],
        ["pedrinho", "E não vá comer os brincos, senhor marquês!", 1.0, 0.8],
        ["narrador", "Às onze horas, todos foram para a beira do riacho. O Doutor Caramujo saiu da água e fez sinal com os chifrinhos.", 1.0, 0.5],
        ["narrador", "Então a água se abriu, como uma cortina. Todos desceram até o fundo e entraram no coche do príncipe.", 1.0, 1.0, "magia"],
        ["narrador", "Coche é uma carruagem bonita, de rei. Esse era puxado por cavalos-marinhos, uns peixinhos com cabeça de cavalo e rabo enroladinho.", 1.0, 0.5],
        ["narrador", "O Rabicó demorou para entrar, com medo. Quando entrou, já tinha perdido um brinco. A Emília ficou furiosa.", 1.0, 0.5],
        ["pedrinho", "Não faz mal. Lá no palácio a gente diz que a moda aqui na terra é um brinco só!", 1.05, 0.8],
        ["narrador", "Até agora: todos estão no coche, a caminho do casamento. O visconde está sábio, e o Rabicó tem um brinco só.", 0.95, 0.8],
        ["narrador", "O caminho era lindo: florestas de algas, que são plantas da água, florestas de coral e até mastros de navios afundados.", 1.0, 0.5],
        ["narrador", "Lá fora passavam peixes enormes. E até um polvo, um bicho mole, com oito braços compridos.", 1.0, 0.6],
        ["narizinho", "Doutor Caramujo, qual é o seu nome científico?", 1.0, 0.5],
        ["narrador", "Nome científico é o nome que os cientistas dão a cada bicho. O doutor não sabia o dele!", 1.0, 0.5],
        ["narizinho", "Visconde, mostre que o senhor sabe!", 1.0, 0.4],
        ["visconde", "O senhor Caramujo é um molusco gastrópode.", 0.95, 0.6],
        ["narizinho", "E qual é o nome científico da Emília?", 1.0, 0.5],
        ["visconde", "A senhora Emília é um animal artificial, que não está em nenhum livro de zoologia.", 0.95, 0.6],
        ["emilia", "Animal é o senhor! Antes de ser visconde, o senhor era um sabugo jogado no chão, perto das vacas!", 1.1, 0.8],
        ["narrador", "Todos riram. E o coche correu até parar na frente do palácio do príncipe.", 1.0, 1.0, "galope"],
        ["narrador", "O príncipe estava esperando, com a corte inteira. Todos bateram palmas e soltaram vaga-lumes de água, que eram os fogos de artifício de lá.", 1.0, 0.5],
        ["narrador", "O príncipe abraçou a noiva. Estava tão emocionado que não conseguia falar.", 0.95, 0.6],
        ["principe", "Preparei quartos lindos para todos. O visconde vai para o quarto das algas, e o marquês para o quarto dos corais vermelhos.", 1.0, 0.5],
        ["narizinho", "Príncipe, o visconde só quer saber de livros: ponha ele numa estante. E para o Rabicó, um chiqueirinho com três abóboras dentro!", 1.0, 0.8],
        ["narrador", "Narizinho e a Emília foram até a casa da Dona Aranha, para escolher os vestidos do casamento.", 1.0, 0.5],
        ["narrador", "E Pedrinho saiu para passear, com o visconde e o Rabicó, até a Floresta Vermelha, uma mata de coral.", 1.0, 1.0],
        ["narrador", "Resumindo: a viagem foi linda, o visconde mostrou que ficou sábio, e todos chegaram ao palácio.", 0.95, 0.8],
        ["narrador", "Mas o Rabicó, comilão como sempre, ia morder uma coisa que não era comida...", 0.9, 0.8],
        ["narrador", "O que será? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    15: {"titulo": "O vestido da cor do mar", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo quinze: O vestido da cor do mar.", 0.95, 1.2],
        ["narrador", "No capítulo passado, todos viajaram de coche até o palácio do príncipe, para o casamento de Narizinho.", 1.0, 0.4],
        ["narrador", "Narizinho foi escolher o vestido. E Pedrinho foi passear com o visconde e o Rabicó.", 1.0, 1.0],
        ["narrador", "No caminho, o Rabicó viu um ouriço-do-mar, uma bolinha cheia de espinhos. E pensou que era comida. Nhoc!", 1.05, 0.5],
        ["rabico", "Ai, ai, ai! Espetou a minha boca!", 1.1, 0.5],
        ["narrador", "Pedrinho tirou os espinhos, com muito cuidado. O Rabicó, chorando, disse que estava com fome. Então Pedrinho deixou ele comer o brinco de amendoim.", 1.0, 0.6],
        ["narrador", "Logo depois, eles acharam um navio velho, afundado na areia. Entraram por um buraco no casco.", 1.0, 0.5],
        ["rabico", "Achei uma raiz de mandioca! Que delícia!", 1.1, 0.5],
        ["narrador", "Mas não era mandioca. Era o braço de um polvo! O Rabicó mordeu, e o polvo enrolou o leitão com os braços compridos.", 1.0, 0.6],
        ["narrador", "Pedrinho pensou depressa e pediu ajuda a uma sardinha que estava olhando.", 1.0, 0.4],
        ["pedrinho", "Senhorita sardinha, corra ao palácio e diga ao príncipe que o marquês está preso por um polvo!", 1.05, 0.5],
        ["narrador", "Mas antes de ela sair, o visconde segurou a sardinha pelo rabinho.", 1.0, 0.4],
        ["visconde", "Senhorita, qual é o seu nome científico?", 0.95, 0.5],
        ["sardinha", "Malcriado!", 1.05, 0.5],
        ["narrador", "A sardinha pensou que era brincadeira de mau gosto e saiu nadando, brava, para levar o recado.", 1.0, 1.0, "transicao"],
        ["narrador", "Enquanto isso, na casa da Dona Aranha, ela mostrava o vestido de casamento.", 1.0, 0.5],
        ["narrador", "Não era de seda nem de renda. Era feito da cor do mar. E nele nadavam peixinhos de verdade: vermelhos, azuis, dourados, redondos, compridos.", 0.95, 0.6],
        ["narrador", "Os peixinhos nadavam sem parar, entre algas que balançavam. Por isso o vestido mudava o tempo todo.", 0.95, 0.5],
        ["narrador", "Narizinho achou tão lindo que ficou tonta e precisou sentar. A Emília quis saber tudo.", 1.0, 0.4],
        ["emilia", "Quem faz este tecido, Dona Aranha?", 1.0, 0.4],
        ["aranha", "A fada Miragem. Eu corto com a tesoura da Imaginação e costuro com a agulha da Fantasia e a linha do Sonho.", 0.95, 0.6],
        ["emilia", "E quanto custa o metro?", 1.05, 0.6],
        ["narizinho", "Fica quieta, Emília! Os peixinhos vão se assustar e fugir do vestido!", 1.0, 0.8],
        ["narrador", "Nessa hora, o príncipe entrou, assustado: o Rabicó estava preso por um polvo! Ele já tinha mandado os soldados.", 1.0, 0.6],
        ["narrador", "Até agora: o Rabicó está preso pelo polvo, e Narizinho ganhou o vestido mais lindo do mundo.", 0.95, 0.8],
        ["narrador", "Os soldados do príncipe eram caranguejos de casca dura. Para chegar depressa, vieram montados em peixes elétricos, que dão choque.", 1.0, 0.5],
        ["narrador", "Os caranguejos encostaram as garras no polvo e deram um choquinho. O polvo levou um susto, soltou o Rabicó e foi embora.", 1.0, 0.6],
        ["narrador", "O Rabicó estava salvo. Mas ainda gritava: coin, coin! Por que será?", 0.95, 2.5],
        ["narrador", "Porque um siri, que é um caranguejinho pequeno, tinha prendido a ponta do rabo dele!", 1.0, 0.5],
        ["pedrinho", "Fica assim mesmo, marquês. Esse siri no rabo é mais bonito que laço de fita!", 1.05, 0.8],
        ["narrador", "Quando chegaram ao palácio, o casamento já tinha começado. Narizinho mandou começar tudo de novo, para os amigos verem.", 1.0, 0.5],
        ["narrador", "Narizinho entrou com o vestido da cor do mar e uma coroa de pérolas. O príncipe veio ao lado, com um manto de escamas.", 1.0, 0.5],
        ["narrador", "Sereias, que são moças com rabo de peixe, cantavam músicas lindas. E Pedrinho pensou: a vovó diz que sereia não existe!", 1.0, 0.6],
        ["narrador", "Então a Emília viu o siri no rabo do marido e desmaiou de vergonha!", 1.05, 0.5],
        ["narrador", "O Doutor Caramujo tirou o siri do rabo do Rabicó e encostou no nariz da Emília. E a boneca acordou na hora.", 1.0, 0.6],
        ["emilia", "Onde estou? Estou vendo o mundo cheio de perninhas!", 1.0, 0.8],
        ["narrador", "Eram as perninhas do siri, pendurado no nariz dela! Todos riram.", 1.0, 0.8],
        ["narrador", "Por fim, um caranguejo-ermitão, que vive dentro de uma concha emprestada, casou os noivos. Narizinho virou princesa!", 1.0, 1.0, "festa"],
        ["narrador", "Faltava só pôr a coroa na cabeça do príncipe. Mas... cadê a coroa? A coroa de rosquinha tinha sumido!", 1.0, 0.5],
        ["narrador", "O príncipe ficou tão bravo que todo mundo saiu correndo. Narizinho e os amigos correram também, e só pararam no sítio, pingando água.", 1.05, 0.6],
        ["emilia", "Eu vi, Narizinho! Foi o Rabicó que comeu a coroa!", 1.1, 0.6],
        ["narrador", "E tinha sido mesmo...", 0.95, 1.0],
        ["narrador", "Resumindo: Narizinho casou com o príncipe, mas o Rabicó comeu a coroa, e a festa acabou em correria.", 0.95, 0.8],
        ["narrador", "Dias depois, numa tarde quente, apareceu no sítio um visitante de bigodes, que andava em duas patas e falava muito bonito.", 0.9, 0.8],
        ["narrador", "Quem será? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    16: {"titulo": "O príncipe visita o sítio", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo dezesseis: O príncipe visita o sítio.", 0.95, 1.2],
        ["narrador", "No capítulo passado, Narizinho casou com o Príncipe Escamado. Mas o Rabicó comeu a coroa de rosquinha, e todos fugiram para o sítio.", 1.0, 1.0],
        ["narrador", "Num dia de sol muito quente, Narizinho e a Emília estavam na sombra da jabuticabeira. De repente, ouviram um miado.", 1.0, 0.5],
        ["narrador", "Que estranho! No sítio não tinha nenhum gato.", 0.95, 0.5],
        ["narrador", "De uma moita saiu um gato preto, de bigodes compridos, que andava em duas patas. Ele sentou no colo de Narizinho, sem cerimônia.", 1.0, 0.5],
        ["narizinho", "Eu conheço você! É o Gato Félix, aquele dos filmes!", 1.05, 0.5],
        ["gato", "Eu mesmo. Ando viajando pelo mundo. Até no fundo do riacho eu estive, trabalhando para o Príncipe Escamado.", 1.0, 0.5],
        ["gato", "E ele mandou um recado: hoje mesmo, ele vem visitar a esposa. E quer conhecer a vovó.", 1.0, 0.6],
        ["narizinho", "Que bom! A vovó e a Tia Nastácia nunca acreditam no que eu conto. Quero ver a cara delas agora!", 1.1, 0.6],
        ["narrador", "Narizinho chamou Pedrinho, e os quatro combinaram uma surpresa para Dona Benta.", 1.0, 0.8],
        ["narrador", "Na hora do almoço, a Emília fazia uma cara tão sabida que Dona Benta desconfiou.", 1.0, 0.5],
        ["dona_benta", "Vocês estão me preparando alguma. Essa cara da Emília não me engana!", 1.0, 0.6],
        ["narrador", "A Emília nunca soube fingir. Quando fingia, fingia demais e estragava tudo.", 1.0, 0.6],
        ["narrador", "Então alguém bateu na porta, bem de leve.", 0.95, 0.8, "porta"],
        ["narrador", "Tia Nastácia foi abrir. Mas antes, espiou pelo buraco da fechadura. E arregalou os olhos.", 0.95, 0.5],
        ["nastacia", "Dona Benta! O terreiro está cheio de peixes, conchas, caranguejos e bichinhos do mar!", 1.05, 0.5],
        ["narizinho", "Não é nada, vovó! É o meu marido, o príncipe, com a corte dele. Eles querem conhecer a senhora.", 1.0, 0.6],
        ["nastacia", "Não abre, Dona Benta! Uma vez um caranguejo me beliscou o dedo!", 1.05, 0.5],
        ["narrador", "E Tia Nastácia trancou a porta com uma tranca de madeira.", 1.0, 0.8, "transicao"],
        ["narrador", "Até agora: o Gato Félix trouxe a notícia, e o príncipe chegou. Mas a porta está trancada!", 0.95, 0.8],
        ["narrador", "Pedrinho saiu pelos fundos e encontrou o príncipe, todo sem graça.", 1.0, 0.4],
        ["principe", "Se a senhora tem medo, é melhor eu voltar. Não quero incomodar.", 0.95, 0.5],
        ["pedrinho", "Nada disso! Se não dá para entrar pela porta, entra pela janela!", 1.05, 0.5],
        ["narrador", "Pedrinho trouxe uma escadinha, e os bichinhos do mar foram subindo e pulando pela janela.", 1.0, 0.5],
        ["narrador", "O primeiro foi o Doutor Caramujo, de óculos. Depois, o próprio príncipe, vestido de rei.", 1.0, 0.5],
        ["narizinho", "Vovó, este é o meu marido, o Príncipe Escamado. Príncipe, esta é a minha vovó, Dona Benta.", 1.0, 0.5],
        ["principe", "Muito prazer, minha senhora. Posso chamar a senhora de vovó também?", 0.95, 0.6],
        ["narrador", "Dona Benta quase desmaiou. Um peixinho falando, igualzinho a gente!", 1.0, 0.5],
        ["narrador", "Depois pularam a Dona Aranha, o Major Agarra e uma sardinha americana, a Miss Sardine. A cada um, Tia Nastácia dava um grito e corria mais um pouco.", 1.0, 0.6],
        ["narrador", "Mas Dona Benta foi se acostumando. E começou a achar graça.", 1.0, 0.5],
        ["dona_benta", "Você tem razão, minha filha. O mundo de vocês é muito mais interessante que o nosso.", 0.95, 0.8],
        ["narrador", "Narizinho mostrou a casa ao príncipe: o relógio, os pratos, o pote de água. Ele gostou mais do guarda-chuva.", 1.0, 0.5],
        ["principe", "Para que serve isto?", 1.0, 0.4],
        ["narizinho", "Para a gente não se molhar.", 1.0, 0.4],
        ["principe", "Então por que vocês não levaram um na viagem para o fundo da água?", 1.0, 0.8],
        ["narrador", "Narizinho riu tanto que deu um beijo na testa do marido.", 1.0, 0.5],
        ["narrador", "O príncipe perguntou pelo visconde. Narizinho contou que ele voltou da viagem tão molhado que ficou mofado, verdinho. Ela embrulhou o visconde num livro velho e esqueceu onde guardou.", 0.95, 0.8],
        ["narizinho", "E o Gato Félix trabalha mesmo para você, príncipe?", 1.0, 0.5],
        ["principe", "Gato Félix? Não conheço. Eu mandei o recado por uma sardinha.", 1.0, 0.8],
        ["narrador", "Narizinho lembrou que o gato tinha cheiro de sardinha. Por que será?", 0.95, 2.5],
        ["narrador", "Será que o gato comeu a mensageira, com recado e tudo? Narizinho achou que sim, mas não disse nada, para não deixar o príncipe triste.", 0.95, 1.0],
        ["narrador", "Resumindo: o príncipe e a corte entraram pela janela, e Dona Benta adorou a visita. Mas esse Gato Félix parece muito esquisito.", 0.95, 0.8],
        ["narrador", "Enquanto isso, a Miss Sardine foi direto para a cozinha da Tia Nastácia. E queria mexer em tudo...", 0.9, 0.8],
        ["narrador", "O que será que vai acontecer? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    17: {"titulo": "A pílula que a Tia Nastácia engoliu", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo dezessete: A pílula que a Tia Nastácia engoliu.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o Príncipe Escamado e a corte entraram no sítio pela janela. Dona Benta adorou a visita.", 1.0, 0.4],
        ["narrador", "E a Miss Sardine, uma sardinha americana, foi direto para a cozinha.", 1.0, 1.0],
        ["narrador", "Tia Nastácia já tinha perdido o medo dos bichinhos do mar. E ficou amiga da Miss Sardine.", 1.0, 0.5],
        ["narrador", "A sardinha era muito curiosa. Perguntava tudo, como uma criança.", 1.0, 0.4],
        ["sardinha", "Que monstro esquisito é esse?", 1.05, 0.4],
        ["nastacia", "Isso é o fogão. E aquela coisa vermelha lá dentro é o fogo. Serve para queimar o dedinho de quem põe o dedinho nele!", 1.0, 0.6],
        ["narrador", "A Miss Sardine subiu na prateleira e provou o sal.", 1.0, 0.4],
        ["sardinha", "Hum! Eu conheço este gosto!", 1.05, 0.4],
        ["nastacia", "É claro. O sal vem do mar, que é a sua terra.", 1.0, 0.6],
        ["narrador", "Depois ela abriu o vidro de pimenta em pó. Escorregou e caiu lá dentro, de cabeça!", 1.05, 0.5],
        ["sardinha", "Socorro! Meus olhos estão ardendo!", 1.1, 0.5],
        ["narrador", "Tia Nastácia tirou a sardinha do vidro e lavou na água da bica, com muito cuidado. Meia hora depois, a ardência passou.", 1.0, 0.6],
        ["nastacia", "Viu? Quem mexe em tudo acaba se machucando. Cuidado com o fogão, menina!", 1.0, 1.0, "transicao"],
        ["narrador", "Enquanto isso, no quintal, Dona Benta conversava com o Doutor Caramujo sobre um pintinho doente.", 1.0, 0.5],
        ["dona_benta", "Nastácia, pegue o pintinho. O doutor vai dar uma pílula para ele.", 1.0, 0.5],
        ["nastacia", "Um caramujo entendendo de remédio? Isso é bobagem, Dona Benta!", 1.0, 0.5],
        ["narrador", "Mas ela pegou o pintinho. O doutor explicou como se fazia.", 1.0, 0.4],
        ["caramujo", "Põe-se a pílula dentro de um canudinho. A ponta do canudinho vai no bico do pinto. Depois, é só assoprar.", 0.95, 0.6],
        ["narrador", "Tia Nastácia pôs a boca na outra ponta do canudinho. Mas antes de assoprar, deu um grito e começou a tossir!", 1.05, 0.5],
        ["narrador", "O que será que aconteceu?", 0.95, 2.5],
        ["nastacia", "O pinto assoprou primeiro! E quem engoliu a pílula fui eu!", 1.1, 0.6],
        ["narrador", "Dona Benta riu muito. E sabe o que aconteceu depois?", 1.0, 0.5],
        ["narrador", "A pílula curou uma tosse que a Tia Nastácia tinha fazia duas semanas! Daí em diante, ela passou a acreditar nas pílulas do doutor.", 1.0, 0.8],
        ["narrador", "Até agora: a sardinha caiu na pimenta, e a Tia Nastácia engoliu a pílula do pintinho, sem querer.", 0.95, 0.8],
        ["narrador", "No quarto de costura, a Emília conversava com a Dona Aranha, que tinha vindo junto com o príncipe.", 1.0, 0.5],
        ["emilia", "Dona Aranha, como a senhora engole um carretel de linha?", 1.0, 0.4],
        ["aranha", "Eu não engulo, menina. Eu já nasci com a linha aqui dentro. E ela nunca acaba.", 0.95, 0.5],
        ["emilia", "Eu sei o que tem dentro de mim: só macela, amarelinha e cheirosa. E eu não como nada. Só de mentirinha.", 1.0, 0.5],
        ["aranha", "É a primeira pessoa que eu conheço que não come!", 0.95, 0.5],
        ["emilia", "Se eu achasse uma aranha de linha vermelha, eu ia morar com ela. Só para ver a linha sair, sair, sair, e nunca acabar!", 1.05, 0.8],
        ["narrador", "No terreiro, Pedrinho conversava com o capitão dos soldados do príncipe, um caranguejo de casca dura que adorava contar vantagem.", 1.0, 0.5],
        ["narrador", "O capitão perguntou qual era o bicho mais perigoso da terra. Ele achava que era o leão.", 1.0, 0.5],
        ["pedrinho", "Para mim, o mais perigoso é a vespa. É pequenininha, voa muito depressa, e quando pica, arde que nem fogo!", 1.05, 0.5],
        ["narrador", "O capitão riu e disse que não tinha medo de vespa nenhuma. Ele tinha casca dura!", 1.0, 0.5],
        ["pedrinho", "Pois eu tenho medo de vespa e não tenho vergonha de dizer. Ter medo de vespa não é vergonha para ninguém!", 1.0, 1.0],
        ["narrador", "Resumindo: os bichinhos do mar se espalharam pelo sítio, cada um conhecendo uma coisa nova.", 0.95, 0.8],
        ["narrador", "E agora, Narizinho ia mostrar ao príncipe a moradora mais importante do sítio, depois da vovó e da Tia Nastácia: a Vaca Mocha.", 1.0, 0.6],
        ["narrador", "Como será que o príncipe vai reagir? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    18: {"titulo": "A Vaca Mocha e a despedida", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo dezoito: A Vaca Mocha e a despedida.", 0.95, 1.2],
        ["narrador", "No capítulo passado, a Miss Sardine caiu na pimenta, e a Tia Nastácia engoliu a pílula do pintinho.", 1.0, 0.4],
        ["narrador", "Agora, Narizinho ia apresentar ao príncipe a Vaca Mocha.", 1.0, 1.0],
        ["narrador", "O príncipe nunca tinha visto uma vaca. No fundo da água não existe vaca.", 1.0, 0.5],
        ["narrador", "Quando chegou perto, ele parou, de olhos arregalados. A vaca era grande como uma casa, para um peixinho!", 1.0, 0.5],
        ["narizinho", "Esta é a Mocha, príncipe. Veja que pelo macio e que chifres pontudos.", 1.0, 0.5],
        ["principe", "E o que é isso pendurado aqui embaixo?", 1.0, 0.4],
        ["narizinho", "São as tetas. São as torneirinhas do leite. Tia Nastácia aperta, e sai leite branquinho.", 1.0, 0.5],
        ["principe", "E esta corda comprida, lá atrás?", 1.0, 0.4],
        ["narizinho", "É o rabo. Serve para espantar as moscas.", 1.0, 0.5],
        ["narizinho", "E os chifres, que são dois espetos tortos na testa, servem para espantar cachorro. Mas fique tranquilo: a Mocha é muito boazinha.", 1.0, 0.6],
        ["principe", "E como é que ela faz o leite?", 1.0, 0.4],
        ["narizinho", "Isso eu não sei. Ela come capim e abóbora, e sai leite pelas torneirinhas.", 1.0, 0.5],
        ["emilia", "Eu sei! Ela come mandioca. Leite é mandioca líquida!", 1.05, 0.5],
        ["narizinho", "Que bobagem, Emília! O Rabicó também come mandioca e não dá leite.", 1.0, 0.4],
        ["emilia", "É porque ele não tem torneirinhas!", 1.05, 0.8],
        ["narrador", "Narizinho ficou tão cansada das ideias da Emília que pôs a boneca de ponta-cabeça no bolso do avental.", 1.0, 0.6],
        ["narrador", "Bem nessa hora, a vaca fez: Muuu! O príncipe levou um susto e caiu sentado. Por que será que ele se assustou tanto?", 0.95, 2.5],
        ["narrador", "Porque ele nunca tinha ouvido um mugido. É um som grosso e comprido, que ele não esperava.", 1.0, 0.5],
        ["principe", "Vamos voltar para casa. Meu coração está batendo muito depressa.", 0.95, 1.0, "transicao"],
        ["narrador", "Até agora: o príncipe conheceu a Vaca Mocha e levou um susto com o mugido.", 0.95, 0.8],
        ["narrador", "Mas quando chegaram em casa, ouviram choro lá dentro. Narizinho correu até a cozinha.", 1.0, 0.5],
        ["narizinho", "O que aconteceu, Tia Nastácia?", 1.0, 0.5],
        ["nastacia", "Foi a Miss Sardine, menina. Eu saí um minutinho, e ela subiu no fogão para espiar a frigideira de gordura quente.", 0.95, 0.5],
        ["nastacia", "Ela achou que a gordura borbulhando era uma lagoa. E pulou dentro!", 0.95, 0.6],
        ["narrador", "Tia Nastácia tirou a sardinha bem depressa, mas a coitada ficou toda queimada.", 0.95, 0.6],
        ["narrador", "Quando o príncipe soube, desmaiou de tristeza. Depois, acordou e só queria uma coisa: voltar para casa, levando a Miss Sardine para se tratar no fundo do mar.", 0.95, 0.8],
        ["dona_benta", "A nossa casa está sempre aberta para o senhor, príncipe. Volte quando quiser.", 0.95, 0.5],
        ["principe", "Muito obrigado, vovó. E venha também nos visitar no reino.", 0.95, 0.5],
        ["dona_benta", "Ah, isso é difícil. Estou muito velha, e posso me molhar no caminho.", 0.95, 0.4],
        ["emilia", "Molhar como? A senhora vai de guarda-chuva!", 1.05, 0.8],
        ["narrador", "Mas que presente dar ao príncipe? Pedrinho teve a ideia.", 1.0, 0.4],
        ["pedrinho", "Tenho umas rodinhas que sobraram do despertador que eu consertei. No mar não existe roda. Ele vai adorar!", 1.05, 0.6],
        ["narrador", "E o príncipe ganhou quatro rodinhas, uma de cada pessoa do sítio.", 1.0, 0.6],
        ["narrador", "Na despedida, todos choraram. Até a Emília apareceu com duas lágrimas nos olhos de linha.", 0.95, 0.5],
        ["narrador", "O príncipe e a corte foram embora, de nariz vermelho de tanto chorar. E todos acenaram com lenços da janela.", 0.9, 1.0],
        ["narrador", "Resumindo: o príncipe adorou a Vaca Mocha, mas a visita terminou triste por causa da Miss Sardine.", 0.95, 0.8],
        ["narizinho", "Ainda bem que o Gato Félix vai voltar. Senão, que saudade...", 0.95, 0.5],
        ["narrador", "Nem bem ela falou, o Gato Félix apareceu no terreiro, correndo.", 1.05, 0.4],
        ["gato", "Socorro! O príncipe está se afogando!", 1.1, 0.6],
        ["narrador", "Um peixe se afogando? Como pode ser? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    19: {"titulo": "As histórias do Gato Félix", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo dezenove: As histórias do Gato Félix.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o príncipe foi embora do sítio, muito triste. E o Gato Félix chegou gritando que o príncipe estava se afogando.", 1.0, 1.0],
        ["narrador", "Narizinho correu até o riacho. Mas o príncipe já não estava lá. Ele tinha se salvado sozinho, porque, afinal, era um peixe!", 1.0, 0.6],
        ["narrador", "Então Narizinho voltou para casa e pôs o gato no colo.", 1.0, 0.4],
        ["narizinho", "Agora você vai contar a sua vida inteirinha, Gato Félix.", 1.0, 0.5],
        ["gato", "Conto, sim. Mas só de noite. De dia, as histórias perdem a graça.", 1.0, 0.8],
        ["narrador", "Quando escureceu, Tia Nastácia acendeu o lampião da sala, que é uma lamparina grande, de fogo, e disse:", 1.0, 0.4],
        ["nastacia", "É hora, gente!", 1.0, 0.6],
        ["narrador", "Todos sentaram em volta do gato. Dona Benta na cadeirinha. Narizinho e Pedrinho na rede. A Emília no colo de Narizinho.", 1.0, 0.5],
        ["narrador", "Até o Visconde de Sabugosa veio ouvir. Ele ainda estava mofado, então Narizinho limpou o mofo e pôs o visconde numa lata, num canto, para não sujar o chão.", 1.0, 0.8, "transicao"],
        ["gato", "Na França, há muito tempo, viveu um gato famoso. Ele usava botas!", 1.0, 0.5],
        ["emilia", "Eu sei! O Gato de Botas!", 1.1, 0.5],
        ["gato", "Isso mesmo. E eu sou neto do neto do neto dele, umas cinquenta vezes.", 1.0, 0.5],
        ["gato", "O meu avô veio para a América no navio de Cristóvão Colombo. Lá embaixo, no porão, ele caçou mais de mil ratos!", 1.05, 0.6],
        ["narrador", "Aí o visconde pôs a cabeça para fora da lata.", 1.0, 0.4],
        ["visconde", "Não acreditem! Eu li a história de Colombo num livro grande da Dona Benta. Não foi nada disso!", 1.0, 0.5],
        ["emilia", "Foi assim, sim! O livro não estava lá. O avô do Félix estava!", 1.05, 0.5],
        ["narrador", "E Narizinho, para acabar a briga, tampou a lata com o visconde dentro.", 1.0, 0.8],
        ["narrador", "Até agora: o Gato Félix diz que é parente do Gato de Botas. Mas o visconde acha que é tudo mentira.", 0.95, 0.8],
        ["gato", "Eu nasci em Nova York, uma cidade dos Estados Unidos, num prédio tão alto que se chama arranha-céu.", 1.0, 0.5],
        ["narizinho", "E arranha o céu de verdade?", 1.0, 0.4],
        ["gato", "Arranha, sim. O céu de lá é todo furadinho!", 1.05, 0.6],
        ["gato", "Um dia, eu viajei num navio velho. Mas o navio bateu numa pedra e começou a afundar.", 0.95, 0.5],
        ["gato", "Então eu tive uma ideia: pulei dentro da boca de um tubarão, e fui escorregando até a barriga dele. Era do tamanho desta sala!", 1.0, 0.6],
        ["narrador", "O visconde empurrou a tampa da lata outra vez.", 1.0, 0.4],
        ["visconde", "Mentira! Barriga de tubarão não é desse tamanho. E um gato não vive lá dentro!", 1.0, 0.5],
        ["emilia", "E o Jonas, da história que a Dona Benta contou? Ele não viveu dentro da barriga de um peixe?", 1.05, 0.8],
        ["narrador", "Todos acharam que a Emília tinha razão. E o gato continuou.", 1.0, 0.4],
        ["gato", "Uns pescadores pegaram o tubarão. Quando abriram a barriga dele, eu pulei para fora, vivinho!", 1.05, 0.5],
        ["gato", "Depois, saí procurando o lugar onde o diabo perdeu as botas.", 1.0, 0.6],
        ["narrador", "Lugar onde o diabo perdeu as botas é um jeito de dizer: um lugar muito, muito longe. Mas o gato achava que era um lugar de verdade.", 1.0, 0.6],
        ["gato", "Eu andei, andei, andei. Perguntei para uma velha, para um velho, para outra velha...", 0.95, 0.5],
        ["emilia", "E daí?", 1.05, 0.4],
        ["gato", "Daí... daí eu parei de procurar e fui cuidar de outra coisa.", 0.95, 1.0],
        ["narrador", "Todo mundo ficou desapontado. A história não tinha fim! Por que será que a história do gato ficou tão sem graça?", 0.95, 2.5],
        ["narrador", "Porque ele inventava tudo, sem pé nem cabeça, e no fim não sabia como terminar.", 1.0, 0.5],
        ["emilia", "Eu nunca saí daqui e sei contar uma história muito mais bonita!", 1.05, 0.6],
        ["dona_benta", "Então vamos dormir. E amanhã, quem conta a história é a Emília.", 1.0, 1.0],
        ["narrador", "Resumindo: o Gato Félix contou uma história cheia de invenções, e o visconde não acreditou em nada.", 0.95, 0.8],
        ["narrador", "Na manhã seguinte, Tia Nastácia contou as galinhas novas e levou um susto. Estava faltando um pintinho!", 1.0, 0.6],
        ["narrador", "Quem será que pegou o pintinho? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    20: {"titulo": "A história da Emília", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo vinte: A história da Emília.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o Gato Félix contou uma história cheia de invenções. E de manhã, sumiu um pintinho do galinheiro.", 1.0, 1.0],
        ["nastacia", "Eram doze pintinhos, Dona Benta. Agora só tem onze. Sumiu justo o mais bonito, o carijozinho!", 1.0, 0.5],
        ["dona_benta", "Será alguma raposa? Ou algum gato vagabundo?", 0.95, 0.6],
        ["pedrinho", "Vamos falar com o visconde. Desde que ficou embrulhado num livro do Sherlock Holmes, ele virou detetive!", 1.05, 0.5],
        ["narrador", "Sherlock Holmes é um detetive famoso dos livros. Ele descobre quem fez as coisas olhando pistas bem pequenininhas.", 1.0, 0.5],
        ["visconde", "Deixem comigo. Vou examinar o local do crime.", 0.95, 0.6],
        ["narrador", "E o visconde passou o dia inteiro no galinheiro, olhando a poeira do chão e catando pelinhos.", 1.0, 0.6],
        ["narrador", "Enquanto isso, a Emília pensou, pensou, pensou. E inventou a história que ia contar de noite.", 1.0, 0.8, "transicao"],
        ["narrador", "Quando o lampião acendeu, a Emília entrou na sala, toda esticadinha, cheia de si.", 1.0, 0.5],
        ["emilia", "Era uma vez um rei, um príncipe e uma fada, que moravam num palácio de cristal, na beira de um lago bem azul.", 0.95, 0.5],
        ["narizinho", "Eu já sabia que vinha rei, príncipe e fada!", 1.0, 0.5],
        ["emilia", "O príncipe disse: Pai, quero casar. Vou procurar uma pastora bem pobrezinha, mas com o coração de ouro.", 0.95, 0.5],
        ["emilia", "E, para viajar escondido, o príncipe virou a fada numa bengalinha. E virou ele mesmo numa formiguinha.", 0.95, 0.8, "magia"],
        ["narizinho", "Eu já sabia que vinha história de virar!", 1.0, 0.5],
        ["emilia", "Não me atrapalhe! A formiguinha andou, andou, andou, e encontrou uma velha. A velha disse:", 0.95, 0.4],
        ["emilia", "Para saber se uma pastora tem bom coração, vire um pobre e vá pedir um pão.", 0.95, 0.8],
        ["narrador", "Até agora: na história da Emília, o príncipe virou formiguinha e foi procurar uma pastora de bom coração.", 0.95, 0.8],
        ["emilia", "A formiguinha virou um pobre e pediu um pão para a primeira pastora. Ela deu uma pedra e disse: Toma, é um pão gostoso!", 0.95, 0.5],
        ["emilia", "Quando a pastora foi lanchar, o pão dela também tinha virado pedra. E ela ficou sem lanche.", 0.95, 0.5],
        ["emilia", "A segunda pastora deu um osso e disse a mesma mentira. E o lanche dela virou osso.", 0.95, 0.6],
        ["emilia", "A terceira pastora era ainda mais pobre, e estava chorando. A madrasta dela era muito brava e só dava um pão velho e duro.", 0.9, 0.5],
        ["emilia", "Mas ela olhou para o pobre e repartiu o pão. E deu o pedaço maior para ele!", 0.95, 0.6],
        ["narrador", "O que será que aconteceu com o pão dessa pastora?", 0.95, 2.5],
        ["emilia", "Virou o doce mais gostoso do mundo! E quanto mais ela comia, mais sobrava!", 1.05, 0.8, "magia"],
        ["emilia", "A madrasta sentiu o cheiro de doce e ficou furiosa. Gritou que ia dar um castigo muito grande na menina.", 0.95, 0.5],
        ["emilia", "Mas nessa hora o pobre entrou e segurou o braço dela. E virou a madrasta num sapo, que foi morar sozinho no brejo.", 0.95, 0.8, "magia"],
        ["emilia", "Depois, o pobre casou com a pastora. Os dois chegaram ao palácio, e ele virou príncipe outra vez. E a pastora virou a princesa mais linda que já existiu!", 0.95, 0.8],
        ["narrador", "Todos bateram palmas.", 1.0, 0.5, "festa"],
        ["dona_benta", "Não é que essa boneca aprendeu a contar histórias como gente grande?", 0.95, 0.5],
        ["gato", "Eu não gostei. História de virar é fácil demais. Apareceu um problema, uma coisa vira outra, e pronto!", 1.0, 0.5],
        ["narizinho", "Não ligue, Emília. A sua história merece nota dez!", 1.0, 0.5],
        ["narrador", "A Emília ficou toda orgulhosa. E mostrou a língua para o Gato Félix.", 1.0, 0.8],
        ["narrador", "Resumindo: a Emília contou a história da pastora de bom coração, e todos adoraram. Só o gato não gostou.", 0.95, 0.8],
        ["dona_benta", "Vamos dormir, crianças. Amanhã, quem conta a história é o visconde.", 1.0, 0.6],
        ["narrador", "Mas de manhã, Tia Nastácia foi ao galinheiro e voltou de mãos na cabeça. Tinha sumido mais um pintinho!", 1.0, 0.6],
        ["narrador", "Será que o visconde vai descobrir o ladrão? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    21: {"titulo": "O visconde detetive", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo vinte e um: O visconde detetive.", 0.95, 1.2],
        ["narrador", "No capítulo passado, a Emília contou uma história linda. Mas sumiu mais um pintinho do galinheiro!", 1.0, 1.0],
        ["pedrinho", "Para mim, é uma raposa que vem de noite.", 1.0, 0.4],
        ["visconde", "Não é raposa, não. Achei no galinheiro um pelinho que não é de raposa, nem de gambá, nem de rato.", 0.95, 0.5],
        ["visconde", "Preciso olhar esse pelo bem de pertinho. Pedrinho, arranje alguma coisa que aumente as coisas.", 0.95, 0.5],
        ["narrador", "Pedrinho trouxe o binóculo da vovó, que é um aparelho de olhar longe. Mas, virado ao contrário, também mostra as coisas grandes.", 1.0, 0.5],
        ["narrador", "O visconde olhou o pelinho com muita atenção.", 0.95, 0.5],
        ["visconde", "Estou na pista do ladrão. É um bicho de quatro patas, da família dos gatos. Agora me deixe sozinho. Preciso pensar.", 0.95, 0.8, "transicao"],
        ["narrador", "Até agora: o visconde achou um pelinho de gato no galinheiro.", 0.95, 0.8],
        ["narrador", "Na varanda, a Emília se balançava na sua redinha. O Gato Félix sentou na frente dela e ficou olhando, balançando o rabo.", 1.0, 0.5],
        ["emilia", "Que tanto olha para mim? Nunca me viu?", 1.05, 0.4],
        ["gato", "Nunca vi uma boneca tão convencida. Pensa que é uma grande contadora de histórias!", 1.0, 0.5],
        ["emilia", "Você está é com inveja da minha história!", 1.05, 0.4],
        ["gato", "Todo mundo só bateu palmas para você não chorar.", 1.0, 0.4],
        ["emilia", "Mentiroso! Você não é americano, nem parente do Gato de Botas, nem foi engolido por tubarão nenhum. É tudo invenção!", 1.1, 0.5],
        ["narrador", "O gato ficou furioso e levantou a pata para arranhar a Emília. Mas ela foi mais rápida: agarrou o bigode dele e puxou com força!", 1.05, 0.5],
        ["narrador", "Arrancou um fio de bigode inteirinho.", 1.0, 0.5],
        ["narrador", "Narizinho chegou correndo, separou os dois e pôs o gato para fora da varanda.", 1.0, 0.6],
        ["narrador", "Logo depois, o visconde apareceu. E a Emília contou a briga inteira.", 1.0, 0.4],
        ["visconde", "Um fio de bigode do gato? Que interessante! Posso estudar esse fio?", 0.95, 0.5],
        ["emilia", "Pode. Mas depois devolva. Quero guardar de lembrança da minha vitória!", 1.05, 0.8],
        ["narrador", "Resumindo até aqui: a Emília brigou com o gato e arrancou um fio do bigode dele. E o visconde levou o fio para examinar.", 0.95, 0.8],
        ["narrador", "De noite, antes de começar a história, o visconde cochichou no ouvido da Tia Nastácia:", 0.95, 0.4],
        ["visconde", "Deixe a vassoura bem pertinho da senhora. No fim da minha história, talvez seja preciso varrer alguma coisa.", 0.9, 0.6],
        ["emilia", "Com a palavra, o senhor Visconde de Sabugosa!", 1.05, 0.5],
        ["visconde", "Era uma vez um gato que não valia nada. Era um gato ladrão.", 0.95, 0.5],
        ["visconde", "Na primeira casa onde morou, descobriram que ele roubava, e mandaram ele embora.", 0.95, 0.4],
        ["visconde", "Na segunda casa, ele roubou a carne da cozinha. E foi mandado embora de novo.", 0.95, 0.4],
        ["visconde", "Então ele chegou num sítio onde moravam duas senhoras, um menino e uma menina. E disse que era um gato importante, nascido num país estrangeiro.", 0.95, 0.5],
        ["narizinho", "Que coincidência! Parece o sítio da vovó...", 1.0, 0.5],
        ["visconde", "Todos trataram o gato muito bem. Mas ele pegou um pintinho carijó. E, no outro dia, mais um pintinho.", 0.9, 0.6],
        ["narrador", "O visconde parou e olhou bem firme para o Gato Félix.", 0.9, 0.5],
        ["gato", "O senhor está me insultando! Parece que está dizendo que o ladrão sou eu!", 1.05, 0.4],
        ["visconde", "E é mesmo! O ladrão é você! Você nunca foi Gato Félix nenhum!", 1.1, 0.5],
        ["gato", "Prove, se for capaz!", 1.1, 0.5],
        ["narrador", "Como será que o visconde vai provar?", 0.95, 2.5],
        ["visconde", "Este pelo, eu achei no galinheiro. E este outro, a Emília arrancou do seu bigode. Os dois são iguaizinhos! Até no cheiro!", 1.0, 0.6],
        ["narrador", "Tia Nastácia pegou a vassoura e foi para cima do falso Gato Félix. O gato deu um pulo e sumiu pela janela, na escuridão da noite.", 1.05, 0.8],
        ["narrador", "Todos gritaram: Viva o nosso Sherlock Holmes! Viva o visconde!", 1.05, 0.6, "festa"],
        ["narrador", "E até a Emília, que era muito envergonhada, deu um beijo na testa do visconde.", 1.0, 0.5],
        ["dona_benta", "Quase fomos injustos com o visconde, só porque ele estava mofado e feio. Ele é um verdadeiro sábio! De hoje em diante, eu cuido dele.", 0.95, 0.6],
        ["narrador", "E Dona Benta guardou o visconde na estante, bem protegido, entre dois livros de matemática.", 0.95, 0.8],
        ["narrador", "Resumindo: o visconde descobriu que o falso Gato Félix era o ladrão de pintinhos, e o gato fugiu pela janela.", 0.95, 0.8],
        ["narrador", "Mas a Emília não esqueceu aquele bigode. E o gato também não esqueceu a Emília...", 0.9, 0.6],
        ["narrador", "O que será que vem por aí? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    22: {"titulo": "A festa das princesas", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo vinte e dois: A festa das princesas.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o visconde descobriu que o falso Gato Félix era o ladrão de pintinhos. E virou o sábio da casa.", 1.0, 1.0],
        ["narrador", "Um dia, Narizinho resolveu dar uma festa. Mas não uma festa qualquer! Ela convidou os personagens dos livros de histórias.", 1.0, 0.5],
        ["pedrinho", "Como você mandou os convites?", 1.0, 0.4],
        ["narizinho", "Por um beija-flor que vem todo dia beijar as rosas da Emília. Ele pegou as cartinhas no bico e foi entregar.", 1.0, 0.5],
        ["narizinho", "Convidei a Cinderela, a Branca de Neve, o Pequeno Polegar, a Capinha Vermelha, o Aladim, o Gato de Botas, o Peter Pan... todo mundo!", 1.05, 0.6],
        ["narrador", "Primeiro, era preciso tirar o visconde de trás da estante, onde ele tinha caído. Estava inteirinho, só que mais mofado e cheio de teia de aranha.", 1.0, 0.5],
        ["pedrinho", "O senhor visconde vai ficar no alto da janela, com o binóculo da vovó, espiando a estrada. Quando aparecer uma poeirinha lá longe, avise!", 1.0, 0.5],
        ["narrador", "E o Rabicó ficou na porta, de fitinha vermelha no rabo, para anunciar quem chegava.", 1.0, 0.6],
        ["dona_benta", "Que festa é essa, Narizinho?", 0.95, 0.4],
        ["narizinho", "Uma festa linda, vovó! Só reis, príncipes, princesas e fadas! Quando ouvir muitas palmas, venha espiar pelo buraco da fechadura.", 1.05, 0.8],
        ["visconde", "Estou vendo uma poeirinha lá longe!", 1.0, 0.8, "galope"],
        ["narrador", "Até agora: Narizinho convidou os personagens das histórias para uma festa. E alguém está chegando!", 0.95, 0.8],
        ["narrador", "Uma carruagem parou no terreiro. E o Rabicó anunciou, todo atrapalhado:", 1.0, 0.4],
        ["rabico", "A senhorita Cinderela, a princesa das botinas de vidro!", 1.0, 0.5],
        ["narizinho", "Não é botina, Rabicó! É sapatinho!", 1.05, 0.5],
        ["narrador", "A Emília correu e se enfiou debaixo da cadeira da princesa, só para ver o famoso sapatinho de perto. Cinderela achou graça e pôs a boneca no colo.", 1.0, 0.5],
        ["emilia", "Um livro diz que o seu sapatinho era de cristal. Mas eu estou vendo um sapatinho de couro!", 1.05, 0.5],
        ["cinderela", "No baile, eu fui de cristal. Mas sapato de cristal aperta muito o pé! Hoje eu uso sapatinho macio.", 0.95, 0.5],
        ["emilia", "E o que aconteceu com a sua madrasta e as suas irmãs?", 1.0, 0.4],
        ["cinderela", "Eu perdoei as três. Hoje elas são boazinhas, e moram numa casinha perto do meu castelo.", 0.95, 0.5],
        ["emilia", "Como a senhora é boa! Eu não sei se perdoava...", 1.0, 0.8],
        ["visconde", "Estou vendo outra poeirinha lá longe!", 1.0, 0.8, "galope"],
        ["narrador", "Era a Branca de Neve. E ela trouxe um presente para a Emília: um espelho mágico, que responde a todas as perguntas.", 1.0, 0.5],
        ["emilia", "Senhor espelho, qual é a boneca que conta as histórias mais bonitas?", 1.0, 0.5],
        ["narrador", "E o espelho respondeu: É a ilustre Marquesa de Rabicó! E a Emília ficou toda prosa.", 1.0, 0.5, "magia"],
        ["emilia", "Onde estão os sete anõezinhos?", 1.0, 0.4],
        ["branca", "Moram comigo no castelo. São tão trabalhadores que o castelo brilha como ouro.", 0.95, 0.5],
        ["emilia", "Por que a senhora não dá um deles para a Tia Nastácia ajudar na cozinha?", 1.0, 0.4],
        ["branca", "Impossível! Se sair um, deixam de ser sete. E sete é um número mágico. Não se mexe nele!", 0.95, 0.8],
        ["narrador", "Depois chegaram duas irmãs princesas, a Rosa Branca e a Rosa Vermelha, que sempre andam juntas.", 1.0, 0.5],
        ["narrador", "A Rosa Branca casou com um urso que virou príncipe. E a Emília quis saber se ainda existia a pele do urso. Por que será?", 0.95, 2.5],
        ["emilia", "Porque eu queria tanto ver uma pele de urso que virou príncipe!", 1.05, 0.8],
        ["narrador", "Resumindo: chegaram a Cinderela, a Branca de Neve e as duas Rosas. E a Emília ganhou um espelho mágico.", 0.95, 0.8],
        ["visconde", "Estou vendo uma poeirinha tão pequenininha que parece poeira de camundongo!", 1.0, 0.6],
        ["narrador", "Quem será esse convidado tão pequeno? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    23: {"titulo": "Convidados de todas as histórias", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo vinte e três: Convidados de todas as histórias.", 0.95, 1.2],
        ["narrador", "No capítulo passado, a Cinderela, a Branca de Neve e as duas Rosas chegaram para a festa de Narizinho.", 1.0, 1.0],
        ["narrador", "Bateram na porta, bem de leve: tic, tic, tic. E o Rabicó anunciou:", 1.0, 0.4],
        ["rabico", "Um senhor pingo de gente, com umas botas maiores do que ele!", 1.0, 0.5],
        ["narrador", "Era o Pequeno Polegar! A Emília agarrou o menininho e levou para o quarto dela, para mostrar os brinquedos.", 1.05, 0.5],
        ["emilia", "Tire essas botas! Como o senhor aguenta tanto peso nos pés?", 1.0, 0.4],
        ["polegar", "Sem elas, eu não valho nada. Sou pequeno demais. Mas cada passo destas botas anda sete léguas, que é muito, muito longe!", 1.0, 0.5],
        ["emilia", "Então o senhor não tem medo nem de jacarepaguá?", 1.05, 0.4],
        ["narrador", "A Emília achava que jacarepaguá era o monstro dos monstros. Mas é só o nome de um bairro do Rio de Janeiro!", 1.0, 0.5],
        ["polegar", "Nem de jacarepaguá, nem de nada!", 1.05, 0.5],
        ["narrador", "O Polegar gostou de um cachimbo velho da Tia Nastácia. Pulou lá dentro e ficou escondidinho. Era o esconderijo perfeito!", 1.0, 0.5],
        ["narrador", "E a Emília, que adorava ganhar presentes, mas não gostava de dar, deu o cachimbo para ele. Foi o primeiro presente que ela deu na vida!", 1.0, 0.8, "transicao"],
        ["narrador", "Até agora: chegou o Pequeno Polegar, com as botas de sete léguas.", 0.95, 0.8],
        ["narrador", "O Polegar contou um segredo. Ele mora num livro de histórias, que é cuidado por uma barata velha, a Dona Carochinha.", 0.95, 0.5],
        ["narrador", "A Dona Carochinha não gosta que os personagens saiam dos livros. Mas eles querem viver aventuras novas, aqui no sítio!", 1.0, 0.6],
        ["narrador", "Depois chegou o Aladim, com a lâmpada maravilhosa. Quem esfrega a lâmpada, chama um gênio que faz tudo o que a gente pede.", 1.0, 0.5],
        ["emilia", "Me empresta a lâmpada? Só um pouquinho! Depois eu devolvo.", 1.05, 0.5],
        ["narizinho", "Não seja pidona, Emília!", 1.0, 0.6],
        ["narrador", "Chegou também o Gato de Botas. E Narizinho contou a história do falso Gato Félix, que dizia ser neto do neto do neto dele.", 1.0, 0.5],
        ["narrador", "O Gato de Botas riu muito. Ele nunca casou, nem teve filhos. Quanto mais netos!", 1.0, 0.8],
        ["narrador", "De repente, a porta abriu com força. Entrou um moço muito assustado, e trancou a porta atrás dele.", 1.05, 0.5, "porta"],
        ["narrador", "Era o Ali Babá. Quarenta ladrões estavam atrás dele, escondidos no terreiro!", 1.0, 0.5],
        ["narrador", "O que será que o Aladim fez?", 0.95, 2.5],
        ["aladim", "Deixe comigo! Gênio, vá lá fora e mande esses quarenta ladrões para bem longe!", 1.05, 0.6],
        ["narrador", "Uma fumacinha saiu da lâmpada e virou um gênio enorme. Logo depois, não havia mais nenhum ladrão no terreiro.", 1.0, 0.8, "magia"],
        ["narrador", "Até agora: o gênio da lâmpada de Aladim mandou os quarenta ladrões embora.", 0.95, 0.8],
        ["narrador", "Não paravam de chegar convidados. Veio o soldadinho de chumbo, veio o Patinho Feio, vieram João e Maria.", 1.0, 0.5],
        ["narrador", "E veio a princesa Sherazade, a que contava mil e uma histórias, com todos os heróis das Mil e Uma Noites. Era tanta gente que eles ficaram no terreiro.", 1.0, 0.5],
        ["narrador", "Lá fora tinha sultão, gênio, cavalo encantado e o marinheiro Simbá. No terreiro não cabia mais nem um alfinete!", 1.0, 0.6],
        ["emilia", "Coitada da Vaca Mocha! Ela olha para um lado e vê um rei. Olha para o outro e vê um anão. Mexe o rabo e bate numa princesa!", 1.05, 0.5],
        ["emilia", "Amanhã ela não vai dar leite nenhum, de tanto susto!", 1.05, 0.8],
        ["narrador", "Só um convidado ainda não tinha chegado: o Peter Pan, o menino que nunca quis crescer. E Pedrinho queria muito conhecer o Peter Pan.", 1.0, 0.8],
        ["narrador", "Resumindo: chegaram o Polegar, o Aladim, o Gato de Botas, o Ali Babá e muitos outros. Só falta o Peter Pan.", 0.95, 0.8],
        ["narrador", "E agora, com tanta princesa e tanto príncipe, o que fazer? Dançar, é claro!", 1.0, 0.6],
        ["narrador", "Mas no meio do baile, uma coisa muito estranha vai bater na porta. Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    24: {"titulo": "O lobo e a varinha de condão", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo vinte e quatro: O lobo e a varinha de condão.", 0.95, 1.2],
        ["narrador", "No capítulo passado, os personagens de todas as histórias chegaram para a festa de Narizinho. Só faltava o Peter Pan.", 1.0, 1.0],
        ["narrador", "Todos dançavam quando o Rabicó abriu a porta para a menina da Capinha Vermelha.", 1.0, 0.5],
        ["capinha", "Boa tarde para todos os presentes, ausentes e parentes!", 1.05, 0.5],
        ["emilia", "Qual é o seu nome certo? Capinha Vermelha ou Chapeuzinho Vermelho?", 1.0, 0.4],
        ["capinha", "Minha vovó me fez esta capinha vermelha, com capuz. Então, pode me chamar de Capinha, de Capuzinho ou de Chapeuzinho. Tanto faz!", 1.0, 0.8],
        ["narizinho", "Tia Nastácia! Traga um café bem gostoso para os nossos convidados!", 1.05, 0.5],
        ["narrador", "Tia Nastácia entrou com a bandeja e arregalou os olhos.", 1.0, 0.4],
        ["nastacia", "Credo! Onde essa menina arranja tanta gente importante? A sala até parece um céu aberto!", 1.0, 0.5],
        ["narrador", "E na saia dela vinha enganchada uma coroinha de rei. Tia Nastácia tinha passado no quintal, cheio de reis e fadas, e esbarrou num reizinho sem querer!", 1.0, 0.5],
        ["nastacia", "Não foi por querer, não! Credo!", 1.05, 0.8, "transicao"],
        ["narrador", "Até agora: chegou a Capinha Vermelha, e a Tia Nastácia serviu café para a festa.", 0.95, 0.8],
        ["narrador", "Mas o visconde, que cochilava no alto da janela, escorregou e caiu lá de cima. Ficou desacordado no chão!", 1.0, 0.5],
        ["emilia", "Já sei! Princesa Cinderela, dê uma varada nele com a sua varinha de condão. Vire o visconde num pilãozinho de socar sal, que a Tia Nastácia está precisando!", 1.05, 0.5],
        ["cinderela", "Vira que vira, vira que vira, vira virando, vira pilão!", 0.95, 0.8, "magia"],
        ["narrador", "E o visconde virou um pilãozinho novinho. A Tia Nastácia agradeceu, mas não teve coragem de usar.", 1.0, 0.5],
        ["nastacia", "Eu fico com dó. Esse pilão já foi visconde!", 1.0, 0.8],
        ["narrador", "Então ouviram uma batida esquisita na porta. Bem diferente das outras.", 0.9, 0.8, "porta"],
        ["capinha", "É o lobo! É aquele lobo da minha história!", 1.1, 0.5],
        ["narrador", "As princesas correram para o fundo da sala. Pedrinho e os príncipes estavam lá fora, longe dali. E o lobo começou a arranhar a porta!", 1.05, 0.5],
        ["emilia", "Só o visconde pode nos salvar! Os sábios sabem de tudo!", 1.05, 0.5],
        ["narrador", "A Cinderela, toda nervosa, bateu a varinha no pilão. Mas deu só meia varada.", 1.0, 0.5, "magia"],
        ["narrador", "E o visconde voltou só pela metade: visconde da cintura para cima, e pilão da cintura para baixo! Não conseguia andar!", 1.0, 0.8],
        ["narrador", "E agora? O que será que a Emília fez?", 0.95, 2.5],
        ["emilia", "Acuda, Tia Nastácia! O lobo quer entrar!", 1.1, 0.5],
        ["narrador", "Tia Nastácia veio da cozinha com a vassoura. Abriu a porta, e espantou o lobo com umas boas vassouradas no focinho!", 1.05, 0.5],
        ["nastacia", "Lobo ordinário! Vá caçar no mato, que aqui não é lugar para você!", 1.05, 0.5],
        ["narrador", "O lobo fugiu correndo. E todas as princesas abraçaram a Tia Nastácia, a mais valente da festa!", 1.0, 1.0, "festa"],
        ["narrador", "Até agora: a Tia Nastácia espantou o lobo com a vassoura.", 0.95, 0.8],
        ["narrador", "Nessa hora, alguém entrou pela janela, voando como um passarinho. Era o Peter Pan! E todos bateram palmas.", 1.05, 0.5],
        ["dona_benta", "Boa tarde, senhor Peter Pan. Só peço uma coisa: não ensine meus netos a voar. Senão, um dia, eles vão para muito longe e esquecem de voltar.", 0.95, 0.5],
        ["narrador", "Peter Pan prometeu que só ensinaria se a vovó deixasse.", 1.0, 0.6],
        ["narrador", "Às seis horas, todos foram embora, com muitos beijos e abraços. E a casa ficou vazia.", 0.95, 0.6],
        ["narrador", "Mas debaixo da mesa estava a lâmpada do Aladim. Atrás do armário, as botas do Gato de Botas. E no chão, a varinha da Cinderela. Eles esqueceram tudo!", 1.05, 0.5],
        ["emilia", "A varinha é minha! Vou brincar de virar o dia inteiro!", 1.1, 0.8],
        ["narrador", "Bateram na porta, bem fraquinho. Era a Dona Carochinha, a barata velha que toma conta dos personagens dos livros.", 0.95, 0.5],
        ["narrador", "Ela veio buscar a lâmpada, as botas, a varinha, e até o espelho mágico da Emília.", 1.0, 0.5],
        ["narrador", "As crianças ficaram tristes, mas devolveram tudo, porque a vovó ensina a respeitar os mais velhos.", 0.95, 0.5],
        ["narrador", "Só que, quando a barata chegou na porteira, a Emília não aguentou:", 1.0, 0.4],
        ["emilia", "Cara de coruja seca!", 1.1, 0.6],
        ["narrador", "E mostrou uma língua tão comprida que a Dona Carochinha saiu correndo!", 1.05, 0.8],
        ["narrador", "Resumindo: a festa foi linda, a Tia Nastácia espantou o lobo, e a Dona Carochinha levou as coisas mágicas embora.", 0.95, 0.8],
        ["narrador", "Mas a Emília ficou com uma ideia na cabeça... e o Pequeno Polegar ainda quer fugir dos livros.", 0.9, 0.6],
        ["narrador", "O que será que vai acontecer? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    25: {"titulo": "O pau que gemia", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo vinte e cinco: O pau que gemia.", 0.95, 1.2],
        ["narrador", "No capítulo passado, os personagens das histórias fizeram uma festa no sítio. E a Tia Nastácia espantou o lobo com a vassoura.", 1.0, 1.0],
        ["narizinho", "Coitada da vovó! De tanto contar histórias, ficou que nem um caju chupado. A gente espreme, espreme, e não sai mais nada.", 1.0, 0.5],
        ["narrador", "Então Dona Benta pediu livros novos a uma livraria de São Paulo. E um dia chegou pelo correio um livro chamado Pinóquio.", 1.0, 0.5],
        ["narrador", "Pinóquio é um boneco de madeira que ganha vida. E quando conta mentira, o nariz dele cresce!", 1.0, 0.5],
        ["pedrinho", "Viva! Vou ler sozinho, debaixo da jabuticabeira!", 1.1, 0.4],
        ["dona_benta", "Alto lá! Quem vai ler sou eu, para todos ouvirem. E só três capítulos por dia, para o livro durar mais.", 0.95, 0.8],
        ["narrador", "Dona Benta lia de um jeito muito gostoso. Trocava as palavras difíceis por palavras fáceis, e fazia uma vozinha fina para o Pinóquio.", 1.0, 0.6],
        ["narrador", "Depois da leitura, a Emília ficou pensando, de mãozinha no queixo.", 1.0, 0.4],
        ["emilia", "Eu tive uma grande ideia. Mas só conto se o Pedrinho me der aquele cavalinho de pau sem rabo que está na gaveta dele.", 1.0, 0.5],
        ["pedrinho", "Se a ideia for boa, eu dou. Palavra de honra!", 1.0, 0.4],
        ["emilia", "O Pinóquio foi feito de um pedaço de pau vivo. Se ainda existir pau dessa qualidade, você pode fazer um irmão para o Pinóquio!", 1.05, 0.6],
        ["pedrinho", "É mesmo! Que ideia boa! Pode pegar o cavalinho, Emília.", 1.1, 0.8, "transicao"],
        ["narrador", "Até agora: a Emília teve a ideia de fazer um irmão para o Pinóquio. E ganhou o cavalinho de pau.", 0.95, 0.8],
        ["visconde", "Não é preciso ir até a Itália, onde o Pinóquio nasceu. Talvez exista pau vivo aqui mesmo, nas nossas matas.", 0.95, 0.5],
        ["narrador", "Pedrinho pegou o machadinho e foi para a mata. Batia em cada árvore e encostava o ouvido, para ver se ela gemia.", 1.0, 0.5],
        ["narrador", "Uma semana inteira, e nenhuma árvore gemeu.", 0.95, 0.5],
        ["pedrinho", "Acho que estou fazendo papel de bobo. Pau de Pinóquio só existe na Itália.", 0.95, 0.6],
        ["narrador", "A Emília ficou preocupada. E se o Pedrinho quisesse o cavalinho de volta? Então ela teve um plano. Foi falar com o visconde.", 1.0, 0.5],
        ["emilia", "Senhor visconde, o senhor sabe gemer?", 1.0, 0.4],
        ["visconde", "Nunca gemi. Mas não deve ser difícil. Ai, ai, ai!", 0.95, 0.5],
        ["emilia", "Muito bem! Agora venha comigo. O senhor vai se esconder no buraco daquele tronco velho, perto da porteira.", 1.0, 0.5],
        ["emilia", "Quando o Pedrinho bater com o machado, o senhor geme bem gemido, com voz de pau velho!", 1.05, 0.8],
        ["narrador", "O Pedrinho voltou da mata, cansado. Passou pelo tronco e deu uma machadada, só por costume.", 1.0, 0.5],
        ["narrador", "O que será que aconteceu?", 0.95, 2.5],
        ["narrador", "O tronco gemeu: Ai! Ai! Ai! E o Pedrinho deu um pulo de susto!", 1.05, 0.5],
        ["pedrinho", "Achei! Achei o pau vivente!", 1.1, 0.5],
        ["narrador", "Ele cortou um bom pedaço do tronco e correu para casa, na maior alegria da vida.", 1.05, 0.6],
        ["narrador", "Na porta, a Emília assobiava, com a cara mais inocente do mundo.", 1.0, 0.4],
        ["emilia", "Achou o quê, Pedrinho?", 1.0, 0.6],
        ["pedrinho", "O pau gemia igualzinho a gente! Com uma voz que lembrava um pouco a do visconde...", 1.05, 0.8],
        ["narrador", "O visconde fingia ler o livro de matemática, dando risadinhas. E a Emília saiu tapando a boca, para não rir alto.", 1.0, 0.5],
        ["emilia", "O Pedrinho caiu como um pato! Agora você é meu, cavalinho. Vou arranjar um rabo de pena de galo para você!", 1.05, 1.0],
        ["narrador", "Resumindo: a Emília e o visconde enganaram o Pedrinho. O tronco não gemeu. Quem gemeu foi o visconde, escondido!", 0.95, 0.8],
        ["narrador", "E agora, com o pedaço de pau na mão, o Pedrinho queria fazer o irmão do Pinóquio. Mas como ele vai ser?", 1.0, 0.6],
        ["narrador", "Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    26: {"titulo": "O concurso de desenho", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo vinte e seis: O concurso de desenho.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o Pedrinho achou um pedaço de pau que gemia. Mas quem gemeu de verdade foi o visconde, escondido no tronco!", 1.0, 1.0],
        ["narrador", "O Pedrinho sacudia o pedaço de pau, mas ele não dava nenhum sinal de vida.", 1.0, 0.4],
        ["pedrinho", "Que esquisito! O tronco gemeu tanto, e este pedaço nem pia!", 1.0, 0.5],
        ["emilia", "É que ele está muito triste, longe do tronco pai dele. As grandes tristezas são mudas. Depois passa.", 1.0, 0.8],
        ["narrador", "Mas como fazer o boneco? Cada um queria de um jeito. Então o Pedrinho fez um concurso.", 1.0, 0.4],
        ["pedrinho", "Concurso de desenho, pessoal! Para tudo! Vovó, pegue o lápis! Tia Nastácia, deixe o fogão! Todo mundo desenhando!", 1.1, 0.8, "transicao"],
        ["narrador", "Meia hora depois, os seis desenhos estavam pregados na parede.", 1.0, 0.5],
        ["narrador", "O da Tia Nastácia parecia um rabisco de carvão. O da Narizinho era bonito, mas igualzinho ao Pinóquio.", 1.0, 0.5],
        ["narrador", "O do visconde era tão científico que ninguém entendia. Cheio de triângulos, com um xis no nariz.", 1.0, 0.5],
        ["narrador", "E o da Emília? Ela pôs boca de sapo, rabo de jacaré, orelhas de morcego, e um olho nas costas.", 1.0, 0.4],
        ["emilia", "É para ninguém pegar ele de surpresa!", 1.05, 0.8],
        ["narrador", "Na votação, cada um votava no seu desenho. Então resolveram tirar a sorte. Os nomes foram para dentro do chapéu, e a vovó ia tirar um.", 1.0, 0.5],
        ["emilia", "Quem tira sou eu! Sou eu! Sou eu!", 1.1, 0.5],
        ["narrador", "A Emília gritava com uma mão escondida no bolso. Por que será?", 0.95, 2.5],
        ["narrador", "Narizinho puxou a mão dela. E lá dentro tinha um papelzinho enrolado, igual aos do chapéu. Sabe o que estava escrito?", 1.0, 0.5],
        ["narrador", "EU! Com letrinha torta de boneca de pano.", 1.0, 0.6],
        ["dona_benta", "Emília, isso foi muito feio. Enganar os outros não se faz.", 0.95, 0.6],
        ["narrador", "Foi a primeira bronca que a Emília levou da Dona Benta. Ela saiu da sala batendo o pezinho, muito brava.", 1.0, 0.6],
        ["narrador", "E a vovó tirou o papel da sorte: Tia Nastácia! Justo o desenho mais feio! Mas sorte é sorte, e a Tia Nastácia ia fazer o boneco.", 1.0, 0.8],
        ["narrador", "Até agora: a Emília tentou trapacear no sorteio e levou uma bronca. E a Tia Nastácia ganhou o concurso.", 0.95, 0.8],
        ["narrador", "Narizinho foi atrás da Emília. Ela estava arrumando as malas: umas caixinhas de fósforo, com vestidos e brinquedos.", 1.0, 0.5],
        ["narizinho", "O que é isso, Emília?", 1.0, 0.4],
        ["emilia", "Vou me mudar. Vou morar na casa do Pequeno Polegar. Ele me convidou na festa!", 1.05, 0.5],
        ["narizinho", "Boba! O Polegar é pequenininho deste tamanho. Você não cabe na casa dele!", 1.0, 0.6],
        ["narrador", "A Emília pensou um pouco. Ia ter que dormir do lado de fora, com as corujas e os morcegos. E ela tinha muito medo deles.", 1.0, 0.5],
        ["emilia", "Então eu fico. Mas você me dá um vestido novo, de seda, com laço de fita?", 1.0, 0.5],
        ["narizinho", "Dou. Mas com uma condição: fazer as pazes com a Tia Nastácia. Ela está triste na cozinha.", 1.0, 0.5],
        ["emilia", "Só faço as pazes se ela me der o alfinete de pombinha.", 1.0, 0.6],
        ["narrador", "Era um alfinete antigo, de aço azul. No lugar da cabeça, tinha uma pombinha de vidro colorido, carijó.", 0.95, 0.5],
        ["narrador", "A Emília queria aquele alfinete mais do que tudo no mundo.", 0.95, 1.0],
        ["narrador", "Resumindo: a Emília ficou brava e quis ir embora. Mas ficou, e agora quer o alfinete de pombinha da Tia Nastácia.", 0.95, 0.8],
        ["narrador", "Enquanto isso, na cozinha, a Tia Nastácia esculpia o irmão do Pinóquio.", 1.0, 0.6],
        ["narrador", "Como será que ele ficou? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    27: {"titulo": "João Faz-de-Conta", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo vinte e sete: João Faz-de-Conta.", 0.95, 1.2],
        ["narrador", "No capítulo passado, a Emília trapaceou no sorteio. E a Tia Nastácia foi para a cozinha fazer o irmão do Pinóquio.", 1.0, 1.0],
        ["narrador", "Uma hora depois, a Tia Nastácia voltou com o boneco na mão.", 1.0, 0.4],
        ["nastacia", "Pronto! Não ficou bonito, mas ficou simpático.", 1.0, 0.6],
        ["narrador", "Todos fizeram: Oh! O boneco era muito feio.", 1.0, 0.4],
        ["narrador", "Os braços saíam do meio do corpo. O nariz era um palito de fósforo. E a cabeça, em forma de castanha de caju, estava presa por um prego torto, com a ponta saindo nas costas.", 1.0, 0.6],
        ["pedrinho", "E o pau gemeu quando você cortou, Tia Nastácia?", 1.0, 0.4],
        ["nastacia", "Nada, nada! Nem um sinalzinho de vida.", 1.0, 0.6],
        ["visconde", "Experimente assoprar o boneco, Pedrinho. Às vezes, um bom assopro dá vida às coisas.", 0.95, 0.5],
        ["emilia", "Boa ideia! Se você assoprar bem assoprado, o João Faz-de-Conta vai viver!", 1.05, 0.5],
        ["narizinho", "João Faz-de-Conta? Que nome é esse, Emília?", 1.0, 0.4],
        ["emilia", "João, porque ele tem cara de João. E Faz-de-Conta, porque só fazendo de conta para achar ele bonito. Faz de conta que não é feio. Faz de conta que não tem pé torto...", 1.05, 0.6],
        ["narrador", "Todos riram. E acharam que a Emília era a melhor botadora de nomes do sítio.", 1.0, 0.8, "transicao"],
        ["narrador", "Até agora: a Tia Nastácia fez o boneco, e a Emília deu um nome para ele: João Faz-de-Conta.", 0.95, 0.8],
        ["narizinho", "Tia Nastácia, a Emília só faz as pazes com você se ganhar o alfinete de pombinha carijó.", 1.0, 0.5],
        ["nastacia", "Um alfinete de tanta estimação? Ah, essa boneca!", 1.0, 0.5],
        ["narrador", "Mas tanto a menina pediu, que a Tia Nastácia tirou o alfinete do peito e deu.", 1.0, 0.4],
        ["nastacia", "Tome lá, interesseira! Qualquer dia você vai pedir os óculos e a dentadura da Dona Benta!", 1.0, 0.6],
        ["narrador", "A Emília bateu palmas de alegria e foi mostrar o alfinete ao cavalinho, que agora tinha um lindo rabo de pena de galo.", 1.05, 0.8],
        ["narrador", "Mas o João Faz-de-Conta não vivia. O Pedrinho assoprou o boneco três dias seguidos, até ficar com as bochechas doendo. Nada!", 1.0, 0.5],
        ["pedrinho", "Chega! Você não vive nunca, seu feioso!", 1.05, 0.5],
        ["narrador", "E jogou o boneco em cima do armário.", 1.0, 0.8],
        ["pedrinho", "Emília, cadê o meu cavalinho? Eu disse que dava se a ideia fosse boa. A ideia não deu certo!", 1.0, 0.5],
        ["emilia", "O cavalinho agora é meu!", 1.1, 0.5],
        ["narrador", "O Pedrinho deu um beliscão na boneca. E a Emília gritou tão alto que todos vieram correndo.", 1.05, 0.5],
        ["dona_benta", "Um menino desse tamanho brigando com uma bonequinha de pano? Que feio, Pedrinho!", 0.95, 0.8],
        ["narrador", "O Pedrinho foi para a mata, para a raiva passar. E parou de novo no tronco que tinha gemido.", 1.0, 0.5],
        ["narrador", "Bateu uma, duas, dez vezes. E o tronco, quietinho. Por que será que agora ele não gemia?", 0.95, 2.5],
        ["narrador", "Porque o visconde não estava mais escondido lá dentro! E o Pedrinho achou, no buraco do tronco, a cartolinha do visconde.", 1.0, 0.5],
        ["pedrinho", "Agora eu entendi! Quem gemeu foi o visconde, a mando da Emília! Eles me fizeram de bobo!", 1.05, 0.5],
        ["pedrinho", "Mas isso não fica assim. Um dia eu dou o troco!", 1.0, 1.0],
        ["narrador", "Resumindo: o João Faz-de-Conta não ganhou vida, e o Pedrinho descobriu o truque da Emília e do visconde.", 0.95, 0.8],
        ["narrador", "Mas enquanto isso, a Narizinho tirou o pobre boneco de cima do armário e levou para passear no riacho...", 0.9, 0.6],
        ["narrador", "O que será que vai acontecer lá? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    28: {"titulo": "A fada vespa", "ambiente": "riacho", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo vinte e oito: A fada vespa.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o João Faz-de-Conta não quis viver, e o Pedrinho jogou o boneco em cima do armário.", 1.0, 1.0],
        ["narrador", "Naquela tarde, a Emília não quis passear. Estava ensinando o abc ao cavalinho.", 1.0, 0.4],
        ["narizinho", "Coitado do João Faz-de-Conta. Ninguém liga para ele. Vou levar o boneco comigo. Talvez o ar do riacho faça bem.", 1.0, 0.6],
        ["narrador", "Narizinho sentou na raiz do velho pé de ingá, na beira do riacho, e fechou os olhos. O mundo ficava três vezes mais bonito de olhos fechados.", 0.95, 0.6],
        ["narrador", "Foi ali que ela viu o Príncipe Escamado pela primeira vez, lembra?", 0.95, 0.8],
        ["narrador", "De repente, ouviu um bocejo: Aaaah! Era o João Faz-de-Conta, se espreguiçando!", 1.0, 0.8, "magia"],
        ["narizinho", "Até que enfim! Eu sabia que o ar do riacho ia fazer você viver!", 1.05, 0.5],
        ["fazdeconta", "Eu sempre fui vivo, Narizinho. Quem mudou foi você. E agora você vai ver uma coisa que sempre existiu aqui, mas você nunca viu. Olhe lá!", 0.95, 0.8, "transicao"],
        ["narrador", "Até agora: perto do riacho, o João Faz-de-Conta acordou e começou a falar.", 0.95, 0.8],
        ["narrador", "Entre as árvores do pomar, umas moças vestidas de véu dançavam de mãos dadas. No meio delas, um ser de orelhas pontudas tocava uma flauta.", 0.95, 0.5],
        ["narizinho", "Eu vi isso num livro da vovó! São as ninfas do bosque, e ele é um fauno!", 1.05, 0.5],
        ["narrador", "Mas os dançarinos ouviram a menina e fugiram correndo. O fauno até deixou cair a flauta!", 1.05, 0.5],
        ["narrador", "Narizinho pegou a flauta. Só que ela era de barro, e estava cheia de vespas! As vespas saíram voando, e só uma ficou presa nos dedos da menina.", 1.0, 0.8, "zumbido"],
        ["fazdeconta", "Eu conheço essa vespa. Acho que é uma fadinha disfarçada.", 0.95, 0.5],
        ["vespa", "Sou fada, sim. Há muitos anos eu procuro um alfinete mágico. Sem ele, não consigo virar uma fada completa.", 0.95, 0.5],
        ["narizinho", "Não é um alfinete de pombinha carijó?", 1.0, 0.4],
        ["vespa", "Isso mesmo! Como você sabe?", 1.05, 0.6],
        ["narrador", "Narizinho percebeu que tinha falado demais. O alfinete estava com a Emília!", 1.0, 0.4],
        ["narizinho", "Eu sonhei. Quer dizer... acho que era de galo, não de pombinha.", 0.95, 0.5],
        ["fazdeconta", "Não conte, Narizinho. Com fadas, é preciso cuidado. Umas são boas, outras não.", 0.9, 0.5],
        ["vespa", "Estou ouvindo tudo! E vou dar uma ferroadinha no nariz dessa menina!", 1.05, 0.5],
        ["narrador", "O que será que o João Faz-de-Conta fez?", 0.95, 2.5],
        ["narrador", "Ele pulou na frente da menina, puxou o prego da cabeça e usou como espada. Mas, sem o prego, a cabeça dele caiu, rolou morro abaixo e foi parar no riacho!", 1.05, 0.5, "agua"],
        ["narrador", "A vespa levou um susto tão grande de ver um boneco sem cabeça, de prego na mão, que sumiu no ar.", 1.0, 0.8],
        ["narrador", "Resumindo até aqui: o Faz-de-Conta salvou a Narizinho da vespa, mas perdeu a cabeça no riacho!", 0.95, 0.8],
        ["narrador", "Narizinho viu uma fumacinha lá longe e correu para lá. Era uma casinha linda, de janelas verdes. Ela bateu na porta.", 1.0, 0.5],
        ["capinha", "Narizinho! Que alegria! Há quanto tempo eu espero a sua visita!", 1.1, 0.5],
        ["narrador", "Era a casa da Capinha Vermelha! E ela tinha uma surpresa.", 1.0, 0.4],
        ["capinha", "Eu estava tomando banho no riacho, e uma coisa em forma de castanha de caju bateu em mim. Era uma cabeça, com boca e nariz!", 1.0, 0.5],
        ["narizinho", "É a cabeça do Faz-de-Conta!", 1.1, 0.5],
        ["narrador", "Em meio minuto, a cabeça voltou para o lugar. E o João Faz-de-Conta contou tudo para a Capinha.", 1.0, 0.5],
        ["capinha", "Quem me dera ter um amigo leal e valente assim!", 0.95, 0.5],
        ["narrador", "Narizinho contou um segredo para a Capinha, bem baixinho: o alfinete da Emília era uma varinha de condão. A Emília podia virar fada!", 0.9, 0.5],
        ["fazdeconta", "Adivinhem quem passou aqui fora? O Peter Pan! Ele disse que qualquer dia vai ao sítio brincar com o Pedrinho.", 1.05, 0.8],
        ["narrador", "Já estava ficando tarde. Narizinho fechou os olhos bem fechados, para guardar aquele sonho bonito na lembrança.", 0.9, 0.8],
        ["narrador", "Resumindo: o João Faz-de-Conta viveu, salvou a Narizinho, e eles visitaram a Capinha Vermelha.", 0.95, 0.8],
        ["narrador", "Mas o que será que a Narizinho vai ver quando abrir os olhos? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    29: {"titulo": "O alfinete de pombinha", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo vinte e nove: O alfinete de pombinha.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o João Faz-de-Conta ganhou vida na beira do riacho, salvou a Narizinho de uma vespa e visitou a Capinha Vermelha.", 1.0, 1.0],
        ["narrador", "Quando Narizinho abriu os olhos, estava de novo na raiz do pé de ingá, com o Faz-de-Conta no colo.", 0.95, 0.5],
        ["narrador", "Mas ele estava parado, mudo, como um boneco de pau qualquer. Narizinho sacudiu o boneco, como se sacode um relógio parado. Nada!", 0.95, 0.5],
        ["narizinho", "Que pena. Voltei para o dia de todo dia. Tão sem graça...", 0.9, 0.8, "transicao"],
        ["narrador", "Ela correu para casa, porque já era quase noite.", 1.05, 0.4],
        ["narizinho", "Vovó! O Faz-de-Conta viveu mais de uma hora! E eu vi ninfas, e um fauno, e quebrei a flauta, e saíram vespas, e uma era fada, e...", 1.15, 0.4],
        ["dona_benta", "Pare, pare, menina! Você me deixa tonta. Vá contar essa história para o Pedrinho.", 0.95, 0.8],
        ["narrador", "Até agora: Narizinho voltou para casa cheia de novidades. Mas a vovó não entendeu nada.", 0.95, 0.8],
        ["narrador", "O Pedrinho chegou da mata, carrancudo. Ainda pensava em como se vingar da Emília e do visconde.", 1.0, 0.4],
        ["narizinho", "Pedrinho, tenho três novidades! A primeira: o Faz-de-Conta viveu! E ele é muito mais sensato que o Pinóquio. É valente e leal.", 1.05, 0.5],
        ["pedrinho", "Mas como ele viveu, se não era de pau vivente?", 1.0, 0.4],
        ["narizinho", "Não sei explicar. Às vezes, a gente muda de estado e começa a ver coisas maravilhosas que estão em volta da gente.", 0.95, 0.6],
        ["narizinho", "A segunda novidade: eu visitei a Capinha Vermelha. E o Peter Pan prometeu vir aqui brincar com você!", 1.05, 0.5],
        ["narrador", "O Pedrinho deu pulos de alegria. Conhecer o Peter Pan era o que ele mais queria no mundo.", 1.05, 0.5],
        ["narizinho", "E a terceira é a mais importante. Aquele alfinete de pombinha que a Tia Nastácia deu para a Emília é uma varinha de condão!", 1.0, 0.5],
        ["narrador", "Uma varinha de condão faz mágica. Com ela, a Emília podia virar fada!", 1.0, 0.8, "magia"],
        ["narrador", "O Pedrinho esqueceu da vingança na mesma hora. Agora, ele queria era agradar a futura fadinha.", 1.0, 0.5],
        ["pedrinho", "Emília, querida! Vou fazer três cavalinhos novos para você, cada um de uma cor. E uma casinha, e um fogãozinho, e um balanço, e umas asinhas para você voar...", 1.0, 0.5],
        ["narrador", "A Emília estranhou tanta gentileza. E foi arregalando os olhos, arregalando, arregalando...", 0.9, 0.5],
        ["narrador", "O que será que aconteceu com os olhos dela?", 0.95, 2.5],
        ["narrador", "Pluft! Arrebentaram! É que os olhos da Emília eram de linha de bordar. E quando ela arregalava demais, a linha arrebentava.", 1.0, 0.5],
        ["emilia", "Malvado! Olha o que você me fez!", 1.05, 0.6],
        ["narrador", "Mas não se preocupe. A Tia Nastácia tem linha de sobra, e bordou olhos novinhos na Emília.", 1.0, 1.0],
        ["narrador", "Resumindo: o João Faz-de-Conta voltou a ser boneco. E todos descobriram que o alfinete de pombinha é mágico.", 0.95, 0.8],
        ["narrador", "Com uma varinha de condão na mão, o que será que a Emília vai aprontar?", 0.95, 0.6],
        ["narrador", "Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    30: {"titulo": "A operação do visconde", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo trinta: A operação do visconde.", 0.95, 1.2],
        ["narrador", "No capítulo passado, todos descobriram que o alfinete de pombinha da Emília é uma varinha de condão.", 1.0, 1.0],
        ["narrador", "Um dia, no sítio, fizeram um concurso de quem tinha a melhor ideia. E quem ganhou foi a Emília, com a ideia de um circo!", 1.0, 0.5],
        ["emilia", "Vai se chamar Circo de Escavalinho!", 1.05, 0.4],
        ["dona_benta", "Não é escavalinho, Emília. É circo de cavalinhos.", 0.95, 0.4],
        ["emilia", "Não, senhora! Escavalinho é o nome do dono do circo: o senhor Pedro Malazarte Escavalinho da Silva!", 1.05, 0.6],
        ["narrador", "O Pedrinho adorou. A Emília ia correr no cavalinho e pular arcos. O João Faz-de-Conta ia engolir espadas e comer fogo. Mas quem ia ser o palhaço?", 1.0, 0.5],
        ["pedrinho", "O visconde! Mas antes, precisamos curar essa mania de sábio dele.", 1.0, 0.6],
        ["narrador", "Nessa hora, ouviram gritos na cozinha. A Tia Nastácia espantava o visconde com o cabo da vassoura.", 1.0, 0.4],
        ["nastacia", "Esse visconde vem me falar de seno, cosseno e mangaritmo! Eu já disse que não sei inglês!", 1.05, 0.5],
        ["visconde", "Mangaritmo?! É logaritmo!", 1.0, 0.5],
        ["narrador", "E o visconde caiu duro no chão, de tanta conta na cabeça.", 1.0, 0.8, "transicao"],
        ["narrador", "Até agora: a Emília inventou um circo, mas o visconde passou mal de tanto estudar.", 0.95, 0.8],
        ["narrador", "O Rabicó foi chamar o Doutor Caramujo. Ele chegou correndo, com a maletinha.", 1.0, 0.4],
        ["caramujo", "Hum! O caso é sério. Sua Excelência está entupido de matemática. Vou tirar isso tudo.", 0.95, 0.5],
        ["narrador", "O visconde é feito de sabugo de milho, e sabugo não sente dor. Então o doutor abriu uma portinha na barriga dele, como quem abre uma caixinha.", 1.0, 0.5],
        ["narrador", "E lá dentro, em vez de tripas, havia um bolo de letras, números e sinais de conta!", 1.0, 0.5],
        ["pedrinho", "Não tire todas as letras, doutor! Senão ele fica bobo demais. Deixe algumas para semente.", 1.0, 0.5],
        ["narrador", "O doutor tirou tudo com uma colherzinha e fechou a portinha com cola. E a Narizinho teve uma ideia.", 1.0, 0.4],
        ["narizinho", "Vamos pôr umas piadas dentro da barriga dele, já que ele vai ser palhaço!", 1.05, 0.5],
        ["narrador", "E puseram três páginas de um livro de piadas, bem dobradinhas. Três dias depois, o visconde estava curado.", 1.0, 0.8],
        ["narrador", "O Pedrinho escreveu o programa do circo: Emília no cavalinho, o homem que come fogo, o palhaço Sabugueira. E, no fim, uma grande surpresa!", 1.0, 0.8],
        ["narrador", "Resumindo até aqui: o doutor tirou a matemática da barriga do visconde, e ele vai ser o palhaço do circo.", 0.95, 0.8],
        ["narrador", "Mas para fazer um circo de verdade, o Pedrinho precisava comprar pano. E o pano custava dez mil réis. Ele só tinha cinco.", 1.0, 0.5],
        ["narrador", "A Dona Benta e a Tia Nastácia compraram os ingressos das cadeiras, com dinheiro de verdade. Mesmo assim, faltava dinheiro.", 1.0, 0.5],
        ["emilia", "Eu arranjo o dinheiro. Mas só se o Pedrinho me der o carrinho de rodas de carretel.", 1.0, 0.4],
        ["pedrinho", "Uma boneca arranjar dinheiro? Pode ficar com o carrinho, se conseguir!", 1.05, 0.6],
        ["narrador", "Naquela tarde, a Dona Benta perdeu os óculos. Procuraram pela casa toda e não acharam.", 1.0, 0.4],
        ["emilia", "Eu procuro, Dona Benta. Mas cobro três mil réis, em notas novas.", 1.0, 0.5],
        ["dona_benta", "Três mil réis? Que boneca interesseira! Está bem, procure.", 0.95, 0.5],
        ["narrador", "Cinco minutos depois, a Emília voltou com os óculos. Onde será que eles estavam?", 0.95, 2.5],
        ["emilia", "No bolso da sua saia amarela, Dona Benta!", 1.05, 0.5],
        ["narrador", "A Dona Benta não entendeu nada. Fazia muito tempo que ela não usava aquela saia!", 1.0, 0.6],
        ["nastacia", "Foi ela quem escondeu os óculos, sinhá! Só para ganhar o dinheiro. Essa boneca está esperta demais!", 1.0, 0.8],
        ["narrador", "Resumindo: a Emília conseguiu o dinheiro do circo. Mas parece que foi com uma esperteza.", 0.95, 0.8],
        ["narrador", "Agora o Pedrinho já pode construir o circo. Como será que vai ficar? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    31: {"titulo": "O circo fica pronto", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo trinta e um: O circo fica pronto.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o visconde foi curado da mania de sábio. E a Emília arranjou o dinheiro para o circo.", 1.0, 1.0],
        ["narrador", "Construir o circo deu muito trabalho. O Pedrinho cavou mais de trinta buracos para fincar os paus. Ficou até com calos nas mãos!", 1.0, 0.5],
        ["emilia", "Se eu fosse você, arranjava um tatu para cavar os buracos. Tatu cava buraco redondinho!", 1.05, 0.5],
        ["pedrinho", "E se eu fosse você, ia pentear macaco!", 1.0, 0.6],
        ["narrador", "Enquanto isso, Narizinho costurava as roupas. A do palhaço tinha um grande sol amarelo nas costas.", 1.0, 0.4],
        ["emilia", "Eu também quero um sol nas costas!", 1.05, 0.4],
        ["narizinho", "Sol, só palhaço usa. Você pode ter uma lua. Quarto crescente.", 1.0, 0.4],
        ["emilia", "Quarto não quero. Quero sala crescente!", 1.1, 0.6],
        ["narrador", "Narizinho riu e deu um abraço na Emília.", 1.0, 0.8],
        ["narrador", "Depois de três dias, o circo ficou pronto, com pano e tudo. Igualzinho a um circo de verdade!", 1.05, 0.5, "festa"],
        ["dona_benta", "Esse meu neto, quando crescer, vai ser um grande homem!", 0.95, 0.8],
        ["narrador", "A tia Moema emprestou o cachorro dela, o Maroto, para tomar conta da porta do circo. Ele tinha ordem de latir, mas não de morder.", 1.0, 0.5],
        ["narrador", "Na hora dos ensaios, a Emília quis escolher o nome da peça de teatro: O Pantasma da Ópera.", 1.0, 0.4],
        ["narizinho", "É fantasma, Emília, com efe!", 1.0, 0.4],
        ["emilia", "Não! Pantasma é um bicho que eu inventei. Tem olho no pé, pé no nariz, nariz no umbigo, umbigo no calcanhar...", 1.1, 0.8, "transicao"],
        ["narrador", "Até agora: o circo ficou pronto, e a Emília inventou um bicho maluco chamado Pantasma.", 0.95, 0.8],
        ["narrador", "Bum! Bum! Bum! Chegou o grande dia. O terreiro estava enfeitado de bandeirinhas. O primeiro a chegar foi o Doutor Caramujo.", 1.0, 0.5],
        ["narizinho", "E o príncipe? Por que ele não veio?", 1.0, 0.4],
        ["caramujo", "Depois da visita ao sítio, o nosso príncipe nunca mais voltou ao reino. Ninguém sabe onde ele está.", 0.9, 0.6],
        ["narrador", "Narizinho ficou muito triste. Será que o príncipe estava perdido? Ela enxugou três lagrimazinhas. Mas os convidados estavam chegando, e ela não quis estragar a festa.", 0.9, 0.8],
        ["narrador", "Chegaram a Dona Aranha com as seis filhas, os caranguejos, o Major Agarra. Ouviram um miado. Seria o falso Gato Félix?", 1.0, 0.5],
        ["narrador", "Não. Era o Gato Félix verdadeiro, o dos filmes!", 1.05, 0.5],
        ["gato", "E tem cocada?", 1.05, 0.4],
        ["emilia", "Só no intervalo! Branca, cor-de-rosa e queimadinha. A Tia Nastácia faz o doce mais gostoso do mundo!", 1.05, 0.6],
        ["narrador", "O João Faz-de-Conta acendeu os lampiões, vestido de casaca verde. Mas o prego furava a casaca nas costas, e o Major Agarra gritou:", 1.0, 0.4],
        ["major", "Arranca o prego!", 1.05, 0.5],
        ["narrador", "O Faz-de-Conta tinha bom gênio e não ligou. E aí começaram a chegar os convidados das histórias.", 1.0, 0.5],
        ["narrador", "O Aladim, o Gato de Botas, o Pequeno Polegar, a Capinha Vermelha, as duas Rosas, o Ali Babá, a Alice do País das Maravilhas... quase todo mundo!", 1.05, 0.5],
        ["pedrinho", "Só não veio quem eu mais queria. O Peter Pan...", 0.9, 0.5],
        ["narizinho", "Talvez ele ainda venha. Ele gosta de fazer tudo diferente.", 0.95, 0.6],
        ["narrador", "E o Pequeno Polegar não parava de gritar: Palhaço! Palhaço!", 1.05, 0.5],
        ["narrador", "O Pedrinho bateu com um martelo numa enxada velha pendurada num barbante: blém, blém, blém! Era o sinal. O espetáculo ia começar!", 1.05, 0.8],
        ["narrador", "Resumindo: o circo ficou pronto, e todos os convidados chegaram. Mas o príncipe sumiu, e Narizinho está preocupada.", 0.95, 0.8],
        ["narrador", "E agora, respeitável público, vai começar o grande espetáculo! O que será que vai acontecer? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    32: {"titulo": "O espetáculo", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo trinta e dois: O espetáculo.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o circo ficou pronto, e os convidados de todas as histórias chegaram.", 1.0, 1.0],
        ["narrador", "O Pedrinho esperou a vovó chegar. Ela veio com o vestido amarelo mais bonito que tinha, do tempo do Imperador. E a Tia Nastácia veio com outro vestido antigo, emprestado pela Dona Benta.", 1.0, 0.5],
        ["narizinho", "Respeitável público! Apresento a minha vovó, Dona Benta! E a Tia Nastácia, a melhor doceira do mundo, que fez as cocadas de hoje!", 1.05, 0.5],
        ["narrador", "Todos bateram palmas.", 1.0, 0.5],
        ["narrador", "Blém, blém, blém! A cortina abriu, e a Emília entrou montada no cavalinho de rabo de pena.", 1.05, 0.5],
        ["narrador", "Ela deu várias voltas. Umas sentada de lado, outras de pé, num pé só!", 1.05, 0.5],
        ["dona_benta", "Nunca pensei que a Emília fosse tão boa! Parece o Tom Mix!", 1.0, 0.5],
        ["narrador", "O João Faz-de-Conta segurou os arcos de papel. A Emília pulou o arco vermelho, depois o azul. Que palmas!", 1.05, 0.5],
        ["narrador", "Mas no arco verde, o cavalinho também quis pular. E o rabo de pena dele ficou enganchado no prego das costas do Faz-de-Conta!", 1.05, 0.5],
        ["narrador", "O público riu tanto! A Emília achou que estavam rindo dela, mostrou a língua e saiu brava.", 1.0, 0.5],
        ["emilia", "Não brinco mais! Não sou palhaço de ninguém!", 1.1, 0.8, "transicao"],
        ["narrador", "Até agora: a Emília correu no cavalinho, mas o rabo de pena ficou preso no prego do Faz-de-Conta.", 0.95, 0.8],
        ["narrador", "E o palhaço? O visconde tinha sumido! O Pedrinho procurou por todo lado e não achou.", 1.0, 0.5],
        ["pedrinho", "Faz-de-Conta, vá engolindo espadas e comendo fogo, enquanto eu procuro o visconde!", 1.05, 0.5],
        ["narrador", "No circo, essas coisas são truques. O Faz-de-Conta engoliu as espadas de mentirinha, e todos bateram palmas.", 1.0, 0.5],
        ["narrador", "Mas quando foi comer o fogo, o nariz dele encostou na brasa. E o nariz era um palito de fósforo! O que será que aconteceu?", 0.95, 2.5],
        ["narrador", "Pegou fogo! Os convidados pularam no picadeiro e apagaram o nariz bem depressa. Só sobrou um toquinho.", 1.05, 0.5],
        ["narrador", "E sabe que ele até ficou mais bonito? O que deixava o Faz-de-Conta feio era aquele nariz de fósforo!", 1.0, 0.8],
        ["pedrinho", "Respeitável público! O nosso palhaço sumiu, e a Emília não quer mais trabalhar. Então, vamos direto para a grande SURPRESA!", 1.05, 0.5],
        ["narrador", "E entrou o menor elefante do mundo! Com tromba, orelhas grandes e tudo. Ele foi andando, devagarinho, em cima de umas garrafas.", 1.0, 0.5],
        ["polegar", "Berra, elefante!", 1.1, 0.5],
        ["narrador", "O elefante berrou. Mas o berro parecia de porco: Óinc, óinc, óinc!", 1.05, 0.5],
        ["narrador", "O Maroto, lá fora, ouviu e entrou por baixo do pano, latindo. O elefante levou um susto e caiu das garrafas. E a fantasia de elefante rasgou no meio!", 1.05, 0.5],
        ["narrador", "E de dentro saiu... o Marquês de Rabicó! Era ele, fantasiado!", 1.05, 0.5],
        ["narrador", "O circo quase veio abaixo de tanta risada. O Rabicó fugiu correndo para o terreiro.", 1.05, 0.8, "festa"],
        ["narrador", "Narizinho entrou com uma placa: INTERVALO. E todos correram para pegar as cocadas da Tia Nastácia.", 1.0, 0.5],
        ["gato", "Quero uma branca, duas cor-de-rosa e três queimadinhas!", 1.05, 0.6],
        ["narrador", "E o visconde? Ele achou um livro velho de matemática no meio dos paus do circo. Ficou tão feliz que saiu passeando com o livro, e só voltou no outro dia!", 1.0, 0.5],
        ["narrador", "O doutor tinha deixado umas letras na barriga dele, para semente, lembra? E elas brotaram.", 1.0, 1.0],
        ["narrador", "Resumindo: o espetáculo foi uma confusão. Mas todo mundo riu muito, e as cocadas fizeram sucesso.", 0.95, 0.8],
        ["narrador", "Só a Narizinho continuava pensando no príncipe. Onde ele estará? Será que alguém pode ajudar a encontrar o príncipe?", 0.9, 0.6],
        ["narrador", "Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    33: {"titulo": "O menino invisível", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo trinta e três: O menino invisível.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o circo foi uma confusão divertida. E Narizinho continuava preocupada com o príncipe.", 1.0, 1.0],
        ["narrador", "Depois que a vovó contou a história de Peter Pan, o Pedrinho não pensava em outra coisa. Ele também não queria crescer.", 1.0, 0.5],
        ["narrador", "Um dia, ele estava em cima da goiabeira, comendo goiaba, e falou sozinho:", 1.0, 0.4],
        ["pedrinho", "Que pena ter que crescer. Se houvesse um jeito de ser menino para sempre...", 0.9, 0.6],
        ["penninha", "Existe uma coisa ainda melhor.", 0.95, 0.6],
        ["narrador", "O Pedrinho levou um susto. Olhou para todos os lados, e não viu ninguém!", 1.0, 0.4],
        ["pedrinho", "Quem está falando?", 1.0, 0.4],
        ["penninha", "Eu! Tenho a sua altura e a sua idade. E sei o jeito de deixar uma pessoa invisível, como eu.", 0.95, 0.5],
        ["narrador", "Invisível quer dizer que ninguém consegue ver. O Pedrinho ficou tão espantado que escorregou da goiabeira. Por sorte, a árvore era baixinha.", 1.0, 0.5],
        ["penninha", "Eu ensino você e sua prima. Mas antes, vocês precisam viajar comigo pelo Mundo das Maravilhas.", 0.95, 0.5],
        ["narrador", "E um mapa dobrado se abriu sozinho no ar. Era o menino invisível segurando! O Pedrinho pegou o mapa.", 1.0, 0.5],
        ["pedrinho", "Que lindo! Até o sítio da vovó está aqui, com o chiqueirinho do Rabicó!", 1.05, 0.5],
        ["pedrinho", "Combinado. Amanhã de madrugada, estou aqui com a minha prima.", 1.0, 0.4],
        ["narrador", "E a voz respondeu: Cocoricocó! Igualzinho a um galo. O Peter Pan também canta assim. Será que era ele?", 1.0, 0.8, "transicao"],
        ["narrador", "Até agora: uma voz invisível convidou o Pedrinho para uma viagem pelo Mundo das Maravilhas.", 0.95, 0.8],
        ["narrador", "O Pedrinho contou tudo para a Narizinho. E decidiram quem ia: Narizinho, Pedrinho, a Emília e o visconde.", 1.0, 0.5],
        ["narrador", "De noite, a Emília arrumou uma malinha de couro: uma pena de papagaio, uma perna de tesoura, o alfinete de pombinha e mais um monte de coisas.", 1.0, 0.5],
        ["emilia", "A gente precisa se precaver!", 1.0, 0.4],
        ["narrador", "A malinha ficou tão cheia que não fechava. O visconde teve que sentar em cima para fechar.", 1.0, 0.8],
        ["narrador", "De madrugada, sem a vovó perceber, foram na ponta dos pés até o pomar. O visconde ia atrás, com a malinha nas costas.", 0.95, 0.5],
        ["narrador", "Cocoricocó! A voz cantou no alto da goiabeira. E uma casca de goiaba caiu na cabeça do Pedrinho.", 1.0, 0.5],
        ["penninha", "Aprendeu a saber onde estou, sem me ver?", 1.0, 0.6],
        ["pedrinho", "É muito difícil andar com um amigo que a gente não vê. Ponha uma pena na cabeça, assim a gente sabe onde você está!", 1.0, 0.5],
        ["emilia", "Tenho uma pena de papagaio na minha malinha!", 1.05, 0.5],
        ["narrador", "Amarraram a pena na testa do menino invisível. E agora a pena flutuava no ar, mostrando onde ele estava.", 1.0, 0.5],
        ["emilia", "Viva o Peninha!", 1.1, 0.8],
        ["narrador", "E daí em diante, o menino invisível ficou se chamando Peninha.", 1.0, 0.8],
        ["narrador", "Resumindo até aqui: o menino invisível ganhou uma pena de papagaio na testa, e um nome: Peninha.", 0.95, 0.8],
        ["narrador", "Para viajar, o Peninha deu uma pitada de pó para cada um cheirar. Era o pó de pirlimpimpim, o pó mais mágico que as fadas inventaram!", 1.0, 0.5],
        ["narrador", "A Emília espirrou: Atchim! Todos ficaram leves como pluma, e saíram flutuando pelo ar, como bolhas de sabão.", 1.0, 0.8, "magia"],
        ["narrador", "Logo sentiram o chão debaixo dos pés. Tinham chegado num vale verde, com um rio de água clarinha e uma montanha azul.", 0.95, 0.5],
        ["penninha", "Estamos no País das Fábulas, a terra dos animais falantes.", 0.95, 0.6],
        ["narrador", "Fábulas são historinhas curtas de bichos que falam e que ensinam alguma coisa.", 1.0, 0.8],
        ["narrador", "Resumindo: com o pó de pirlimpimpim, todos voaram até o País das Fábulas.", 0.95, 0.8],
        ["narrador", "E à beira do rio, um carneirinho branco ia beber água... Quem será que vem chegando? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    34: {"titulo": "A cigarra e a formiga", "ambiente": "riacho", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo trinta e quatro: A cigarra e a formiga.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o menino invisível, o Peninha, levou todos para o País das Fábulas, com o pó de pirlimpimpim.", 1.0, 1.0],
        ["narrador", "Na beira do rio, apareceu um homem de cabeleira cacheada, sapato de fivela e chapéu de três pontas.", 1.0, 0.5],
        ["penninha", "É o senhor de La Fontaine, um francês muito sábio. Ele observa os bichos e escreve fábulas.", 0.95, 0.5],
        ["narrador", "O carneirinho bebia água no rio. Então saiu da floresta um lobo, de cara feia.", 0.95, 0.5],
        ["lobo", "Que desaforo é esse de sujar a água que eu vou beber?", 1.0, 0.4],
        ["carneirinho", "Desculpe, senhor lobo. Mas o senhor está na parte de cima do rio. A água desce de lá para cá. Eu não posso sujar a sua água.", 0.95, 0.5],
        ["lobo", "E no ano passado, você falou mal de mim!", 1.0, 0.4],
        ["carneirinho", "Não pode ser, senhor lobo. Eu só tenho três meses!", 0.95, 0.5],
        ["emilia", "E eles falam mesmo! Falam igualzinho gente!", 1.05, 0.5],
        ["narrador", "O lobo não tinha mais desculpas e avançou para o carneirinho. Mas o senhor de La Fontaine pulou da moita e deu uma bengalada no focinho dele!", 1.05, 0.5],
        ["narrador", "O lobo fugiu para a floresta, de rabo entre as pernas.", 1.0, 0.8, "transicao"],
        ["narrador", "Até agora: o senhor de La Fontaine salvou o carneirinho do lobo.", 0.95, 0.8],
        ["narizinho", "Au revoir, senhor de La Fontaine!", 1.0, 0.4],
        ["narrador", "Au revoir é tchau, em francês. Narizinho queria dizer bom dia, que é bonjour, mas se atrapalhou! O sábio achou graça e deu um beijo na testa dela.", 1.0, 0.5],
        ["lafontaine", "Pode falar português comigo. Eu entendo todas as línguas, dos bichos e das gentes.", 0.95, 0.6],
        ["narrador", "A Emília achou a cabeleira dele muito comprida. Tirou da malinha a perna de tesoura e ofereceu.", 1.0, 0.4],
        ["emilia", "É para o senhor cortar o cabelo.", 1.0, 0.4],
        ["lafontaine", "Mas essa tesoura só tem uma perna, bonequinha!", 1.0, 0.4],
        ["emilia", "Então corte o cabelo de um lado só!", 1.05, 0.8],
        ["narrador", "O senhor de La Fontaine riu muito. E levou todos para ver outra fábula: a da cigarra e da formiga.", 1.0, 0.8],
        ["narrador", "Até agora: os meninos conheceram o senhor de La Fontaine, e a Emília ofereceu meia tesoura para ele.", 0.95, 0.8],
        ["narrador", "Era inverno na fábula. A cigarra, que tinha passado o verão inteiro cantando, estava com fome e com frio. Bateu na porta do formigueiro.", 0.95, 0.5, "porta"],
        ["narrador", "Uma formiga velhinha e rabugenta abriu a porta.", 0.95, 0.4],
        ["formiga_velha", "O que a senhora quer?", 0.95, 0.4],
        ["cigarra", "Estou com fome e com frio. A senhora me dá uma folhinha e um cantinho para ficar?", 0.9, 0.5],
        ["formiga_velha", "E o que você fazia no verão, enquanto a gente trabalhava?", 0.95, 0.4],
        ["cigarra", "Eu cantava, senhora formiga.", 0.9, 0.4],
        ["formiga_velha", "Cantava? Pois agora dance!", 1.0, 0.5],
        ["narrador", "E fechou a porta no nariz da cigarra. O que será que a Emília fez?", 0.95, 2.5],
        ["emilia", "Não fique triste, cigarra! Tome umas folhinhas para comer. E vamos ensinar uma coisa para aquela formiga.", 1.05, 0.5],
        ["narrador", "A Emília fez uma fogueirinha para a cigarra se esquentar. E bateu de novo na porta do formigueiro.", 1.0, 0.5],
        ["emilia", "Dona formiga! A cigarra trabalhou, sim! Ela cantou o verão inteiro para alegrar a senhora enquanto trabalhava. Cantar também é trabalho!", 1.05, 0.5],
        ["narrador", "A formiga velha ficou quieta, pensando. E percebeu que, sem a música da cigarra, o verão tinha sido bem mais chato.", 0.95, 0.5],
        ["formiga_velha", "Está bem. Entre, cigarra. Tem comida para todas. E à noite, você canta para nós.", 0.95, 0.6],
        ["lafontaine", "Que interessante! Essa bonequinha mudou o fim da minha fábula. E ficou mais bonita!", 0.95, 0.8, "festa"],
        ["narrador", "Resumindo: a Emília ajudou a cigarra, e a formiga aprendeu a dividir.", 0.95, 0.8],
        ["narrador", "Nessa hora, ouviram ao longe: Cocoricocó! Era o Peninha voltando. E vinha com mais um sábio.", 1.0, 0.6],
        ["narrador", "Quem será? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    35: {"titulo": "O leão e o burro", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo trinta e cinco: O leão e o burro.", 0.95, 1.2],
        ["narrador", "No capítulo passado, a Emília ajudou a cigarra, e a formiga aprendeu a dividir.", 1.0, 1.0],
        ["narrador", "O Peninha chegou com um homem enrolado num pano branco, como os gregos antigos.", 1.0, 0.4],
        ["emilia", "Quem é esse moço que perdeu a roupa e veio embrulhado na toalha?", 1.05, 0.5],
        ["lafontaine", "Não é toalha, bonequinha. É a roupa dos gregos de antigamente. Este é Esopo, o primeiro homem que escreveu fábulas.", 0.95, 0.5],
        ["esopo", "Uma bonequinha que fala! Em Atenas, eu vi muitas bonecas, mas todas mudas. Diga uma bobagem para o velho Esopo ver.", 0.95, 0.5],
        ["emilia", "Assim, de encomenda, eu não sei...", 1.0, 0.8],
        ["narrador", "Os dois sábios foram conversar numa pedra. E os meninos seguiram o Peninha floresta adentro.", 1.0, 0.5, "transicao"],
        ["narrador", "Até agora: os meninos conheceram Esopo e foram explorar a floresta.", 0.95, 0.8],
        ["narrador", "No caminho, encontraram a Menina do Leite, a Laura, com um pote de leite na cabeça.", 1.0, 0.4],
        ["laura", "Vou vender este leite e comprar ovos. Dos ovos nascem pintinhos. Vendo as galinhas e compro porcos. Vendo os porcos e compro vacas!", 1.05, 0.5],
        ["narrador", "Na fábula, ela sonha tanto que tropeça, e o pote cai e quebra. A Emília esperava o pote cair. Mas não caía!", 1.0, 0.5],
        ["laura", "Isso já aconteceu uma vez. Agora eu uso uma lata bem fechada. Assim posso sonhar quanto quiser, sem derramar o leite!", 1.05, 0.8],
        ["narrador", "E foi embora, sorrindo. Depois, chegaram na montanha do leão, o rei dos animais.", 1.0, 0.4],
        ["penninha", "Tem uma fresta lá no alto da pedra. De lá, a gente vê tudo sem perigo. Venham na ponta dos pés!", 0.9, 0.5],
        ["narrador", "Lá dentro da caverna, o leão estava num trono, com todos os bichos em volta. Havia uma doença no reino, e o leão queria descobrir de quem era a culpa.", 0.95, 0.8],
        ["leao", "Amigos! Talvez a culpa seja minha. Eu já fiz muita maldade com os outros bichos.", 0.95, 0.5],
        ["narrador", "Mas ele só falou isso para parecer bonzinho. Sabia que ninguém ia ter coragem de concordar.", 1.0, 0.4],
        ["raposa", "Que bobagem, Majestade! O senhor é um santo!", 1.05, 0.5],
        ["narrador", "E o tigre, e o urso, e todos os bichos fortes de dentes afiados, fizeram a mesma coisa. E a raposa disse que todos eram santos.", 1.0, 0.5],
        ["narrador", "Por fim, chegou a vez do burro.", 0.95, 0.4],
        ["burro", "Eu só como capim. Nunca fiz mal a ninguém. Nem coice eu dou, porque meus pés estão inchados.", 0.9, 0.5],
        ["raposa", "Aí está o culpado! Ele só não dá coice porque está com o pé inchado. Fora com o burro!", 1.05, 0.5],
        ["leao", "Que o burro seja expulso do reino! Tigre, leve ele daqui!", 1.0, 0.5],
        ["narrador", "Mas isso é muito injusto! Por que o burro, que não fez nada, ficou com a culpa?", 0.95, 2.5],
        ["narrador", "Porque ele era o mais fraco. E ninguém tinha coragem de culpar os mais fortes. É isso que a fábula ensina.", 0.95, 0.5],
        ["narrador", "O tigre ia pegar o burro, quando uma pinha caiu do alto da caverna, bem na cabeça dele! Ploft! Ficou com um galo enorme.", 1.05, 0.5],
        ["narrador", "Foi o Peninha que jogou!", 1.05, 0.4],
        ["pedrinho", "Que pontaria!", 1.1, 0.4],
        ["penninha", "Agora vamos fugir! O leão já sentiu o nosso cheiro!", 1.1, 0.8, "galope"],
        ["narrador", "Resumindo: os bichos quiseram culpar o burro, que não tinha feito nada. Mas o Peninha defendeu o burro.", 0.95, 0.8],
        ["narrador", "E todos saíram correndo montanha abaixo, cada um para um lado... Para onde será que eles foram parar?", 1.0, 0.6],
        ["narrador", "Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    36: {"titulo": "O reino dos macacos", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo trinta e seis: O reino dos macacos.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o Peninha salvou o burro na caverna do leão. E todos fugiram correndo.", 1.0, 1.0],
        ["narrador", "Na corrida, o Peninha pulou no lombo do burro, e os outros se perderam no mato.", 1.0, 0.5],
        ["narrador", "Sem saber, entraram no reino dos macacos. Os guardas amarraram todos com cipó e levaram ao rei, Simão Quatorze.", 1.0, 0.5],
        ["emilia", "Eu tenho passaporte, senhor rei! Olhe aqui!", 1.05, 0.4],
        ["narrador", "E mostrou o alfinete de pombinha. O rei nunca tinha visto um alfinete.", 1.0, 0.4],
        ["rei_macaco", "No meu reino, o passaporte é uma banana. Mas aceito este. Soltem a boneca!", 1.0, 0.5],
        ["narrador", "O Pedrinho cochichou na língua do pê, que os macacos não entendem, para a Emília ir buscar o Peninha. E ela saiu de fininho.", 1.0, 0.8, "transicao"],
        ["narrador", "Até agora: os meninos foram presos pelos macacos. Só a Emília escapou.", 0.95, 0.8],
        ["rei_macaco", "Menina, o que você acha da minha corte?", 1.0, 0.4],
        ["narrador", "Narizinho viu só macacos peludos. Mas era esperta, e respondeu com elogios.", 1.0, 0.4],
        ["narizinho", "Estou maravilhada, Majestade! Nunca vi um rei tão bonito, nem damas tão lindas!", 1.05, 0.5],
        ["rei_macaco", "Que menina encantadora! Ela vai morar aqui para sempre, na árvore mais alta, e casar com o macaco mais gentil!", 1.05, 0.5],
        ["narrador", "Narizinho fez uma careta. Elogiou demais, e agora tinha que ficar!", 1.0, 0.5],
        ["rei_macaco", "E você, de cartola? O que acha do meu reino?", 1.0, 0.4],
        ["visconde", "Acho que esta malinha é muito pesada para um velho como eu. Posso dormir um pouco?", 0.9, 0.6],
        ["narrador", "Então trouxeram o Pedrinho. E o Pedrinho nunca mentia.", 1.0, 0.4],
        ["pedrinho", "O que eu acho? Não é reino nenhum. É um macacão num galho, cercado de outros macacos!", 1.05, 0.5],
        ["rei_macaco", "Que atrevido! Amarrem esse menino num tronco até amanhã!", 1.05, 0.8],
        ["narrador", "E agora? Narizinho presa na árvore, o Pedrinho amarrado no tronco, e o visconde dormindo num galho.", 0.95, 0.8],
        ["narrador", "Até agora: o rei dos macacos prendeu todos. Mas a Emília foi buscar ajuda.", 0.95, 0.8],
        ["narrador", "Veio a noite. Os macacos dormiram, roncando alto. Será que alguém vem salvar os meninos?", 0.95, 2.5],
        ["narrador", "Lá longe, um galo cantou: Cocoricocó! Era o Peninha, montado no burro, a galope!", 1.05, 0.5, "galope"],
        ["penninha", "Eu pus uma plantinha de dormir no poço onde os macacos bebem água. Eles só acordam amanhã, ao meio-dia!", 1.0, 0.5],
        ["narrador", "O Peninha desceu a Narizinho da árvore, desamarrou o Pedrinho e acordou o visconde.", 1.0, 0.4],
        ["pedrinho", "Espere! Antes, quero dar uma lição no rei.", 1.0, 0.4],
        ["narrador", "E cortou a barba do rei dos macacos, enquanto ele roncava. Sem a barba, amanhã ninguém vai reconhecer o rei!", 1.05, 0.6],
        ["narrador", "Todos montaram no burro e voltaram, a galope, até a pedra onde os dois sábios ainda conversavam.", 1.05, 0.5],
        ["narizinho", "Adeus, senhor de La Fontaine! Adeus, senhor Esopo! Depois eu conto tudo direitinho!", 1.05, 0.5],
        ["emilia", "Apareçam no sítio para tomar um cafezinho! Mas o senhor Esopo vá de calça e paletó, senão a Tia Nastácia se assusta!", 1.05, 0.5],
        ["narrador", "O senhor de La Fontaine ficou olhando os meninos sumirem na poeira, e suspirou: Felicidade, o teu nome é ser criança!", 0.9, 1.0],
        ["narrador", "Resumindo: o Peninha salvou todo mundo do reino dos macacos.", 0.95, 0.8],
        ["narrador", "Mas a viagem pelo Mundo das Maravilhas ainda não acabou. O Peninha tem mais pó de pirlimpimpim...", 0.95, 0.6],
        ["narrador", "Para onde eles vão agora? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    37: {"titulo": "O burro falante", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo trinta e sete: O burro falante.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o Peninha salvou os meninos do reino dos macacos. E todos voltaram montados no burro.", 1.0, 1.0],
        ["narrador", "No sítio, a Dona Benta estava preocupada. As crianças tinham sumido desde cedo, sem tomar café.", 1.0, 0.4],
        ["nastacia", "Não se preocupe, sinhá. Quando a fome apertar, eles largam todos os reis do mundo e voltam correndo.", 1.0, 0.6],
        ["narrador", "Às seis horas, viram uma nuvem de poeira na estrada. Era um burro, com todo mundo montado em cima!", 1.0, 0.8, "galope"],
        ["nastacia", "Credo, sinhá! O visconde vem flutuando no ar, com uma pena de papagaio em cima dele!", 1.05, 0.5],
        ["emilia", "É o Peninha! O menino invisível! Ele está segurando o visconde!", 1.1, 0.5],
        ["narrador", "Narizinho contou a viagem inteira para a vovó. E a Dona Benta ficou de boca aberta.", 1.0, 0.4],
        ["pedrinho", "E o burro agora é nosso, vovó! Nós salvamos ele. E é um burro falante!", 1.05, 0.6],
        ["dona_benta", "Se o papagaio fala, por que não um burro? Nastácia, leve umas espigas de milho e água fresquinha para ele.", 0.95, 0.5],
        ["narrador", "A Tia Nastácia pôs o milho no chão, mas de longe, com medo. O burro comeu, bebeu, e disse:", 1.0, 0.4],
        ["burro", "Muito obrigado, tia! Deus lhe pague!", 0.95, 0.5],
        ["nastacia", "Acuda, sinhá! Ele fala mesmo!", 1.1, 0.5],
        ["narrador", "E a Tia Nastácia correu para a cozinha. Mas depois, o medo passou. Ela virou a melhor amiga do burro.", 1.0, 0.8, "transicao"],
        ["narrador", "Até agora: os meninos voltaram para casa com um burro falante, e a Tia Nastácia virou amiga dele.", 0.95, 0.8],
        ["narrador", "O Peninha foi embora naquela noite. Prometeu voltar um dia, para levar todos ao Mar dos Piratas.", 1.0, 0.5],
        ["dona_benta", "Ah, que pena eu não ter ido. Conversar com o senhor de La Fontaine seria uma alegria para a minha velhice.", 0.9, 0.5],
        ["pedrinho", "Tive uma ideia, vovó! A senhora vai com a gente! Já sabemos o caminho, e temos o burro!", 1.1, 0.5],
        ["dona_benta", "Que maluquice, Pedrinho! Eu tenho mais de setenta anos!", 1.0, 0.6],
        ["narrador", "Mas ela pensou três dias. E por fim, disse:", 0.95, 0.4],
        ["dona_benta", "Vou, menino, vou! Mas não me apresse mais.", 0.95, 0.6],
        ["narrador", "As crianças ficaram felizes demais. Levar a vovó para o País das Fábulas! E tudo em segredo, para a Tia Nastácia não saber.", 1.05, 0.5],
        ["dona_benta", "Nastácia, vou passear com as crianças. Se alguém perguntar, estou na casa do compadre Teodureto.", 0.95, 0.6],
        ["narrador", "A vovó não conseguia subir no burro. Por que será?", 0.95, 2.5],
        ["burro", "Ali na frente tem um toco. Pode servir de banquinho, senhora.", 0.95, 0.5],
        ["narrador", "Com a ajuda de todos, a vovó montou. O Pedrinho deu uma pitada do pó de pirlimpimpim para ela.", 1.0, 0.4],
        ["dona_benta", "Pedrinho! Você sabe que eu não cheiro rapé!", 1.0, 0.5],
        ["pedrinho", "Não é rapé, vovó! É pó de pirlimpimpim!", 1.05, 0.5],
        ["narrador", "Todos cheiraram. A cabeça rodou, e veio um zumbido no ouvido: Fiunnn...", 1.0, 0.8, "magia"],
        ["dona_benta", "Acudam! Acho que vou desmaiar!", 1.05, 0.5],
        ["pedrinho", "Calma, vovó! É assim mesmo. Quando o fiunnn parar, a gente chegou.", 1.0, 1.0],
        ["narrador", "Resumindo: a vovó criou coragem e foi viajar com os netos, com o pó de pirlimpimpim.", 0.95, 0.8],
        ["narrador", "Mas o Pedrinho ainda não sabia usar o pó direito. E deu pó demais...", 0.95, 0.6],
        ["narrador", "Onde será que eles foram parar? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    38: {"titulo": "O pássaro Roca", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo trinta e oito: O pássaro Roca.", 0.95, 1.2],
        ["narrador", "No capítulo passado, a vovó foi viajar com os netos, com o pó de pirlimpimpim. Mas o Pedrinho deu pó demais.", 1.0, 1.0],
        ["narrador", "Em vez do campo verde do País das Fábulas, chegaram num deserto, com pedras pretas de um lado e o mar do outro.", 1.0, 0.5],
        ["narrador", "Só havia duas árvores gêmeas, muito esquisitas. Os troncos eram cheios de escamas, e as raízes eram curvas como unhas.", 1.0, 0.5],
        ["narrador", "A vovó sentou numa raiz para descansar. Mas a raiz se mexeu!", 1.0, 0.5],
        ["dona_benta", "Acudam! A raiz mexeu!", 1.1, 0.5],
        ["narrador", "E as duas árvores subiram para o céu, levando o burro pendurado pelo cabresto!", 1.05, 0.5],
        ["dona_benta", "Não eram árvores! Eram as pernas do pássaro Roca! E eu sentei no dedo dele!", 1.05, 0.6],
        ["narrador", "O pássaro Roca é um pássaro gigante das histórias das Mil e Uma Noites. Tão grande que nem viu os meninos lá embaixo.", 1.0, 0.8, "transicao"],
        ["narrador", "Até agora: o pássaro Roca voou para o céu, levando o burro pendurado!", 0.95, 0.8],
        ["pedrinho", "Pelo mapa, o castelo do barão de Münchhausen é aqui perto. Ele é o melhor atirador do mundo. Pode cortar o cabresto com um tiro!", 1.05, 0.5],
        ["narrador", "O Pedrinho cheirou um pouquinho de pó e sumiu como fumaça. A vovó ficou tremendo de medo.", 1.0, 0.5],
        ["narizinho", "Não tenha medo, vovó! Eu e o Pedrinho já vivemos tantas aventuras que nem sei mais o que é medo.", 1.0, 0.5],
        ["dona_benta", "É que eu sou de carne, minha filha. Quando você crescer, vai entender.", 0.9, 0.6],
        ["narrador", "Logo o Pedrinho voltou com o barão. Ele tinha botas grandes, chapéu de três pontas e uma espingarda antiga.", 1.0, 0.5],
        ["barao", "Não se preocupe, minha senhora. Eu estou aqui para ajudar.", 0.95, 0.5],
        ["narrador", "A vovó ofereceu uma coxinha da galinha que tinha trazido. E o barão, sentado no chão, comeu quase tudo sozinho!", 1.0, 0.6],
        ["barao", "Eu não atiro com bala no pássaro Roca. Atiro caroços de cereja! Eles nascem nas costas dele, e viram árvores. Já plantei umas cem cerejeiras lá em cima!", 1.0, 0.8],
        ["narrador", "Até agora: o barão de Münchhausen chegou para salvar o burro.", 0.95, 0.8],
        ["narrador", "A espingarda do barão era bem antiga. Ela tinha uma pedrinha chamada pederneira, que faz uma faísca para o tiro sair.", 1.0, 0.5],
        ["narrador", "Mas a Emília, curiosa, escondeu a pedrinha, só para ver o que o barão ia fazer.", 1.0, 0.5],
        ["narrador", "O pássaro Roca desceu, voando sobre o mar. O barão mirou e apertou o gatilho: plec! Nada!", 1.0, 0.5],
        ["barao", "Onde foi parar a pederneira da minha espingarda?", 1.0, 0.4],
        ["narrador", "E agora? Como fazer a faísca?", 0.95, 2.5],
        ["narrador", "O Pedrinho pegou duas pedras da praia e bateu uma na outra, bem perto da espingarda. Saiu uma faísca linda!", 1.05, 0.5],
        ["narrador", "Bum! O tiro acertou bem no cabresto. E o burro caiu no mar: tchibum!", 1.1, 0.8, "agua"],
        ["burro", "Socorro! Eu não sei nadar!", 1.1, 0.5],
        ["narrador", "O barão e o Pedrinho entraram no mar e puxaram o burro para a praia, um pela orelha, outro pelo rabo.", 1.05, 0.5],
        ["narrador", "O burro botou para fora toda a água salgada que tinha engolido. E sentou na areia, mais calmo.", 1.0, 0.5],
        ["dona_benta", "Está melhorzinho?", 0.95, 0.4],
        ["burro", "Um pouco, senhora. Muito obrigado.", 0.9, 1.0],
        ["narrador", "Resumindo: o barão e o Pedrinho salvaram o burro do pássaro Roca.", 0.95, 0.8],
        ["emilia", "E o visconde? Ele estava amarrado no burro, com a minha malinha!", 1.05, 0.6],
        ["narrador", "O visconde tinha sumido! Onde será que ele está? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    39: {"titulo": "O ovo gigante", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo trinta e nove: O ovo gigante.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o barão e o Pedrinho salvaram o burro. Mas o visconde sumiu, com a malinha da Emília.", 1.0, 1.0],
        ["narrador", "Narizinho estava catando conchinhas na praia, quando gritou:", 1.0, 0.4],
        ["narizinho", "Corram! Achei o visconde!", 1.1, 0.5],
        ["narrador", "O visconde estava meio enterrado na areia, desacordado. O mar tinha levado os braços, as pernas e a cartolinha dele.", 0.9, 0.5],
        ["narrador", "Mas ele não tinha largado a malinha da Emília. Que visconde leal!", 0.95, 0.6],
        ["narrador", "Todos ficaram tristes. E a Emília? O que será que ela fez?", 0.95, 2.5],
        ["narrador", "Ela abriu a malinha, para ver se estava tudo lá dentro! Depois guardou o visconde na malinha.", 1.0, 0.5],
        ["emilia", "A Tia Nastácia é muito habilidosa. Com este pedaço de visconde, ela faz um visconde novinho, ainda mais bonito!", 1.0, 0.6],
        ["dona_benta", "Pobre visconde. A gente nunca deu muito valor a ele. E ele foi tão fiel...", 0.9, 0.8, "transicao"],
        ["narrador", "Até agora: o visconde ficou muito estragado no mar, e a Emília guardou ele na malinha, para consertar.", 0.95, 0.8],
        ["narrador", "Uma carruagem veio buscar todos para o castelo do barão. Mas no meio do caminho, entre as montanhas, parou de repente.", 1.0, 0.5, "galope"],
        ["barao", "Uma pedra redonda enorme caiu da montanha e fechou a passagem.", 1.0, 0.5],
        ["narrador", "Mas não era pedra. Era um ovo do pássaro Roca! Maior do que dois barris, um em cima do outro!", 1.05, 0.5],
        ["dona_benta", "Que pena a Nastácia não estar aqui para ver um ovo deste tamanho!", 1.0, 0.5],
        ["narrador", "Para passar, precisavam quebrar o ovo. O barão mandou buscar picaretas.", 1.0, 0.4],
        ["emilia", "Eu também quero uma! Eu também quero quebrar o ovo!", 1.1, 0.5],
        ["narrador", "Bateram, bateram, até a casca rachar. O barão gritou:", 1.0, 0.4],
        ["barao", "Fujam! Vai escorrer clara e gema e virar tudo omelete!", 1.1, 0.5],
        ["narrador", "Todos subiram no barranco. A vovó foi puxada pelos meninos.", 1.0, 0.5],
        ["dona_benta", "Viver mais de setenta anos para fugir de virar omelete!", 1.0, 0.6],
        ["narrador", "Mas o que será que saiu do ovo?", 0.95, 2.5],
        ["narrador", "Um pintinho gigante! Ele pôs a cabeça para fora e piou tão alto que dava para ouvir de muito longe.", 1.05, 0.5],
        ["barao", "Se a mãe dele ouvir, vem correndo! Vamos amarrar o bico desse pintão!", 1.05, 0.5],
        ["narrador", "Todos ajudaram, até a vovó. E amarraram o bico do pintão com cordas. Mas o pássaro Roca já tinha ouvido, e vinha voando como um furacão de penas!", 1.05, 0.5],
        ["barao", "Corram para o castelo!", 1.1, 0.5],
        ["narrador", "Até a vovó esqueceu os setenta anos e correu como um veado! Chegou ao castelo sem fôlego.", 1.05, 0.6],
        ["narrador", "Lá da torre, o Pedrinho viu o pássaro Roca chegar no ninho. Ela viu o filhote, ficou feliz, e nem percebeu o bico amarrado!", 1.0, 1.0],
        ["narrador", "Resumindo: do ovo gigante nasceu um pintão, e todos fugiram para o castelo do barão.", 0.95, 0.8],
        ["narrador", "Mas como eles vão voltar para o sítio agora? Isso fica para o próximo capítulo de Reinações de Narizinho. Até lá!", 1.0, 0.8],
    ]},
    40: {"titulo": "De volta ao sítio", "ambiente": "nenhum", "trechos": [
        ["narrador", "Reinações de Narizinho. Capítulo quarenta, o último: De volta ao sítio.", 0.95, 1.2],
        ["narrador", "No capítulo passado, do ovo gigante nasceu um pintão, e todos fugiram para o castelo do barão.", 1.0, 1.0],
        ["narrador", "Naquela noite, a vovó foi dormir cedo. Os meninos ficaram ouvindo as histórias de caçada do barão.", 1.0, 0.4],
        ["narrador", "Mas chegou um mensageiro a cavalo, com uma carta do Imperador. O barão precisava voltar para o país dele, a Alemanha, na mesma hora.", 1.0, 0.5],
        ["barao", "Vocês podem ficar no castelo quanto tempo quiserem. Os criados vão cuidar de vocês.", 0.95, 0.5],
        ["emilia", "Traga um presente para mim!", 1.05, 0.4],
        ["narrador", "E o barão foi embora.", 1.0, 0.8, "transicao"],
        ["narrador", "Até agora: o barão precisou viajar, e os meninos ficaram sozinhos no castelo, com a vovó.", 0.95, 0.8],
        ["dona_benta", "O que vai ser de mim, neste castelo, com o pássaro Roca morando aqui do lado?", 0.9, 0.5],
        ["pedrinho", "Calma, vovó. Do jeito que eu trouxe a senhora, eu levo de volta.", 1.0, 0.5],
        ["narrador", "O Pedrinho deu uma pitada de pó para a vovó. Ela cheirou... e nada. Mais uma pitada. Nada!", 1.0, 0.5],
        ["narrador", "Por que será que o pó não funcionou?", 0.95, 2.5],
        ["narrador", "Porque o pó molhou na água do mar, quando o Pedrinho entrou para salvar o burro. E o pó de pirlimpimpim aguenta tudo, menos sal!", 1.0, 0.5],
        ["narrador", "Nessa hora, ouviram um barulho de asas lá fora. O pássaro Roca vinha voando para o castelo!", 1.05, 0.5],
        ["dona_benta", "Socorro! Acudam!", 1.1, 0.5],
        ["narrador", "Todos corriam de um lado para o outro. E foi a Emília que teve a grande ideia.", 1.0, 0.4],
        ["emilia", "Fechem os olhos, com toda a força!", 1.1, 0.5],
        ["narrador", "Todos fecharam os olhos bem fechados, como a gente faz nos sonhos. E quando abriram...", 0.9, 0.8, "magia"],
        ["narrador", "Estavam no sítio de novo, pertinho da porteira!", 1.05, 0.8],
        ["narrador", "Até agora: com a ideia da Emília, todos voltaram para casa.", 0.95, 0.8],
        ["dona_benta", "Não contem nada para a Nastácia. Vamos fingir que fomos à casa do compadre Teodureto.", 0.9, 0.5],
        ["narrador", "Mas a Tia Nastácia estava na varanda, de mãos na cintura.", 1.0, 0.4],
        ["nastacia", "Não precisa dizer nada, sinhá. O burro chegou primeiro e me contou a história inteirinha!", 1.0, 0.6],
        ["narrador", "E a Dona Benta, que nunca tinha mentido na vida, foi para o quarto, toda envergonhada.", 0.95, 0.8],
        ["narrador", "No dia seguinte, chegou uma carta da mãe do Pedrinho. Ele precisava voltar para a cidade.", 0.95, 0.5],
        ["pedrinho", "Que pena! Justo agora que temos o burro falante e o Peninha para viajar!", 0.95, 0.5],
        ["narrador", "O Pedrinho tomou café com os bolinhos da Tia Nastácia e montou no cavalinho.", 0.95, 0.4],
        ["pedrinho", "Adeus, vovó! Adeus, Narizinho! Adeus, Tia Nastácia! Adeus, Emília! Adeus, Faz-de-Conta!", 1.0, 0.5],
        ["emilia", "Espere! Você esqueceu de se despedir do visconde!", 1.05, 0.4],
        ["pedrinho", "Mas o visconde está todo estragado, Emília.", 0.95, 0.4],
        ["emilia", "Está estragado, mas não acabou. Eu escutei aqui dentro: a alma dele está vivinha! A Tia Nastácia vai fazer ele de novo.", 1.0, 0.6],
        ["narrador", "O Pedrinho riu e deu um beijinho no pedaço de sabugo. Depois saiu a galope, para a Emília não ver duas lágrimas que caíam dos olhos dele.", 0.9, 1.0, "galope"],
        ["narrador", "Resumindo: todos voltaram do Mundo das Maravilhas, e o Pedrinho foi para a cidade, com saudade.", 0.95, 0.8],
        ["narrador", "E assim terminam as Reinações de Narizinho.", 0.9, 0.6],
        ["narrador", "Mas o sítio continua lá. O visconde vai ganhar braços novos. O Peninha prometeu voltar. E a Narizinho ainda pensa no Príncipe Escamado, onde quer que ele esteja.", 0.9, 0.8],
        ["narrador", "Essas, porém, são outras histórias. Obrigado por ouvir. Até a próxima!", 0.9, 1.0, "festa"],
    ]},
}

for n, c in CAPITULOS.items():
    palavras = sum(len(t[1].split()) for t in c["trechos"])
    print(f"Capítulo {n}, {c['titulo']}: {len(c['trechos'])} trechos, cerca de "
          f"{palavras / 130 + sum(t[3] for t in c['trechos']) / 60:.1f} min.")

In [ ]:
# ── 6. Criar as vozes e ouvir (troque a semente de quem não ficou bom) ──────
import hashlib, os, shutil, requests
from IPython.display import Audio, display

def gerar(texto, destino, **campos):
    arquivos = campos.pop("files", None)
    dados = {"text": texto, "language": "pt", "effect_preset": "raw", **campos}
    r = requests.post(f"{BASE}/generate", data=dados, files=arquivos, timeout=1800)
    if r.status_code != 200:
        raise SystemExit(f"Erro {r.status_code} em /generate: {r.text[:500]}\nLog: {LOG_PATH}")
    with open(destino, "wb") as f:
        f.write(r.content)

def chave(*partes):
    return hashlib.sha1(repr(partes).encode()).hexdigest()[:8]

REF = {}   # personagem -> (wav de referência, texto)
for nome, v in VOZES.items():
    # A fala mais longa do personagem no primeiro capítulo em que ele aparece.
    texto = next(max(falas, key=len) for _, c in sorted(CAPITULOS.items())
                 if (falas := [t[1] for t in c["trechos"] if t[0] == nome]))
    arq = f"{nome}_{chave(v, texto)}.wav"
    wav = os.path.join(PASTA_VOZES, arq)
    antigo = os.path.join(LIVRO, "cap01", "vozes", arq)   # versão do primeiro teste
    if not os.path.exists(wav) and os.path.exists(antigo):
        shutil.copy(antigo, wav)
    if not os.path.exists(wav):
        if v.get("amostra"):
            with open(v["amostra"], "rb") as a:
                gerar(texto, wav, seed=v["semente"], files={"ref_audio": ("amostra.wav", a)})
        else:
            gerar(texto, wav, instruct=v["voz"], seed=v["semente"])
    REF[nome] = (wav, texto)
    print(f"{nome} ({v['voz']}, semente {v['semente']}): {texto}")
    display(Audio(wav))

In [ ]:
# ── 7. Narrar e montar cada capítulo (A: só voz; B: vinheta, sons leves, riacho) 
import os, subprocess
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

def narrar(n):
    pasta = os.path.join(LIVRO, f"cap{n:02d}", "trechos")
    os.makedirs(pasta, exist_ok=True)
    arquivos = []
    for i, t in enumerate(CAPITULOS[n]["trechos"], 1):
        quem, texto, vel, pausa = t[:4]
        efeito = t[4] if len(t) > 4 else None
        ref_wav, ref_texto = REF[quem]
        wav = os.path.join(pasta, f"{i:03d}_{quem}_{chave(os.path.basename(ref_wav), texto, vel)}.wav")
        if not os.path.exists(wav):
            with open(ref_wav, "rb") as ref:
                gerar(texto, wav, seed=VOZES[quem]["semente"], ref_text=ref_texto, speed=vel,
                      files={"ref_audio": ("ref.wav", ref, "audio/wav")})
            print(f"{n}.{i:03d} {quem}: {texto}")
        arquivos.append((wav, pausa, efeito))
    return arquivos

def tom(freq, dur, vol, sr):
    t = np.arange(int(dur * sr)) / sr
    return (vol * np.sin(2 * np.pi * freq * t) * np.exp(-3 * t)).astype("float32")

def vinheta(sr):
    # Três notas suaves (dó, mi, sol), sem ataque forte.
    v = np.zeros(int(2.2 * sr), dtype="float32")
    for k, f in enumerate([523.25, 659.25, 783.99]):
        nota = tom(f, 1.4, 0.12, sr)
        ini = int(k * 0.35 * sr)
        v[ini:ini + len(nota)] += nota
    return v

def riacho(n, sr):
    # Ruído suave filtrado, que lembra água correndo, bem baixo e com entrada lenta.
    rng = np.random.default_rng(7)
    r = np.convolve(rng.standard_normal(n + 400).astype("float32"),
                    np.ones(40, dtype="float32") / 40, mode="valid")[:n]
    r *= 0.01 * (1 + 0.3 * np.sin(2 * np.pi * 0.15 * np.arange(n) / sr))
    fade = min(n // 2, 4 * sr)
    r[:fade] *= np.linspace(0, 1, fade)
    r[-fade:] *= np.linspace(1, 0, fade)
    return r

# ── Sonoplastia leve (só na versão B) ────────────────────────────────────────
# Sete sons curtos, sintetizados aqui mesmo, sempre iguais (mesmo som = mesmo sentido).
# Entram só nas pausas, nunca por cima da fala; ataque suave; pico baixo.

def _env(n, sr, ataque=0.03, queda=4.0):
    t = np.arange(n) / sr
    a = np.clip(t / ataque, 0, 1)
    return (a * np.exp(-queda * t)).astype("float32")

def _nota(f, dur, vol, sr, queda=3.0):
    t = np.arange(int(dur * sr)) / sr
    som = np.sin(2 * np.pi * f * t) + 0.25 * np.sin(2 * np.pi * 2 * f * t)
    return (vol * som * _env(len(t), sr, 0.02, queda)).astype("float32")

def _suave(x, k):
    return np.convolve(x, np.ones(k, dtype="float32") / k, mode="same").astype("float32")

def _junta(sr, dur, pecas):
    out = np.zeros(int(dur * sr), dtype="float32")
    for ini, p in pecas:
        i = int(ini * sr)
        p = p[:len(out) - i]
        out[i:i + len(p)] += p
    return out

def ef_transicao(sr):   # troca de cena: duas notas que descem (sol, dó)
    return _junta(sr, 1.3, [(0, _nota(783.99, 1.0, 0.07, sr)), (0.3, _nota(523.25, 1.0, 0.07, sr))])

def ef_magia(sr):       # encanto, sonho, feitiço: quatro notas agudas subindo
    fs = [1046.5, 1318.5, 1568.0, 2093.0]
    return _junta(sr, 1.5, [(k * 0.12, _nota(f, 0.9, 0.035, sr, 4.0)) for k, f in enumerate(fs)])

def ef_agua(sr):        # mergulho / água: ruído abafado + duas bolhas
    rng = np.random.default_rng(11)
    n = int(0.6 * sr)
    chua = _suave(rng.standard_normal(n).astype("float32"), 60) * _env(n, sr, 0.04, 6.0) * 0.6
    def bolha(f0):
        t = np.arange(int(0.12 * sr)) / sr
        return (0.04 * np.sin(2 * np.pi * (f0 + 900 * t) * t) * _env(len(t), sr, 0.01, 20)).astype("float32")
    return _junta(sr, 0.9, [(0, chua), (0.35, bolha(500)), (0.55, bolha(650))])

def ef_galope(sr):      # cavalinho: pac, pac, pac (quatro batidinhas de madeira)
    def pac(f):
        t = np.arange(int(0.08 * sr)) / sr
        return (0.09 * np.sin(2 * np.pi * f * t) * _env(len(t), sr, 0.003, 45)).astype("float32")
    return _junta(sr, 1.2, [(k * 0.25, pac(700 if k % 2 else 600)) for k in range(4)])

def ef_porta(sr):       # batida na porta: toc, toc, toc (grave e abafado)
    def toc():
        t = np.arange(int(0.12 * sr)) / sr
        return (0.12 * np.sin(2 * np.pi * 160 * t) * _env(len(t), sr, 0.004, 30)).astype("float32")
    return _junta(sr, 1.0, [(k * 0.28, toc()) for k in range(3)])

def ef_zumbido(sr):     # abelhas: zum-zum baixinho, entra e sai devagar
    t = np.arange(int(1.2 * sr)) / sr
    z = sum(np.sin(2 * np.pi * h * 210 * t) / h for h in (1, 2, 3))
    z *= 0.5 + 0.5 * np.sin(2 * np.pi * 9 * t)
    janela = np.sin(np.pi * t / t[-1]) ** 2
    return (0.025 * z * janela).astype("float32")

def ef_festa(sr):       # alegria, festa, final feliz: acorde curto (dó, mi, sol juntos)
    return _junta(sr, 1.4, [(0, _nota(f, 1.3, 0.04, sr, 2.5)) for f in (523.25, 659.25, 783.99)])

EFEITOS = {"transicao": ef_transicao, "magia": ef_magia, "agua": ef_agua, "galope": ef_galope,
           "porta": ef_porta, "zumbido": ef_zumbido, "festa": ef_festa}

def exportar(audio, sr, pasta, nome):
    tmp = os.path.join(pasta, nome + ".wav")
    sf.write(tmp, audio, sr)
    mp3 = os.path.join(pasta, nome + ".mp3")
    # Volume normalizado para fone e caixinha: -16 LUFS, pico máximo -1.5 dBTP.
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp,
                    "-af", "loudnorm=I=-16:TP=-1.5:LRA=11", "-ar", "44100",
                    "-codec:a", "libmp3lame", "-q:a", "2", mp3], check=True)
    print(f"{nome}.mp3: {len(audio) / sr / 60:.1f} min")
    display(Audio(mp3))

def montar(n, arquivos):
    partes, marcas, sr, pos = [], [], None, 0
    for wav, pausa, efeito in arquivos:
        a, s = sf.read(wav, dtype="float32", always_2d=True)
        sr = sr or s
        assert s == sr, f"taxa de amostragem diferente em {wav}"
        fala = a.mean(axis=1)
        pos += len(fala)
        if efeito:
            som = EFEITOS[efeito](sr)
            pausa = max(pausa, len(som) / sr + 0.4)   # o som cabe inteiro na pausa
            marcas.append((pos + int(0.15 * sr), som))
        partes += [fala, np.zeros(int(pausa * sr), dtype="float32")]
        pos += int(pausa * sr)
    voz = np.concatenate(partes)
    # Sons das pausas: 15 dB abaixo da fala (RMS) e pico até 1/3 do pico da fala.
    ativo = voz[np.abs(voz) > 0.01]
    rms_voz = float(np.sqrt((ativo ** 2).mean())) if len(ativo) else 0.1
    sons = np.zeros_like(voz)
    for ini, som in marcas:
        sa = som[np.abs(som) > 1e-4]
        g = rms_voz * 10 ** (-15 / 20) / float(np.sqrt((sa ** 2).mean()))
        g = min(g, np.abs(voz).max() / 3 / float(np.abs(som).max()))
        sons[ini:ini + len(som)] += (som * g)[:len(sons) - ini]
    fundo = riacho(len(voz), sr) if CAPITULOS[n].get("ambiente", "riacho") == "riacho" else 0
    um_seg = np.zeros(sr, dtype="float32")
    pasta = os.path.join(LIVRO, f"cap{n:02d}")
    exportar(np.concatenate([um_seg, voz, um_seg]), sr, pasta, f"cap{n:02d}_A_so_voz")
    exportar(np.concatenate([vinheta(sr), um_seg, voz + sons + fundo, um_seg, vinheta(sr)]),
             sr, pasta, f"cap{n:02d}_B_com_ambiente")

REFAZER = False   # True: narra de novo capítulos que já têm os dois MP3 prontos

for n in CAPITULOS_PARA_GERAR:
    print(f"\n=== Capítulo {n}: {CAPITULOS[n]['titulo']} ===")
    prontos = [os.path.join(LIVRO, f"cap{n:02d}", f"cap{n:02d}_{v}.mp3")
               for v in ("A_so_voz", "B_com_ambiente")]
    if not REFAZER and all(map(os.path.exists, prontos)):
        print("Já estava pronto (para refazer, use REFAZER = True).")
        continue
    montar(n, narrar(n))
print("\nPronto! Os arquivos estão no Drive, em", LIVRO)

# Se um trecho ficou ruim, apague o .wav dele na pasta capNN/trechos/ e rode esta célula de novo.